# A股市场日度跟踪系统 v1.23

本版本新增行业指数分析板块：按 demo 的中证/国证/韭圈儿/申万数据源获取行业指数K线与估值，生成行业指数日K线、鱼盆模型、数据表、行业指数鱼盆快照，并在市场快照数据表中新增“行业指数快照”sheet。PE历史分位数固定使用中证K线字段“滚动市盈率”。

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
A股市场日度跟踪工具
===================

用途
----
面向A股个人投资者的日度市场跟踪脚本：
- 跟踪宽基指数、行业指数与行业ETF日K线表现
- 生成技术分析图、鱼盆模型图、市场快照图、行业指数快照、ETF轮动热力图
- 导出单标的数据表、市场快照数据表与PDF汇总文档

版本历史
--------
v1.0  初始日度版：沿用周跟踪程序的标的池与输出结构，将周K线改为日K线。
v1.1  指标扩展：新增MA5/10/20/60/200、MACD(12,26,9)、KDJ(9,3,3)、RSI(6,12,24)、量比。
v1.2  鱼盆模型统一：指数与行业ETF鱼盆模型统一使用MA5/MA20。
v1.3  ETF增强：新增相对沪深300超额收益、资金净流入估算、20日滚动波动率与ETF轮动热力图。
v1.4  本地化修复：输出目录固定到本地output_daily_tracker，修复MPLCONFIGDIR路径导致的只读文件系统问题。
v1.5  数据源稳定性修复：增加东方财富、Sina、腾讯、乐咕等备用源容错，默认禁用系统代理以减少ProxyError。
v1.6  图表与中文修复：优化中文字体、恢复高清出图，并采用单标的生成后立即关闭figure的方式控制内存。
v1.7  展示窗口修复：图表展示窗口改为100个交易日，MA200继续基于480个交易日预热计算。
v1.8  日K线扩展：新增BOLL布林带，并将BOLL上轨/中轨/下轨同步输出到单标的数据表。
v1.9  时间轴优化：所有图表使用交易日序号绘图，去掉非交易日空档；每周显示一个刻度，优先周五，周五休盘则回退周四；日期格式YY/MM/DD。
v1.10 快照与热力图优化：快照第一列增加“指数(代码)”/“ETF(代码)”表头，表格线条调细调淡，快照按MA20乖离率降序排列。
v1.11 市场快照Excel改为写入真正数值并设置number_format，避免“以文本形式存储的数字”绿三角；ETF热力图改为近20日/近5日涨跌幅，近20日在上、近5日在下，并按近5日涨跌幅从左到右降序排列；补充代码注释。
v1.12 MACD面板DIF/DEA颜色互换；所有宽基与ETF抓取全历史K线用于计算MA20乖离率全历史分位；鱼盆MA20乖离率面板新增分位数趋势线、正/负乖离20%与80%参考虚线、超买/超卖状态标记；输出由Word改为PDF，并将全部图表按原始清晰度写入PDF。
v1.13 本次优化：MA20乖离率分位改为“有符号分位数”表达，右轴0上方为正乖离超买强度、0下方为负乖离超卖强度；分位趋势线改为橙色实线；严重超买/严重超卖区域增加淡红/淡绿背景；PDF生成改为Matplotlib PdfPages，去掉reportlab依赖；新增本地K线缓存，首次全量保存，后续增量更新。
v1.14 本次优化：鱼盆数据与快照新增“金叉/死叉天数”和“区间涨跌幅(%)”；单标Excel改为输出全历史K线与全历史鱼盆数据；新增指标缓存，只有原始K线或基准数据出现增量/修正时才重新计算全历史分位数，否则直接读取本地指标缓存，避免重复计算。
v1.15 BUG修复：统一所有成交量/均量/量比计算口径，MA5均量、MA20均量均直接由成交量rolling mean计算；指标缓存版本升级为v115强制重算，避免旧缓存污染；MA20乖离率分位数改为全历史静态排序分位，正负乖离分开排名，重复值取平均分位。
v1.16 BUG修复：宽基指数成交量统一从“股”换算为“手”（÷100），避免中证500等指数成交量/均量放大100倍；所有均量、量比在指标层强制同源重算；成交量显示单位阈值统一；鱼盆柱、鱼盆/KDJ/RSI异常背景统一使用成交量面板红绿体系，分位趋势线改为深灰色。
v1.17 BUG修复：成交量单位改为“按成交额/成交量隐含单价自动识别股/手”，不再对所有指数粗暴÷100；无成交量数据的指数（如仅close-only源）保留N/A，不再显示0.0；快照成交量/均量统一保留两位小数；修复pandas新版本读取指标缓存时 errors="ignore" 导致的 invalid error value specified 警告；鱼盆分位趋势线由深灰色调整为灰色。
v1.18 BUG修复：成交量口径统一为“手”口径；宽基指数大多数源返回股数，统一除以100转为手，中证2000乐咕源已为同口径则不再折算；ETF沿用新浪/东方财富返回的交易量口径并统一按手解释。所有成交量、MA5均量、MA20均量、量比重新由同一成交量序列计算，快照单位显示为亿手/万手；鱼盆分位趋势线颜色进一步调浅。
v1.19 BUG修复：启用独立 raw_v119 原始K线缓存目录，避免旧缓存中股/手单位混杂污染均量；成交量按行识别股口径并转为手；修复 MA5-MA20 交叉日出现 nan 的问题；重新加入禁用系统代理逻辑。
v1.20 BUG修复：修复 pandas 3.x 在 normalize_volume_units 中向 int64 成交量序列写入 float 折算值时触发 LossySetitemError/TypeError 的问题；成交量归一化前显式转为 float64，确保无缓存首次全量运行可以完成。
v1.21 本次优化：按用户指定口径修复成交量折算，科创50(000688)、北证50(899050)以及所有ETF每日成交量在指标计算前固定除以100，避免部分ETF历史日成交量股/手混杂；ETF图表、ETF快照中的成交量/均量统一以“万手”展示。
v1.22 本次优化：进一步按用户明确口径，将所有宽基指数与所有行业ETF的历史每日成交量在进入指标计算、数据表、图表和快照前统一除以100，内部单位统一为“手”；升级raw_v122/v122缓存，避免旧版本成交量口径继续污染均量和量比。
v1.23 本次优化：新增行业指数分析板块，按中证K线滚动市盈率计算PE历史分位，并输出行业指数日K线、鱼盆模型、数据表、鱼盆快照和市场快照Excel sheet。

说明
----
资金净流入估算不是交易所官方口径，而是公开价量数据构造的代理指标：
    max(成交额 - 5日均成交额, 0) × 价格方向系数 / 1e8
其中：
    价格方向系数 = (收盘 - 开盘) / (最高 - 最低)，裁剪到 [-1, 1]
适合观察资金边际强化 / 退潮，不替代真实逐笔资金流。
"""
from __future__ import annotations

import os
from pathlib import Path

# Matplotlib 配置目录改为当前脚本所在目录下的本地可写路径，避免 /mnt 只读报错
BASE_DIR = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
MPLCONFIGDIR = BASE_DIR / ".mplconfig_daily_tracker"
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))
MPLCONFIGDIR.mkdir(parents=True, exist_ok=True)

import re
import warnings
import logging
from datetime import datetime, timedelta
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

try:
    import akshare as ak
except Exception:
    ak = None

warnings.filterwarnings("ignore")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)


mpl.rcParams["font.sans-serif"] = [
    "SimHei", "Microsoft YaHei", "PingFang SC",
    "Heiti SC", "STHeiti", "WenQuanYi Micro Hei", "Arial Unicode MS",
    "DejaVu Sans",
]
mpl.rcParams["axes.unicode_minus"] = False
mpl.rcParams["figure.dpi"] = 150


# ══════════════════════════════════════════════════════════════════════════════
# §1  全局参数
# ══════════════════════════════════════════════════════════════════════════════
# TODAY用于输出文件夹、报告标题、判断当日是否已经收盘。
TODAY = datetime.today()

# 输出目录默认放在当前脚本/Notebook同级目录下，避免写入系统只读路径。
OUTPUT_DIR = str(BASE_DIR / "output_daily_tracker")

# 本地K线缓存目录：首次运行全量下载并保存，后续优先读取本地CSV，再向后追加增量数据。
CACHE_ENABLED = True
CACHE_DIR = str(BASE_DIR / "data_cache_daily_tracker")
CACHE_INCREMENT_LOOKBACK_DAYS = 15  # 增量更新时向前回看若干天，覆盖可能的复权/补数据修正。

# 图表与Excel展示最近100个交易日；MA200等长期指标仍然用480日历史预热计算。
DISPLAY_BARS = 100
PREHEAT_BARS = 480
# 成交量显示单位阈值：大于等于1亿时用“亿”，否则用“万”。
VOLUME_DISPLAY_E_THRESHOLD = 1e8
DAILY_BARS = DISPLAY_BARS  # backward-compatible alias：兼容旧函数里仍引用 DAILY_BARS 的位置。

# 技术指标基础参数：
# 1）图表仍只展示最近 DISPLAY_BARS 个交易日；
# 2）行情抓取改为 ALL_HISTORY_START 起的全历史，用于计算 MA20 乖离率全历史分位数；
# 3）FISH_WINDOW 仅保留兼容旧函数签名，不再用于分位数滚动窗口。
MA_WINDOWS = (5, 10, 20, 60, 200)
FISH_WINDOW = None
FETCH_BUFFER_DAYS = 900
MARKET_CLOSE_HHMM = (15, 0)
ALL_HISTORY_START = "19900101"

# 行情抓取日期：尽量获取全历史数据，避免MA20乖离率分位数只基于近两三年样本。
KLINE_START = ALL_HISTORY_START
KLINE_END = TODAY.strftime("%Y%m%d")

BROAD_INDICES = [
    ("上证指数",  "000001",  None,       "index", "000001.SH"),
    ("上证50",    "000016",  "上证50",   "index", None),
    ("沪深300",   "000300",  "沪深300",  "index", None),
    ("中证500",   "000905",  "中证500",  "index", None),
    ("中证1000",  "000852",  "中证1000", "index", None),
    ("中证2000",  "932000",  None,       "index", "932000.CSI"),
    ("创业板指",  "399006",  None,       "index", "399006.SZ"),
    ("深证成指",  "399001",  None,       "index", "399001.SZ"),
    ("科创50",    "000688",  None,       "index", "000688.SH"),
    ("北证50",    "899050",  None,       "index", None),
]

SECTOR_ETFS = [
    ("银行ETF",       "512800", None, "etf", None),
    ("证券ETF",       "512880", None, "etf", None),
    ("证券保险ETF",   "512070", None, "etf", None),
    ("房地产ETF",     "512200", None, "etf", None),
    ("红利ETF",       "510880", None, "etf", None),
    ("创新药ETF",     "159992", None, "etf", None),
    ("医药ETF",       "512010", None, "etf", None),
    ("医疗ETF",       "512170", None, "etf", None),
    ("AI ETF",        "515070", None, "etf", None),
    ("半导体ETF",     "512480", None, "etf", None),
    ("半导体设备ETF", "159516", None, "etf", None),
    ("科创芯片ETF",   "588200", None, "etf", None),
    ("计算机ETF",     "159998", None, "etf", None),
    ("通信ETF",       "515880", None, "etf", None),
    ("消费电子ETF",   "159732", None, "etf", None),
    ("游戏ETF",       "516010", None, "etf", None),
    ("机器人ETF",     "159770", None, "etf", None),
    ("新能源ETF",     "516160", None, "etf", None),
    ("新能源车ETF",   "515030", None, "etf", None),
    ("光伏ETF",       "515790", None, "etf", None),
    ("电网ETF",       "159326", None, "etf", None),
    ("煤炭ETF",       "515220", None, "etf", None),
    ("消费ETF",       "159928", None, "etf", None),
    ("酒ETF",         "512690", None, "etf", None),
    ("家电ETF",       "159996", None, "etf", None),
    ("传媒ETF",       "512980", None, "etf", None),
    ("有色ETF",       "512400", None, "etf", None),
    ("化工ETF",       "159870", None, "etf", None),
    ("石油ETF",       "561360", None, "etf", None),
    ("建材ETF",       "159745", None, "etf", None),
    ("农业ETF",       "159825", None, "etf", None),
    ("军工ETF",       "512660", None, "etf", None),
    ("卫星ETF",       "159206", None, "etf", None),
    ("黄金ETF",       "518880", None, "etf", None),
    ("中概互联ETF",   "513050", None, "etf", None),
]

ALL_TARGETS = BROAD_INDICES + SECTOR_ETFS


C = {
    "price":"#1565C0","ma5":"#FF6F00","ma10":"#2E7D32","ma20":"#C62828","ma60":"#6A1B9A","ma200":"#37474F",
    "boll_upper":"#90CAF9","boll_mid":"#546E7A","boll_lower":"#90CAF9","boll_fill":"#E3F2FD",
    "vol_up":"#EF5350","vol_dn":"#26A69A","vol_ma5":"#FF6F00","vol_ma20":"#6A1B9A",
    "vol_ratio":"#888888","dif":"#FF6F00","dea":"#1565C0","hist_up":"#EF5350","hist_dn":"#26A69A",
    "kdj_k":"#1565C0","kdj_d":"#FF6F00","kdj_j":"#8E24AA",
    # 异常区间背景统一使用成交量面板红绿体系，通过 alpha 控制深浅。
    "kdj_overbought":"#EF5350","kdj_oversold":"#26A69A",
    "rsi6":"#5D4037","rsi12":"#00897B","rsi24":"#AD1457",
    "rsi_overbought":"#EF5350","rsi_oversold":"#26A69A",
    "fish_price":"#1565C0","fish_mas":"#FF6F00","fish_mal":"#C62828",
    "fish_golden":"#C62828","fish_death":"#2E7D32",
    "fish_vol_up":"#EF5350","fish_vol_dn":"#26A69A","fish_vmas":"#FF6F00",
    "fish_vmal":"#6A1B9A","fish_vratio":"#888888",
    # 鱼盆乖离率柱同样使用成交量红/绿，低分位用浅色，高分位用成交量面板主色。
    "dev_pos_lo":(1.0,0.92,0.92),"dev_pos_hi":(0.937,0.325,0.314),
    "dev_neg_lo":(0.91,0.98,0.96),"dev_neg_hi":(0.149,0.651,0.604),
}


def safe_fn(s: str) -> str:
    return re.sub(r'[\\/:*?"<>|]+', "_", str(s)).strip().replace(" ", "_")


def pick_col(df: pd.DataFrame, names: Iterable[str]) -> Optional[str]:
    for x in names:
        if x in df.columns:
            return x
    return None


def infer_date(df):  return pick_col(df, ["日期", "date", "Date", "时间"])
def infer_close(df): return pick_col(df, ["收盘", "收盘价", "close", "Close", "指数"])
def infer_open(df):  return pick_col(df, ["开盘", "开盘价", "open", "Open"])
def infer_high(df):  return pick_col(df, ["最高", "最高价", "high", "High"])
def infer_low(df):   return pick_col(df, ["最低", "最低价", "low", "Low"])
def infer_vol(df):   return pick_col(df, ["成交量", "volume", "Volume", "vol"])
def infer_amt(df):   return pick_col(df, ["成交额", "amount", "Amount", "成交额(元)"])
def infer_chg(df):   return pick_col(df, ["涨跌幅", "change_pct", "pct_chg"])


def set_date_index(df: pd.DataFrame) -> pd.DataFrame:
    dc = infer_date(df)
    if dc:
        df = df.copy()
        df[dc] = pd.to_datetime(df[dc], errors="coerce")
        df = df.dropna(subset=[dc]).sort_values(dc).set_index(dc)
    return df


def _fmtv(v, n=2):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "N/A"
    return f"{float(v):.{n}f}"


def _lerp(c1, c2, t: float):
    return tuple(c1[i] * (1 - t) + c2[i] * t for i in range(3))


def _exch(symbol: str) -> str:
    if symbol.startswith(("399", "159", "150")):
        return "sz"
    if symbol.startswith("899"):
        return "bj"
    return "sh"



def _weekly_tick_positions(index):
    """每周只取一个交易日作为刻度：优先周五，周五休盘则周四，否则取该周最后一个交易日。"""
    idx = pd.to_datetime(index)
    if len(idx) == 0:
        return [], []
    pos = pd.Series(range(len(idx)), index=idx)
    iso = idx.isocalendar()
    tick_pos, tick_lab = [], []
    for _, grp in pos.groupby([iso.year, iso.week]):
        week_dates = list(grp.index)
        chosen = None
        for weekday in (4, 3):  # Friday -> Thursday
            cand = [d for d in week_dates if d.weekday() == weekday]
            if cand:
                chosen = cand[-1]
                break
        if chosen is None:
            chosen = week_dates[-1]
        tick_pos.append(int(pos.loc[chosen]))
        tick_lab.append(pd.Timestamp(chosen).strftime("%y/%m/%d"))
    return tick_pos, tick_lab

def format_xaxis(ax, index=None, freq="W"):
    if index is None or len(index) == 0:
        return
    tick_pos, tick_lab = _weekly_tick_positions(index)
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_lab, rotation=45, ha="right", fontsize=7)
    ax.set_xlim(-0.8, len(index) - 0.2)


def _draw_candlestick(ax, x, open_, high, low, close, width=0.62):
    if open_ is None or high is None or low is None:
        ax.plot(x, close.values, color=C["price"], lw=1.0, label="收盘价")
        return
    for i in range(len(x)):
        o = float(open_.iloc[i]); h = float(high.iloc[i]); l = float(low.iloc[i]); c = float(close.iloc[i])
        color = C["vol_up"] if c >= o else C["vol_dn"]
        ax.vlines(x[i], l, h, color=color, linewidth=0.7, alpha=0.95)
        body_low = min(o, c)
        body_h = max(abs(c - o), 1e-6)
        rect = mpl.patches.Rectangle((x[i] - width/2, body_low), width, body_h,
                                     facecolor=color, edgecolor=color, linewidth=0.5, alpha=0.90)
        ax.add_patch(rect)


def _stage_extrema_points(high: pd.Series, low: pd.Series, window: int = 5, max_points: int = 4, min_gap: int = 8):
    """在展示窗口内找阶段性高低点，避免价格标注过密。"""
    if high is None or low is None or len(high) == 0:
        return [], []
    hs = pd.Series(high).astype(float).reset_index(drop=True)
    ls = pd.Series(low).astype(float).reset_index(drop=True)
    high_cands, low_cands = [], []
    for i in range(1, len(hs)-1):
        l = max(0, i-window); r = min(len(hs), i+window+1)
        if hs.iloc[i] >= hs.iloc[l:r].max():
            high_cands.append((i, hs.iloc[i]))
        if ls.iloc[i] <= ls.iloc[l:r].min():
            low_cands.append((i, ls.iloc[i]))
    def pick(cands, reverse_value=True):
        # 先按极值强度取，再按时间排序显示
        sorted_cands = sorted(cands, key=lambda x: x[1], reverse=reverse_value)
        picked = []
        for i, v in sorted_cands:
            if all(abs(i-j) >= min_gap for j, _ in picked):
                picked.append((i, v))
            if len(picked) >= max_points:
                break
        return sorted(picked, key=lambda x: x[0])
    return pick(high_cands, True), pick(low_cands, False)

def annotate_stage_extrema(ax, x, high: pd.Series, low: pd.Series, kind="etf"):
    dec = 3 if kind == "etf" else 2
    highs, lows = _stage_extrema_points(high, low)
    for i, v in highs:
        ax.annotate(f"{v:.{dec}f}", xy=(x[i], v), xytext=(0, 7), textcoords="offset points",
                    ha="center", va="bottom", fontsize=7, color="#C62828")
    for i, v in lows:
        ax.annotate(f"{v:.{dec}f}", xy=(x[i], v), xytext=(0, -9), textcoords="offset points",
                    ha="center", va="top", fontsize=7, color="#2E7D32")


def _is_market_closed(now=None):
    now = now or datetime.now()
    hh, mm = MARKET_CLOSE_HHMM
    return (now.hour, now.minute) >= (hh, mm)


def select_display_window(df: pd.DataFrame, bars: int = DISPLAY_BARS, now=None) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    now = now or datetime.now()
    out = df.copy().sort_index()
    today = pd.Timestamp(now.date())
    out = out[out.index.normalize() <= today]
    if not _is_market_closed(now) and not out.empty and out.index[-1].normalize() == today:
        out = out.iloc[:-1]
    return out.iloc[-bars:] if len(out) > bars else out


def _dev_color(dev_v: float, pct_v: float):
    if pd.isna(dev_v):
        return "#CFD8DC"
    t = 0 if pd.isna(pct_v) else max(0, min(1, pct_v / 100.0))
    rgb = _lerp(C["dev_pos_lo"], C["dev_pos_hi"], t) if dev_v >= 0 else _lerp(C["dev_neg_lo"], C["dev_neg_hi"], t)
    return (*rgb, 1.0)


def _normalize_ohlc(df_raw: pd.DataFrame) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    df = df_raw.copy()
    rename_map = {
        "date": "日期", "Date": "日期",
        "open": "开盘", "Open": "开盘",
        "close": "收盘", "Close": "收盘",
        "high": "最高", "High": "最高",
        "low": "最低", "Low": "最低",
        "volume": "成交量", "Volume": "成交量",
        "amount": "成交额", "Amount": "成交额",
        "pct_chg": "涨跌幅", "change_pct": "涨跌幅",
    }
    df = df.rename(columns=rename_map)
    if infer_date(df) is None and len(df.columns) > 0:
        df = df.rename(columns={df.columns[0]: "日期"})
    df = set_date_index(df)
    if df.empty:
        return df
    for c in [infer_open(df), infer_high(df), infer_low(df), infer_close(df), infer_vol(df), infer_amt(df), infer_chg(df)]:
        if c and c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df.sort_index()


def _fix_split_daily(df_daily: pd.DataFrame, name="") -> pd.DataFrame:
    cc = infer_close(df_daily)
    if not cc or len(df_daily) < 10:
        return df_daily
    df = df_daily.copy()
    price_cols = [c for c in [cc, infer_open(df), infer_high(df), infer_low(df)] if c and c in df.columns]
    for _ in range(5):
        close = df[cc].astype(float)
        drops = close.pct_change()[lambda s: s < -0.35]
        if drops.empty:
            break
        split_date = drops.index[0]
        pos = df.index.get_loc(split_date)
        if pos < 3:
            break
        pre = float(close.iloc[max(0, pos-5): pos].mean())
        post = float(close.iloc[pos: min(len(close), pos+5)].mean())
        if pre <= 0:
            break
        raw = post / pre
        factor = raw
        for frac in [0.5, 1/3, 0.25, 0.2, 2/3, 0.75, 0.8]:
            if abs(raw - frac) / frac < 0.08:
                factor = frac
                break
        print(f"    [ADJUST] {name}: 折算日 {split_date.date()} raw={raw:.4f} factor={factor:.4f}")
        mask = df.index < split_date
        for col in price_cols:
            df.loc[mask, col] = df.loc[mask, col].astype(float) * factor
    return df


def _require_akshare():
    if ak is None:
        raise ImportError("akshare 未安装或不可用，请先 pip install akshare")


def _fetch_index_close_from_legu(symbol: str) -> pd.DataFrame:
    try:
        import requests as req_lib
        import py_mini_racer
        import akshare.stock_feature.stock_a_pe_and_pb as legu_m
        code_map = {"932000": "932000.CSI", "399006": "399006.SZ", "399001": "399001.SZ", "000688": "000688.SH"}
        if symbol not in code_map:
            return pd.DataFrame()
        js = py_mini_racer.MiniRacer()
        js.eval(legu_m.hash_code)
        token = js.call("hex", datetime.now().date().isoformat()).lower()
        url = "https://legulegu.com/api/stockdata/index-basic-pe"
        r = req_lib.get(url, params={"token": token, "indexCode": code_map[symbol]},
                        **legu_m.get_cookie_csrf(url="https://legulegu.com/stockdata/sz50-ttm-lyr"), timeout=12)
        data = r.json()
        if not data.get("data"):
            return pd.DataFrame()
        df = pd.DataFrame(data["data"])
        if "close" not in df.columns:
            return pd.DataFrame()
        df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_convert("Asia/Shanghai").dt.tz_localize(None)
        df = df[["date", "close"]].rename(columns={"date": "日期", "close": "收盘"})
        df = _normalize_ohlc(df)
        return df[df.index >= pd.Timestamp(KLINE_START)]
    except Exception as e:
        print(f"    [WARN] 乐咕日频兜底({symbol}): {e}")
        return pd.DataFrame()


def fetch_kline_index(symbol: str, name="") -> pd.DataFrame:
    _require_akshare()
    start_ts = pd.Timestamp(KLINE_START)
    em_sym_map = {"932000": "csi932000", "000001": "sh000001", "399006": "sz399006", "399001": "sz399001", "000688": "sh000688", "899050": "bj899050"}
    em_sym = em_sym_map.get(symbol, f"{_exch(symbol)}{symbol}")
    try:
        df = ak.stock_zh_index_daily_em(symbol=em_sym, start_date=KLINE_START, end_date=KLINE_END)
        df = _normalize_ohlc(df)
        df = df[df.index >= start_ts]
        if not df.empty:
            print(f"    [SRC-IDX] 东方财富指数日线（{len(df)}日）")
            return df
    except Exception as e:
        print(f"    [WARN] 东方财富指数({symbol}): {type(e).__name__}")
    try:
        df = ak.stock_zh_index_daily(symbol=f"{_exch(symbol)}{symbol}")
        df = _normalize_ohlc(df)
        df = df[df.index >= start_ts]
        if not df.empty:
            print(f"    [SRC-IDX] 新浪指数日线（{len(df)}日）")
            return df
    except Exception as e:
        print(f"    [WARN] 新浪指数({symbol}): {type(e).__name__}")
    tx_prefix = {"399001": "sz", "399006": "sz", "899050": "bj", "000688": "sh", "000001": "sh", "000016": "sh", "000300": "sh", "000905": "sh", "000852": "sh"}.get(symbol, _exch(symbol))
    try:
        df = ak.stock_zh_index_daily_tx(symbol=f"{tx_prefix}{symbol}")
        df = _normalize_ohlc(df)
        df = df[df.index >= start_ts]
        if not df.empty:
            print(f"    [SRC-IDX] 腾讯指数日线（{len(df)}日）")
            return df
    except Exception as e:
        print(f"    [WARN] 腾讯指数({symbol}): {type(e).__name__}")
    df = _fetch_index_close_from_legu(symbol)
    if not df.empty:
        print(f"    [SRC-IDX] 乐咕 close-only 日线（{len(df)}日）")
        return df
    print(f"    [WARN] 指数K线({symbol}) 所有数据源均失败")
    return pd.DataFrame()


def _fetch_etf_nav_hybrid_daily(symbol: str, name="") -> pd.DataFrame:
    _require_akshare()
    start_ts = pd.Timestamp(KLINE_START)
    try:
        nav = ak.fund_open_fund_info_em(fund=symbol, indicator="单位净值走势", period="成立来")
        nav = nav.rename(columns={"净值日期": "日期", "单位净值": "收盘"})
        nav = _normalize_ohlc(nav)[["收盘"]]
        nav = nav[nav.index >= start_ts]
        if nav.empty:
            raise ValueError("NAV为空")
    except Exception:
        return pd.DataFrame()
    sina = pd.DataFrame()
    try:
        df_s = ak.fund_etf_hist_sina(symbol=f"{_exch(symbol)}{symbol}")
        if df_s is not None and not df_s.empty:
            df_s = df_s.rename(columns={"date": "日期", "open": "开盘", "high": "最高", "low": "最低", "close": "收盘_sina", "volume": "成交量"})
            sina = set_date_index(df_s)
            sina = sina[sina.index >= start_ts]
            for col in ["开盘", "最高", "最低", "收盘_sina", "成交量"]:
                if col in sina.columns:
                    sina[col] = pd.to_numeric(sina[col], errors="coerce")
    except Exception:
        pass
    if sina.empty:
        nav["涨跌幅"] = nav["收盘"].pct_change() * 100
        return nav.dropna(how="all")
    merged = nav.join(sina, how="left")
    if "收盘_sina" in merged.columns:
        scale = (merged["收盘"] / merged["收盘_sina"].replace(0, np.nan)).fillna(1.0).clip(0.05, 20.0)
        for col in ["开盘", "最高", "最低"]:
            if col in merged.columns:
                merged[col] = merged[col] * scale
    if "成交量" in merged.columns:
        merged["成交额"] = merged["收盘"] * merged["成交量"]
    merged["涨跌幅"] = merged["收盘"].pct_change() * 100
    return merged[merged["收盘"].notna()]


def fetch_kline_etf(symbol: str, name="") -> pd.DataFrame:
    _require_akshare()
    start_ts = pd.Timestamp(KLINE_START)
    try:
        df = ak.fund_etf_hist_em(symbol=symbol, period="daily", adjust="qfq", start_date=KLINE_START, end_date=KLINE_END)
        df = _normalize_ohlc(df)
        df = df[df.index >= start_ts]
        if not df.empty:
            print(f"    [SRC-ETF] 东方财富ETF日线（{len(df)}日）")
            return df
    except Exception as e:
        print(f"    [WARN] 东方财富ETF({symbol}): {type(e).__name__}")
    df_nav = _fetch_etf_nav_hybrid_daily(symbol, name)
    if not df_nav.empty:
        print(f"    [SRC-ETF] NAV混合日线（{len(df_nav)}日）")
        return df_nav
    try:
        df = ak.fund_etf_hist_sina(symbol=f"{_exch(symbol)}{symbol}")
        df = df.rename(columns={"date": "日期", "open": "开盘", "close": "收盘", "high": "最高", "low": "最低", "volume": "成交量"})
        df = _normalize_ohlc(df)
        df = df[df.index >= start_ts]
        df = _fix_split_daily(df, name)
        if infer_amt(df) is None and infer_close(df) and infer_vol(df):
            df["成交额"] = df[infer_close(df)] * df[infer_vol(df)]
        if infer_chg(df) is None and infer_close(df):
            df["涨跌幅"] = df[infer_close(df)].pct_change() * 100
        if not df.empty:
            print(f"    [SRC-ETF] 新浪备用ETF日线（{len(df)}日）")
            return df
    except Exception as e:
        print(f"    [WARN] 新浪备用ETF({symbol}): {e}")
    print(f"    [WARN] ETF K线({symbol}) 所有数据源均失败")
    return pd.DataFrame()


def fetch_kline(symbol: str, kind: str, name="") -> pd.DataFrame:
    return fetch_kline_index(symbol, name) if kind == "index" else fetch_kline_etf(symbol, name)


def _cache_file_path(symbol: str, kind: str) -> str:
    folder = os.path.join(CACHE_DIR, kind)
    os.makedirs(folder, exist_ok=True)
    return os.path.join(folder, f"{symbol}.csv")


def _read_cached_kline(symbol: str, kind: str) -> pd.DataFrame:
    path = _cache_file_path(symbol, kind)
    if not os.path.exists(path):
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        df = _normalize_ohlc(df)
        if not df.empty:
            print(f"    [CACHE] 读取本地历史K线（{len(df)}日）")
        return df
    except Exception as e:
        print(f"    [WARN] 读取缓存失败({symbol}): {e}")
        return pd.DataFrame()


def _write_cached_kline(symbol: str, kind: str, df: pd.DataFrame):
    if df is None or df.empty:
        return
    path = _cache_file_path(symbol, kind)
    out = df.copy().sort_index()
    out = out[~out.index.duplicated(keep="last")]
    out.reset_index().rename(columns={out.index.name or "index": "日期"}).to_csv(path, index=False, encoding="utf-8-sig")


def _fetch_online_with_start(symbol: str, kind: str, name: str, start_yyyymmdd: str) -> pd.DataFrame:
    global KLINE_START
    old_start = KLINE_START
    try:
        KLINE_START = start_yyyymmdd
        return fetch_kline(symbol, kind, name)
    finally:
        KLINE_START = old_start


def fetch_kline_cached(symbol: str, kind: str, name="") -> pd.DataFrame:
    """优先读取本地缓存；缓存为空则全量获取；缓存已有则向前回看若干天增量更新。"""
    if not CACHE_ENABLED:
        return fetch_kline(symbol, kind, name)
    cached = _read_cached_kline(symbol, kind)
    if cached.empty:
        df = fetch_kline(symbol, kind, name)
        if not df.empty:
            _write_cached_kline(symbol, kind, df)
            print(f"    [CACHE] 已写入本地历史K线（{len(df)}日）")
        return df
    latest = cached.index.max()
    inc_start = (pd.Timestamp(latest) - pd.Timedelta(days=CACHE_INCREMENT_LOOKBACK_DAYS)).strftime("%Y%m%d")
    inc = _fetch_online_with_start(symbol, kind, name, inc_start)
    if inc is not None and not inc.empty:
        merged = pd.concat([cached, inc]).sort_index()
        merged = merged[~merged.index.duplicated(keep="last")]
        _write_cached_kline(symbol, kind, merged)
        added = max(len(merged) - len(cached), 0)
        print(f"    [CACHE] 增量更新完成：新增/修正 {added} 日，缓存共 {len(merged)} 日")
        return merged
    print("    [CACHE] 增量源失败，使用本地缓存继续运行")
    return cached



def calc_ma(s, windows=MA_WINDOWS):
    return {f"MA{w}": s.rolling(w, min_periods=1).mean() for w in windows}


def calc_bollinger(close, window=20, n_std=2):
    mid = close.rolling(window, min_periods=1).mean()
    std = close.rolling(window, min_periods=1).std(ddof=0)
    upper = mid + n_std * std
    lower = mid - n_std * std
    return upper, mid, lower


def calc_macd(close, fast=12, slow=26, signal=9):
    ef = close.ewm(span=fast, adjust=False).mean(); es = close.ewm(span=slow, adjust=False).mean()
    dif = ef - es; dea = dif.ewm(span=signal, adjust=False).mean(); hist = (dif - dea) * 2
    return dif, dea, hist


def calc_kdj(high, low, close, n=9):
    low_n = low.rolling(n, min_periods=1).min(); high_n = high.rolling(n, min_periods=1).max()
    rsv = ((close - low_n) / (high_n - low_n).replace(0, np.nan) * 100).fillna(50.0)
    k = rsv.ewm(com=2, adjust=False).mean(); d = k.ewm(com=2, adjust=False).mean(); j = 3 * k - 2 * d
    return k, d, j


def calc_rsi(close, period=14):
    d = close.diff(); au = d.clip(lower=0).ewm(span=period, adjust=False).mean(); ad = (-d.clip(upper=0)).ewm(span=period, adjust=False).mean()
    return 100 - 100 / (1 + au / ad.replace(0, np.nan))


def calc_volume_ratio(vol, ma_window=5):
    """计算成交量均量和量比。

    量比 = 当日成交量 / N日均量。这里不做单位换算，不混用成交额，
    所有均量都直接由同一条“成交量”序列 rolling.mean() 得到，
    避免因指数/ETF数据源单位不同导致均量和成交量不在同一口径。
    """
    vol = pd.to_numeric(vol, errors="coerce").replace([np.inf, -np.inf], np.nan)
    vol = vol.where(vol >= 0)
    vma = vol.rolling(ma_window, min_periods=1).mean()
    ratio = vol / vma.replace(0, np.nan)
    return vma, ratio


def calc_volume_indicators(vol):
    """统一输出 MA5均量、MA20均量、量比。

    后续所有图表、鱼盆数据表、快照表都只使用这里生成的三列，
    确保“成交量、MA5均量、MA20均量、量比”完全同源。
    """
    vol = pd.to_numeric(vol, errors="coerce").replace([np.inf, -np.inf], np.nan)
    vol = vol.where(vol >= 0)
    vma5 = vol.rolling(5, min_periods=1).mean()
    vma20 = vol.rolling(20, min_periods=1).mean()
    ratio = vol / vma5.replace(0, np.nan)
    return vma5, vma20, ratio


def normalize_volume_units(vol: pd.Series, kind: str = "", symbol: str = "", amount: Optional[pd.Series] = None) -> pd.Series:
    """统一成交量内部口径为“手”。

    v1.22口径：
    - 所有宽基指数：历史每日成交量统一除以100；
    - 所有行业ETF：历史每日成交量统一除以100；
    - 0或负数视为缺失，避免参与均量和量比计算；
    - amount参数仅保留兼容旧调用，不再用成交额/成交量自动猜单位。
    """
    idx = vol.index if hasattr(vol, "index") else None
    v = pd.to_numeric(vol, errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float64")
    v = v.where(v > 0)
    if not v.notna().any():
        return pd.Series(np.nan, index=idx, dtype="float64")
    scale = 100.0 if kind in {"index", "etf"} else 1.0
    return (v / scale).astype("float64")


def _assert_volume_indicators(out: pd.DataFrame, name: str = ""):
    """内部一致性检查：均量必须等于成交量自身的滚动均值。"""
    try:
        vol = pd.to_numeric(out["成交量"], errors="coerce")
        exp5 = vol.rolling(5, min_periods=1).mean()
        exp20 = vol.rolling(20, min_periods=1).mean()
        if "MA5均量" in out.columns:
            diff5 = (pd.to_numeric(out["MA5均量"], errors="coerce") - exp5).abs().dropna()
            if not diff5.empty and diff5.max() > max(1e-6, exp5.dropna().abs().max() * 1e-10):
                print(f"    [WARN] {name} MA5均量一致性检查异常，已按成交量重新计算")
                out["MA5均量"] = exp5
        if "MA20均量" in out.columns:
            diff20 = (pd.to_numeric(out["MA20均量"], errors="coerce") - exp20).abs().dropna()
            if not diff20.empty and diff20.max() > max(1e-6, exp20.dropna().abs().max() * 1e-10):
                print(f"    [WARN] {name} MA20均量一致性检查异常，已按成交量重新计算")
                out["MA20均量"] = exp20
        out["量比"] = vol / pd.to_numeric(out["MA5均量"], errors="coerce").replace(0, np.nan)
    except Exception as e:
        print(f"    [WARN] 成交量指标一致性检查跳过: {e}")
    return out


def calc_deviation(close, ma_long):
    return (close - ma_long) / ma_long.replace(0, np.nan) * 100


def calc_dev_pct(dev, window=None):
    """计算 MA20 乖离率全历史分位数（正负乖离分开）。

    当前口径为“全历史静态排序分位数”：
    - 每次拿到完整历史数据后，先计算全部历史每个交易日的 MA20 乖离率；
    - 正乖离样本单独排名，数值越大，分位数越高，表示超买强度；
    - 负乖离样本单独按绝对值排名，负得越深，分位数越高，表示超卖强度；
    - 若多个交易日乖离率完全相同，使用 average rank 取平均分位数。

    因此，只有全历史唯一最大正乖离/最深负乖离才会得到 100；
    如果最大值存在重复，重复值会共享平均分位数，不会全部硬记为100。
    """
    dev = pd.to_numeric(dev, errors="coerce").replace([np.inf, -np.inf], np.nan)
    res = pd.Series(np.nan, index=dev.index, dtype="float64")

    pos = dev[dev > 0].dropna()
    if not pos.empty:
        # method="average"：重复值使用平均排名；pct=True：返回0-1分位。
        res.loc[pos.index] = pos.rank(method="average", pct=True) * 100

    neg = dev[dev < 0].dropna()
    if not neg.empty:
        # 负乖离越负越极端，所以用绝对值/相反数排名。
        neg_strength = -neg
        res.loc[neg.index] = neg_strength.rank(method="average", pct=True) * 100

    # 乖离率恰好等于0时，认为没有超买/超卖强度。
    zero_idx = dev[dev == 0].index
    if len(zero_idx) > 0:
        res.loc[zero_idx] = 0.0
    return res


def calc_signed_dev_pct(dev: pd.Series, pct: pd.Series) -> pd.Series:
    """将0-100强度分位数转换为右轴使用的有符号分位数：正乖离+0~+100，负乖离-0~-100。"""
    dev = pd.to_numeric(dev, errors="coerce")
    pct = pd.to_numeric(pct, errors="coerce")
    signed = pct.where(dev >= 0, -pct)
    return signed.where(dev.notna() & pct.notna())


def calc_dev_reference_lines(dev: pd.Series) -> dict:
    """基于全历史MA20乖离率，计算正/负乖离20%与80%参考线。"""
    dev = pd.to_numeric(dev, errors="coerce").dropna()
    pos = dev[dev >= 0]
    neg = dev[dev < 0]
    out = {"pos20": np.nan, "pos80": np.nan, "neg20": np.nan, "neg80": np.nan}
    if len(pos) >= 3:
        out["pos20"] = float(np.percentile(pos, 20))
        out["pos80"] = float(np.percentile(pos, 80))
    if len(neg) >= 3:
        # 负乖离越负越极端：neg20靠近0，neg80更深。
        out["neg20"] = float(np.percentile(neg, 80))
        out["neg80"] = float(np.percentile(neg, 20))
    return out


def dev_status_label(last_dev, last_signed_pct):
    """根据有符号分位数输出鱼盆面板右上角状态。"""
    if pd.isna(last_dev) or pd.isna(last_signed_pct):
        return "N/A", "#37474F"
    if last_signed_pct >= 80:
        return "↑严重超买", "#B71C1C"
    if last_signed_pct >= 20:
        return "↑超买", "#E65100"
    if last_signed_pct <= -80:
        return "↓严重超卖", "#1B5E20"
    if last_signed_pct <= -20:
        return "↓超卖", "#00695C"
    return "中性", "#37474F"


def find_crossovers(ms, ml):
    cr = (ms > ml).astype(int).diff()
    return cr[cr == 1].index, cr[cr == -1].index


def _price_direction_factor(open_, high, low, close):
    span = (high - low).replace(0, np.nan)
    direction = ((close - open_) / span).replace([np.inf, -np.inf], np.nan)
    fallback = np.sign(close.diff())
    fallback = fallback.where(fallback != 0, np.sign(close - open_))
    direction = direction.fillna(fallback).clip(-1, 1)
    return direction.fillna(0.0)



def prepare_analysis_df(df_raw: pd.DataFrame, kind="etf", benchmark_close: Optional[pd.Series] = None) -> pd.DataFrame:
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    df = df_raw.copy().sort_index()
    cc, oc, hc, lc, vc, ac, gc = infer_close(df), infer_open(df), infer_high(df), infer_low(df), infer_vol(df), infer_amt(df), infer_chg(df)
    if cc is None:
        return pd.DataFrame()
    out = pd.DataFrame(index=df.index)
    out["收盘"] = pd.to_numeric(df[cc], errors="coerce")
    out["开盘"] = pd.to_numeric(df[oc], errors="coerce") if oc and oc in df.columns else out["收盘"]
    out["最高"] = pd.to_numeric(df[hc], errors="coerce") if hc and hc in df.columns else out[["开盘", "收盘"]].max(axis=1)
    out["最低"] = pd.to_numeric(df[lc], errors="coerce") if lc and lc in df.columns else out[["开盘", "收盘"]].min(axis=1)
    out["成交量"] = normalize_volume_units(pd.to_numeric(df[vc], errors="coerce"), kind=kind, symbol="", amount=(pd.to_numeric(df[ac], errors="coerce") if ac and ac in df.columns else None)) if vc and vc in df.columns else pd.Series(np.nan, index=out.index)
    out["成交额"] = pd.to_numeric(df[ac], errors="coerce") if ac and ac in df.columns else out["收盘"] * out["成交量"]
    out["涨跌幅"] = pd.to_numeric(df[gc], errors="coerce") if gc and gc in df.columns else out["收盘"].pct_change() * 100
    for k, v in calc_ma(out["收盘"]).items():
        out[k] = v
    bu, bm, bl = calc_bollinger(out["收盘"])
    out["BOLL上轨"], out["BOLL中轨"], out["BOLL下轨"] = bu, bm, bl
    dif, dea, hist = calc_macd(out["收盘"])
    out["DIF"], out["DEA"], out["MACD柱"] = dif, dea, hist
    k, d, j = calc_kdj(out["最高"], out["最低"], out["收盘"])
    out["K"], out["D"], out["J"] = k, d, j
    out["RSI6"], out["RSI12"], out["RSI24"] = calc_rsi(out["收盘"], 6), calc_rsi(out["收盘"], 12), calc_rsi(out["收盘"], 24)
    if out["成交量"].notna().any():
        vma5, vma20, vr = calc_volume_indicators(out["成交量"])
    else:
        vma5 = pd.Series(np.nan, index=out.index)
        vma20 = pd.Series(np.nan, index=out.index)
        vr = pd.Series(np.nan, index=out.index)
    out["MA5均量"], out["MA20均量"], out["量比"] = vma5, vma20, vr
    out = _assert_volume_indicators(out)
    out["近5日涨跌(%)"], out["近20日涨跌(%)"] = out["收盘"].pct_change(5) * 100, out["收盘"].pct_change(20) * 100
    out["MA5乖离率(%)"] = calc_deviation(out["收盘"], out["MA5"])
    out["MA20乖离率(%)"] = calc_deviation(out["收盘"], out["MA20"])
    # MA20乖离率分位数基于该标的全历史日线计算，而不是近100日展示窗口。
    out["MA20乖离率分位(%)"] = calc_dev_pct(out["MA20乖离率(%)"])
    out["MA20乖离率有符号分位(%)"] = calc_signed_dev_pct(out["MA20乖离率(%)"], out["MA20乖离率分位(%)"])
    golden, death = find_crossovers(out["MA5"], out["MA20"])
    cross = pd.Series("", index=out.index)
    for dt in golden:
        if dt in cross.index:
            cross.loc[dt] = "金叉↑"
    for dt in death:
        if dt in cross.index:
            cross.loc[dt] = "死叉↓"
    out["MA5-MA20交叉"] = cross
    if kind == "etf":
        if benchmark_close is not None and not benchmark_close.empty:
            bench = benchmark_close.reindex(out.index).ffill()
            bench_ret = bench.pct_change() * 100
            out["沪深300涨跌幅(%)"] = bench_ret
            out["超额收益(%)"] = out["涨跌幅"] - bench_ret
        direction = _price_direction_factor(out["开盘"], out["最高"], out["最低"], out["收盘"])
        abnormal_amt = (out["成交额"] - out["成交额"].rolling(5, min_periods=1).mean()).clip(lower=0)
        out["资金净流入估算(亿元)"] = abnormal_amt * direction / 1e8
        out["20日滚动波动率(%)"] = out["收盘"].pct_change().rolling(20, min_periods=20).std() * 100
    return out.dropna(how="all")



def plot_daily_kline(name, symbol, df, save_dir, kind="etf"):
    df = select_display_window(df, DISPLAY_BARS)
    if df.empty:
        print(f"    [SKIP] {name} 日K数据为空")
        return None
    dates = df.index
    x = np.arange(len(df))
    close = df["收盘"]
    open_ = df["开盘"] if "开盘" in df.columns else None
    high = df["最高"] if "最高" in df.columns else None
    low = df["最低"] if "最低" in df.columns else None
    chg = df["涨跌幅"]
    vol = df["成交量"] if "成交量" in df.columns else pd.Series(np.nan, index=dates)
    vol_ma5 = df["MA5均量"] if "MA5均量" in df.columns else pd.Series(np.nan, index=dates)
    vr = df["量比"] if "量比" in df.columns else pd.Series(np.nan, index=dates)
    bu = df["BOLL上轨"] if "BOLL上轨" in df.columns else pd.Series(np.nan, index=dates)
    bm = df["BOLL中轨"] if "BOLL中轨" in df.columns else pd.Series(np.nan, index=dates)
    bl = df["BOLL下轨"] if "BOLL下轨" in df.columns else pd.Series(np.nan, index=dates)
    display_title = _target_display_title(name, symbol, kind)
    fn_prefix = _target_fn_prefix(name, symbol, kind)
    fig = plt.figure(figsize=(22, 28))
    fig.patch.set_facecolor("#FAFAFA")
    fig.suptitle(f"{display_title}   日K线技术分析    截至{TODAY.strftime('%Y年%m月%d日')}   （近{len(df)}个交易日）", fontsize=13, fontweight="bold", y=0.999)
    gs = gridspec.GridSpec(5, 1, figure=fig, height_ratios=[4, 1.6, 1.6, 1.1, 1.1], hspace=0.24)
    ax_p = fig.add_subplot(gs[0]); ax_v = fig.add_subplot(gs[1], sharex=ax_p); ax_mc = fig.add_subplot(gs[2], sharex=ax_p); ax_kdj = fig.add_subplot(gs[3], sharex=ax_p); ax_rsi = fig.add_subplot(gs[4], sharex=ax_p)

    ax_p.set_facecolor("#F8F9FA")
    ax_p.fill_between(x, bl.values, bu.values, alpha=0.12, color=C["boll_fill"])
    ax_p.plot(x, bu.values, color=C["boll_upper"], lw=0.9, ls="--", label="BOLL上轨")
    ax_p.plot(x, bm.values, color=C["boll_mid"], lw=0.9, ls="--", label="BOLL中轨")
    ax_p.plot(x, bl.values, color=C["boll_lower"], lw=0.9, ls="--", label="BOLL下轨")
    _draw_candlestick(ax_p, x, open_, high, low, close)
    if high is not None and low is not None:
        annotate_stage_extrema(ax_p, x, high, low, kind)
    for w, ck in zip([5, 10, 20, 60, 200], ["ma5", "ma10", "ma20", "ma60", "ma200"]):
        col = f"MA{w}"
        if col in df.columns:
            ax_p.plot(x, df[col].values, lw=1.0, color=C[ck], label=col)
    ax_p.set_ylabel("价格", fontsize=9)
    ax_p.legend(loc="upper left", fontsize=7, ncol=5, framealpha=0.88)
    ax_p.grid(True, alpha=0.22, ls="--")
    fc = "#B71C1C" if float(chg.iloc[-1]) >= 0 else "#1B5E20"
    info = (
        f"收盘:{_fmtv(close.iloc[-1],3)}  日涨跌:{_fmtv(chg.iloc[-1],2)}%\n"
        f"MA5:{_fmtv(df['MA5'].iloc[-1],3)}  MA20:{_fmtv(df['MA20'].iloc[-1],3)}  MA200:{_fmtv(df['MA200'].iloc[-1],3)}\n"
        f"BOLL上:{_fmtv(bu.iloc[-1],3)} 中:{_fmtv(bm.iloc[-1],3)} 下:{_fmtv(bl.iloc[-1],3)}"
    )
    ax_p.text(0.995, 0.97, info, transform=ax_p.transAxes, fontsize=8, ha="right", va="top", color=fc,
              bbox=dict(boxstyle="round,pad=0.35", fc="white", ec=fc, alpha=0.62))
    format_xaxis(ax_p, dates, "W")

    ax_v.set_facecolor("#F8F9FA")
    if vol.notna().any() and vol.fillna(0).sum() > 0:
        bc = [C["vol_up"] if c >= 0 else C["vol_dn"] for c in chg.fillna(0)]
        dv, unit = _get_volume_display_unit(kind, vol.max(skipna=True))
        ax_v.bar(x, vol.values / dv, color=bc, alpha=0.65, width=0.7, label=f"成交量({unit})")
        ax_v.plot(x, vol_ma5.values / dv, color=C["vol_ma5"], lw=1.0, label=f"量MA5({unit})")
        if "MA20均量" in df.columns:
            ax_v.plot(x, df["MA20均量"].values / dv, color=C["vol_ma20"], lw=1.0, label=f"量MA20({unit})")
        ax_v.set_ylabel(f"成交量({unit})", fontsize=8)
        ax_v.legend(loc="upper left", fontsize=7, framealpha=0.88)
        avr = ax_v.twinx()
        avr.plot(x, vr.values, color=C["vol_ratio"], lw=1.1, alpha=0.70, label="量比(量/MA5)")
        avr.axhline(1, color=C["vol_ratio"], lw=0.5, ls="--", alpha=0.40)
        avr.set_ylabel("量比", fontsize=7, color=C["vol_ratio"])
        avr.tick_params(axis="y", labelcolor=C["vol_ratio"], labelsize=6)
        avr.set_ylim(0, max((vr.dropna().quantile(0.99) if vr.notna().any() else 2.0) * 1.15, 3.0))
        v_info = f"量:{_fmtv(vol.iloc[-1]/dv,2)}{unit}  MA5:{_fmtv(vol_ma5.iloc[-1]/dv,2)}{unit}"
        if "MA20均量" in df.columns:
            v_info += f"  MA20:{_fmtv(df['MA20均量'].iloc[-1]/dv,2)}{unit}"
        v_info += f"  量比:{_fmtv(vr.iloc[-1],2)}"
        ax_v.text(0.995, 0.96, v_info, transform=ax_v.transAxes, fontsize=7.5, ha="right", va="top", color=C["vol_ratio"],
                  bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C["vol_ratio"], alpha=0.62))
    ax_v.grid(True, alpha=0.20, ls="--")
    format_xaxis(ax_v, dates, "W")

    ax_mc.set_facecolor("#F8F9FA")
    ax_mc.plot(x, df["DIF"].values, C["dif"], lw=1.2, label="DIF")
    ax_mc.plot(x, df["DEA"].values, C["dea"], lw=1.2, label="DEA")
    ax_mc.bar(x, df["MACD柱"].values, color=[C["hist_up"] if v >= 0 else C["hist_dn"] for v in df["MACD柱"].fillna(0)], alpha=0.65, width=0.7, label="MACD柱")
    ax_mc.axhline(0, color="#455A64", lw=0.6)
    ax_mc.set_ylabel("MACD", fontsize=8)
    ax_mc.legend(loc="upper left", fontsize=7, framealpha=0.88)
    ax_mc.grid(True, alpha=0.20, ls="--")
    ax_mc.text(0.995, 0.96, f"DIF:{_fmtv(df['DIF'].iloc[-1],2)}  DEA:{_fmtv(df['DEA'].iloc[-1],2)}  柱:{_fmtv(df['MACD柱'].iloc[-1],2)}", transform=ax_mc.transAxes, fontsize=7.5, ha="right", va="top", color=C["dif"], bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C["dif"], alpha=0.62))
    format_xaxis(ax_mc, dates, "W")

    ax_kdj.set_facecolor("#F8F9FA")
    ax_kdj.axhspan(80, 120, facecolor=C["kdj_overbought"], alpha=0.10, zorder=0)
    ax_kdj.axhspan(-20, 20, facecolor=C["kdj_oversold"], alpha=0.10, zorder=0)
    ax_kdj.plot(x, df["K"].values, color=C["kdj_k"], lw=1.1, label="K")
    ax_kdj.plot(x, df["D"].values, color=C["kdj_d"], lw=1.1, label="D")
    ax_kdj.plot(x, df["J"].values, color=C["kdj_j"], lw=1.1, label="J")
    ax_kdj.axhline(80, color="#B0BEC5", lw=0.6, ls="--"); ax_kdj.axhline(20, color="#B0BEC5", lw=0.6, ls="--")
    ax_kdj.set_ylabel("KDJ", fontsize=8)
    ax_kdj.legend(loc="upper left", fontsize=7, framealpha=0.88)
    ax_kdj.grid(True, alpha=0.20, ls="--")
    ax_kdj.text(0.995, 0.96, f"K:{_fmtv(df['K'].iloc[-1],1)}  D:{_fmtv(df['D'].iloc[-1],1)}  J:{_fmtv(df['J'].iloc[-1],1)}", transform=ax_kdj.transAxes, fontsize=7.5, ha="right", va="top", color=C["kdj_j"], bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C["kdj_j"], alpha=0.62))
    format_xaxis(ax_kdj, dates, "W")

    ax_rsi.set_facecolor("#F8F9FA")
    ax_rsi.axhspan(70, 100, facecolor=C["rsi_overbought"], alpha=0.10, zorder=0)
    ax_rsi.axhspan(0, 30, facecolor=C["rsi_oversold"], alpha=0.10, zorder=0)
    ax_rsi.plot(x, df["RSI6"].values, color=C["rsi6"], lw=1.1, label="RSI6")
    ax_rsi.plot(x, df["RSI12"].values, color=C["rsi12"], lw=1.1, label="RSI12")
    ax_rsi.plot(x, df["RSI24"].values, color=C["rsi24"], lw=1.1, label="RSI24")
    ax_rsi.axhline(70, color="#B0BEC5", lw=0.6, ls="--"); ax_rsi.axhline(30, color="#B0BEC5", lw=0.6, ls="--")
    ax_rsi.set_ylabel("RSI", fontsize=8)
    ax_rsi.legend(loc="upper left", fontsize=7, framealpha=0.88)
    ax_rsi.grid(True, alpha=0.20, ls="--")
    ax_rsi.text(0.995, 0.96, f"RSI6:{_fmtv(df['RSI6'].iloc[-1],1)}  RSI12:{_fmtv(df['RSI12'].iloc[-1],1)}  RSI24:{_fmtv(df['RSI24'].iloc[-1],1)}", transform=ax_rsi.transAxes, fontsize=7.5, ha="right", va="top", color=C["rsi24"], bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C["rsi24"], alpha=0.62))
    format_xaxis(ax_rsi, dates, "W")

    plt.tight_layout(rect=[0, 0, 1, 0.997]); os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, f"{fn_prefix}_日K线技术分析.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    import gc; gc.collect()
    print(f"    [OK] 日K线图 → {os.path.basename(out)}")
    return out


def _draw_colorbar(fig, ax3):
    bbox = ax3.get_position(); bx = bbox.x1 + 0.005; bw = 0.012; bym = bbox.y0 + bbox.height * 0.5; n = 50
    for (y0, h, clo, chi, top_lbl, bot_lbl, xlabel) in [
        (bym, bbox.height*0.45, C["dev_pos_lo"], C["dev_pos_hi"], "高分位(超买)", "低分位", "正乖离"),
        (bbox.y0, bbox.height*0.45, C["dev_neg_lo"], C["dev_neg_hi"], "高分位(超卖)", "低分位", "负乖离"),
    ]:
        ba = fig.add_axes([bx, y0, bw, h])
        colors = np.array([_lerp(clo, chi, t/n) for t in range(n+1)])
        if xlabel == "正乖离": colors = colors[::-1]
        ba.imshow(colors.reshape(n+1,1,3), aspect="auto", extent=[0,1,0,1])
        ba.set_xticks([]); ba.set_yticks([0,1]); ba.set_yticklabels([bot_lbl, top_lbl] if xlabel == "正乖离" else [top_lbl, bot_lbl], fontsize=5.5)
        ba.set_xlabel(xlabel, fontsize=6, labelpad=2); ba.yaxis.set_label_position("right"); ba.yaxis.tick_right()



def plot_fish_bowl(name, symbol, df, save_dir, kind="etf"):
    """生成鱼盆模型图。

    注意：本函数传入的是全历史 df_analysis，函数内部只截取最近 DISPLAY_BARS 根用于展示。
    这样既能保持图表简洁，又能让MA20乖离率分位数和参考线使用全历史样本。
    """
    df_full = df.copy().sort_index()
    ref_lines = calc_dev_reference_lines(df_full["MA20乖离率(%)"]) if "MA20乖离率(%)" in df_full.columns else {}
    df = select_display_window(df_full, DISPLAY_BARS)
    if df.empty:
        return pd.DataFrame(), None
    dates = df.index
    x = np.arange(len(df))
    close = df["收盘"]
    chg = df["涨跌幅"]
    vol = df["成交量"] if "成交量" in df.columns else pd.Series(np.nan, index=df.index)
    has_v = vol.notna().any() and vol.fillna(0).sum() > 0
    ms, ml = df["MA5"], df["MA20"]
    dev, pct = df["MA20乖离率(%)"], df["MA20乖离率分位(%)"]
    pct_signed = df["MA20乖离率有符号分位(%)"] if "MA20乖离率有符号分位(%)" in df.columns else calc_signed_dev_pct(dev, pct)
    golden, death = find_crossovers(ms, ml)
    display_title = _target_display_title(name, symbol, kind)
    fn_prefix = _target_fn_prefix(name, symbol, kind)
    fig = plt.figure(figsize=(21, 20))
    fig.patch.set_facecolor("#FAFAFA")
    fig.suptitle(f"{display_title}   鱼盆模型（MA5/MA20）    截至{TODAY.strftime('%Y年%m月%d日')}   （近{len(df)}个交易日）", fontsize=12, fontweight="bold", y=0.999)
    gs = gridspec.GridSpec(3, 1, figure=fig, height_ratios=[3, 1.5, 2.5], hspace=0.28)
    ax1 = fig.add_subplot(gs[0]); ax2 = fig.add_subplot(gs[1], sharex=ax1); ax3 = fig.add_subplot(gs[2], sharex=ax1)

    # 面板1：价格 + MA5/MA20 + 金叉死叉 + 阶段高低点
    ax1.set_facecolor("#F8F9FA")
    ax1.fill_between(x, close.min()*0.990, close.max()*1.010, where=(ms.values > ml.values), alpha=0.04, color="#EF5350")
    ax1.fill_between(x, close.min()*0.990, close.max()*1.010, where=(ms.values <= ml.values), alpha=0.04, color="#26A69A")
    _draw_candlestick(ax1, x, df["开盘"] if "开盘" in df.columns else None, df["最高"] if "最高" in df.columns else None, df["最低"] if "最低" in df.columns else None, close)
    if "最高" in df.columns and "最低" in df.columns:
        annotate_stage_extrema(ax1, x, df["最高"], df["最低"], kind)
    ax1.plot(x, ms.values, C["fish_mas"], lw=1.2, label="MA5")
    ax1.plot(x, ml.values, C["fish_mal"], lw=1.5, label="MA20")
    for gd in golden:
        i = df.index.get_indexer([gd])[0]
        yp = close.loc[gd]
        ax1.scatter(i, yp*0.986, marker="^", s=95, color=C["fish_golden"], zorder=6)
        ax1.annotate(f"金叉\n{gd.strftime('%m/%d')}", xy=(i, yp*0.986), xytext=(0,-22), textcoords="offset points", fontsize=6.5, color=C["fish_golden"], ha="center", va="top")
    for dd in death:
        i = df.index.get_indexer([dd])[0]
        yp = close.loc[dd]
        ax1.scatter(i, yp*1.014, marker="v", s=95, color=C["fish_death"], zorder=6)
        ax1.annotate(f"死叉\n{dd.strftime('%m/%d')}", xy=(i, yp*1.014), xytext=(0,10), textcoords="offset points", fontsize=6.5, color=C["fish_death"], ha="center", va="bottom")
    state = "多头期" if ms.iloc[-1] > ml.iloc[-1] else "空头期"
    sc = "#C62828" if ms.iloc[-1] > ml.iloc[-1] else "#2E7D32"
    ax1.set_ylabel("价格", fontsize=9)
    cl_leg=[Line2D([0],[0],marker="^",color="w",markerfacecolor=C["fish_golden"],markersize=9,label=f"金叉({len(golden)}次)"), Line2D([0],[0],marker="v",color="w",markerfacecolor=C["fish_death"],markersize=9,label=f"死叉({len(death)}次)")]
    h1,l1=ax1.get_legend_handles_labels(); ax1.legend(h1+cl_leg, l1+[x.get_label() for x in cl_leg], loc="upper left", fontsize=8, ncol=3, framealpha=0.90)
    ax1.grid(True, alpha=0.22, ls="--")
    ax1.text(0.995,0.97,f"收盘:{_fmtv(close.iloc[-1],3)}  日涨跌:{_fmtv(chg.iloc[-1],2)}%  {state}\nMA5:{_fmtv(ms.iloc[-1],3)}  MA20:{_fmtv(ml.iloc[-1],3)}", transform=ax1.transAxes, fontsize=8.5, ha="right", va="top", color=sc, bbox=dict(boxstyle="round,pad=0.35", fc="white", ec=sc, alpha=0.62))
    format_xaxis(ax1, dates, "W")

    # 面板2：成交量 + MA5/MA20均量 + 量比
    ax2.set_facecolor("#F8F9FA")
    if has_v:
        vms = df["MA5均量"]
        vml = df["MA20均量"] if "MA20均量" in df.columns else vol.rolling(20, min_periods=1).mean()
        vr = df["量比"]
        bc = [C["fish_vol_up"] if c >= 0 else C["fish_vol_dn"] for c in chg.fillna(0)]
        dv, unit = _get_volume_display_unit(kind, vol.max(skipna=True))
        ax2.bar(x, vol.values/dv, color=bc, alpha=0.60, width=0.7, label=f"成交量({unit})")
        ax2.plot(x, vms.values/dv, color=C["fish_vmas"], lw=1.1, label=f"MA5均量({unit})")
        ax2.plot(x, vml.values/dv, color=C["fish_vmal"], lw=1.1, label=f"MA20均量({unit})")
        ax2.set_ylabel(f"成交量({unit})", fontsize=8)
        ax2r = ax2.twinx(); ax2r.plot(x, vr.values, color=C["fish_vratio"], lw=1.1, alpha=0.70, label="量比(量/MA5)"); ax2r.axhline(1, color=C["fish_vratio"], lw=0.5, ls="--", alpha=0.40)
        ax2r.set_ylabel("量比", fontsize=7, color=C["fish_vratio"]); ax2r.tick_params(axis="y", labelcolor=C["fish_vratio"], labelsize=6); ax2r.set_ylim(0, max((vr.dropna().quantile(0.99) if vr.notna().any() else 2.0)*1.15, 3.0))
        from matplotlib.lines import Line2D as _L2D
        extra_h = [_L2D([0],[0],color=C["fish_vratio"],lw=1.1,alpha=0.70,label="量比(量/MA5)")]
        h2,l2=ax2.get_legend_handles_labels(); ax2.legend(h2+extra_h, l2+["量比(量/MA5)"], loc="upper left", fontsize=7.5, framealpha=0.88)
        ax2.text(0.995,0.96,f"量:{_fmtv(vol.iloc[-1]/dv,2)}{unit}  MA5:{_fmtv(vms.iloc[-1]/dv,2)}{unit}  MA20:{_fmtv(vml.iloc[-1]/dv,2)}{unit}  量比:{_fmtv(vr.iloc[-1],2)}", transform=ax2.transAxes, fontsize=7.5, ha="right", va="top", color=C["fish_vratio"], bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C["fish_vratio"], alpha=0.62))
    ax2.grid(True, alpha=0.20, ls="--")
    format_xaxis(ax2, dates, "W")

    # 面板3：MA20乖离率 + 全历史分位数趋势 + 正负乖离20/80分位参考线
    ax3.set_facecolor("#F8F9FA")
    # 严重超买/严重超卖区域背景：基于全历史正/负乖离80%参考线。
    y_min = np.nanmin(dev.values) if np.isfinite(dev.values).any() else -1
    y_max = np.nanmax(dev.values) if np.isfinite(dev.values).any() else 1
    if not pd.isna(ref_lines.get("pos80", np.nan)):
        ax3.axhspan(ref_lines["pos80"], max(y_max, ref_lines["pos80"]), color=C["vol_up"], alpha=0.10, zorder=0)
    if not pd.isna(ref_lines.get("neg80", np.nan)):
        ax3.axhspan(min(y_min, ref_lines["neg80"]), ref_lines["neg80"], color=C["vol_dn"], alpha=0.10, zorder=0)
    colors = [_dev_color(dv, pv) for dv, pv in zip(dev.values, pct.values)]
    ax3.bar(x, dev.values, color=colors, width=0.7, label="MA20乖离率")
    ax3.axhline(0, color="#455A64", lw=0.8, zorder=3)
    line_handles = []
    if not pd.isna(ref_lines.get("pos20", np.nan)):
        line_handles.append(ax3.axhline(ref_lines["pos20"], color="#FFCCBC", lw=0.9, ls="--", alpha=0.85, label=f"正乖离20%≈{ref_lines['pos20']:.1f}%"))
        line_handles.append(ax3.axhline(ref_lines["pos80"], color=C["dev_pos_hi"], lw=0.9, ls="--", alpha=0.85, label=f"正乖离80%≈{ref_lines['pos80']:.1f}%"))
    if not pd.isna(ref_lines.get("neg20", np.nan)):
        line_handles.append(ax3.axhline(ref_lines["neg20"], color="#B2DFDB", lw=0.9, ls="--", alpha=0.85, label=f"负乖离20%≈{ref_lines['neg20']:.1f}%"))
        line_handles.append(ax3.axhline(ref_lines["neg80"], color=C["dev_neg_hi"], lw=0.9, ls="--", alpha=0.85, label=f"负乖离80%≈{ref_lines['neg80']:.1f}%"))
    ax3.set_ylabel("MA20乖离率(%)", fontsize=8)
    ax3.grid(True, alpha=0.18, ls="--")
    if line_handles:
        ax3.legend(handles=line_handles, loc="upper left", fontsize=7, ncol=2, framealpha=0.90)

    ax3r = ax3.twinx()
    # 右轴：有符号分位数。0以上表示正乖离超买强度，0以下表示负乖离超卖强度。
    ax3r.plot(x, pct_signed.values, color="#BDBDBD", lw=1.45, ls="-", alpha=0.98, label="有符号分位数")
    for yv in [-80, -20, 0, 20, 80]:
        ax3r.axhline(yv, color="#90A4AE", lw=0.55 if yv else 0.75, ls=":" if yv else "-", alpha=0.65)
    ax3r.set_ylim(-100, 100)
    ax3r.set_ylabel("有符号分位数(%)", fontsize=8, color="#BDBDBD")
    ax3r.tick_params(axis="y", labelcolor="#BDBDBD", labelsize=7)

    pct_last = pct.iloc[-1]
    signed_last = pct_signed.iloc[-1]
    status, pc3 = dev_status_label(dev.iloc[-1], signed_last)
    ax3.text(0.995,0.96,f"乖离率:{_fmtv(dev.iloc[-1],2)}%  分位:{_fmtv(signed_last,1)}%  {status}", transform=ax3.transAxes, fontsize=8, ha="right", va="top", color=pc3, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=pc3, alpha=0.62))
    _draw_colorbar(fig, ax3)
    format_xaxis(ax3, dates, "W")

    plt.tight_layout(rect=[0, 0, 0.94, 0.997]); os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, f"{fn_prefix}_鱼盆模型.png")
    plt.savefig(out, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    import gc; gc.collect()
    print(f"    [OK] 鱼盆图   → {os.path.basename(out)}")
    fish = pd.DataFrame({"收盘价": close, "日涨跌(%)": chg.round(2), "MA5": ms.round(4), "MA20": ml.round(4), "MA20乖离率(%)": dev.round(3), "MA20乖离率分位(%)": pct.round(1), "MA20乖离率有符号分位(%)": pct_signed.round(1)}, index=dates)
    if has_v:
        fish["成交量"] = vol; fish["MA5均量"] = df["MA5均量"].round(0); fish["MA20均量"] = df["MA20均量"].round(0); fish["量比"] = df["量比"].round(3)
    cc2 = pd.Series("", index=dates)
    for gd in golden:
        if gd in cc2.index: cc2.loc[gd] = "金叉↑"
    for dd in death:
        if dd in cc2.index: cc2.loc[dd] = "死叉↓"
    fish["MA5-MA20交叉"] = cc2
    if kind == "etf":
        for col in ["超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)", "近5日涨跌(%)", "近20日涨跌(%)"]:
            if col in df.columns:
                fish[col] = df[col].round(3)
    return fish, out



def export_excel(name, symbol, df_analysis, df_fish, save_dir, kind="etf"):
    if (df_analysis is None or df_analysis.empty) and (df_fish is None or df_fish.empty):
        print(f"    [SKIP] {name} 无数据，跳过Excel")
        return None
    fn_prefix = _target_fn_prefix(name, symbol, kind)
    os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, f"{fn_prefix}_数据表.xlsx")
    try:
        engine = "xlsxwriter"
        try:
            import xlsxwriter  # noqa: F401
        except Exception:
            engine = "openpyxl"
        with pd.ExcelWriter(out, engine=engine) as w:
            if df_analysis is not None and not df_analysis.empty:
                cols1 = ["开盘", "收盘", "最高", "最低", "成交量", "成交额", "涨跌幅", "MA5", "MA10", "MA20", "MA60", "MA200", "BOLL上轨", "BOLL中轨", "BOLL下轨", "近5日涨跌(%)", "近20日涨跌(%)"]
                cols1 = [c for c in cols1 if c in df_analysis.columns]
                df_analysis[cols1].round(4).to_excel(w, sheet_name="日K线数据")
                cols2 = ["DIF", "DEA", "MACD柱", "K", "D", "J", "RSI6", "RSI12", "RSI24", "MA5均量", "MA20均量", "量比", "MA5乖离率(%)", "MA20乖离率(%)", "MA20乖离率分位(%)", "MA20乖离率有符号分位(%)", "MA5-MA20交叉"]
                if kind == "etf":
                    cols2 += ["沪深300涨跌幅(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)"]
                cols2 = [c for c in cols2 if c in df_analysis.columns]
                df_analysis[cols2].round(4).to_excel(w, sheet_name="技术指标")
            if df_fish is not None and not df_fish.empty:
                df_fish.to_excel(w, sheet_name="鱼盆数据")
        print(f"    [OK] 数据表  → {os.path.basename(out)}")
        return out
    except Exception as e:
        print(f"    [WARN] Excel导出失败({name}): {e}")
        return None
    fn_prefix = _target_fn_prefix(name, symbol, kind)
    os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, f"{fn_prefix}_数据表.xlsx")
    try:
        engine = "xlsxwriter"
        try:
            import xlsxwriter  # noqa: F401
        except Exception:
            engine = "openpyxl"
        with pd.ExcelWriter(out, engine=engine) as w:
            if df_analysis is not None and not df_analysis.empty:
                cols1 = ["开盘", "收盘", "最高", "最低", "成交量", "成交额", "涨跌幅", "MA5", "MA10", "MA20", "MA60", "MA200", "近5日涨跌(%)", "近20日涨跌(%)"]
                cols1 = [c for c in cols1 if c in df_analysis.columns]
                df_analysis[cols1].round(4).to_excel(w, sheet_name="日K线数据")
                cols2 = ["DIF", "DEA", "MACD柱", "K", "D", "J", "RSI6", "RSI12", "RSI24", "MA5均量", "量比", "MA5乖离率(%)", "MA20乖离率(%)", "MA20乖离率分位(%)", "MA20乖离率有符号分位(%)", "MA5-MA20交叉"]
                if kind == "etf": cols2 += ["沪深300涨跌幅(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)"]
                cols2 = [c for c in cols2 if c in df_analysis.columns]
                df_analysis[cols2].round(4).to_excel(w, sheet_name="技术指标")
            if df_fish is not None and not df_fish.empty:
                df_fish.to_excel(w, sheet_name="鱼盆数据")
        print(f"    [OK] 数据表  → {os.path.basename(out)}")
        return out
    except Exception as e:
        print(f"    [WARN] Excel导出失败({name}): {e}")
        return None



def _format_snapshot_tables(records: List[dict], kind="etf"):
    id_col = "指数(代码)" if kind == "index" else "ETF(代码)"
    if not records:
        return pd.DataFrame(), [], id_col, ""
    if kind == "index":
        ordered_cols = ["收盘价", "日涨跌(%)", "MA5", "MA20", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比", "MA5-MA20交叉日"]
    else:
        ordered_cols = ["收盘价", "日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)", "MA5", "MA20", "MA5乖离率(%)", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比", "MA5-MA20交叉日"]
    df = pd.DataFrame(records)
    for col in ["标的"] + ordered_cols:
        if col not in df.columns:
            df[col] = "N/A"
    df = df[["标的"] + ordered_cols].rename(columns={"标的": id_col})
    for col in ordered_cols:
        if col != "MA5-MA20交叉日":
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "MA20乖离率(%)" in df.columns:
        df = df.sort_values(["MA20乖离率(%)", "MA20乖离率分位(%)"], ascending=False, na_position="last")
    vol_cols = [c for c in ["成交量", "MA5均量", "MA20均量"] if c in df.columns]
    max_vol = max((df[c].abs().max() for c in vol_cols if df[c].notna().any()), default=0)
    vol_div, vol_unit = _get_volume_display_unit(kind, max_vol)
    vol_col_labels = {c: f"{c}({vol_unit})" for c in vol_cols}
    display_cols = [vol_col_labels.get(c, c) for c in df.columns]
    cell_text, cell_colors = [], []
    def fmt(col, row):
        v = row.get(col)
        if pd.isna(v) if not isinstance(v, str) else (str(v) == "N/A"):
            return "N/A", "white"
        if col == id_col:
            return str(v), "white"
        if col == "MA20乖离率分位(%)":
            pct_v = float(v); dev_v = float(row.get("MA20乖离率(%)", np.nan)); t = max(0, min(1, pct_v/100.0))
            if not pd.isna(dev_v):
                clr = (_lerp((1.0,0.98,0.96),(0.85,0.18,0.18),t)+(0.85,)) if dev_v >= 0 else (_lerp((0.96,0.99,0.97),(0.18,0.55,0.22),t)+(0.85,))
            else:
                clr = "white"
            return f"{pct_v:.0f}%", clr
        if col in {"MA5乖离率(%)", "MA20乖离率(%)"}:
            fv = float(v); clr = (1.0,0.94,0.94,0.60) if fv > 0 else (0.94,1.0,0.95,0.60) if fv < 0 else "white"
            return f"{fv:+.2f}%", clr
        if col in {"日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)"}:
            fv = float(v); clr = (1.0,0.90,0.90,0.50) if fv >= 0 else (0.90,1.0,0.90,0.50)
            return f"{fv:+.2f}%", clr
        if col == "20日滚动波动率(%)":
            return f"{float(v):.2f}%", "white"
        if col == "MA5-MA20交叉日":
            sv = str(v); clr = (1.0,0.92,0.92,0.70) if "金叉" in sv else (0.92,1.0,0.92,0.70) if "死叉" in sv else "white"
            return sv, clr
        if col in {"MA5", "MA20", "收盘价"}:
            return f"{float(v):.3f}", "white"
        if col in vol_cols:
            return f"{float(v)/vol_div:.2f}", "white"
        if col == "量比":
            fv = float(v); clr = (1.0,0.93,0.93,0.50) if fv > 1.2 else (0.93,1.0,0.93,0.50) if fv < 0.8 else "white"
            return f"{fv:.2f}", clr
        if col == "资金净流入估算(亿元)":
            fv = float(v); clr = (1.0,0.93,0.93,0.55) if fv > 0 else (0.93,1.0,0.93,0.55) if fv < 0 else "white"
            return f"{fv:+.2f}", clr
        try:
            return f"{float(v):.2f}", "white"
        except Exception:
            return str(v), "white"
    for _, row in df.iterrows():
        rt, rc = [], []
        for col in df.columns:
            t, c = fmt(col, row)
            rt.append(t); rc.append(c)
        cell_text.append(rt); cell_colors.append(rc)
    df_text = pd.DataFrame(cell_text, columns=display_cols)
    note = f"成交量/均量单位：{vol_unit}；量比=成交量÷MA5均量"
    return df_text, cell_colors, id_col, note


def _snapshot_numeric_tables(records: List[dict], kind="etf"):
    """构造快照Excel专用的数据表。

    与 _format_snapshot_tables 不同：
    - PNG快照需要把数字格式化成带百分号的字符串，方便直接绘图；
    - Excel快照需要写入真正的 int/float，避免Excel提示“以文本形式存储的数字”。

    返回：
    - df_num：真实数值表，除第一列和交叉日外均为数值类型；
    - colors：与PNG快照一致的单元格底色；
    - number_formats：每一列对应的Excel数字格式；
    - note：成交量单位说明。
    """
    id_col = "指数(代码)" if kind == "index" else "ETF(代码)"
    if not records:
        return pd.DataFrame(), [], {}, ""

    # 列顺序保持与PNG快照一致，避免Excel和图片的阅读口径不一致。
    if kind == "index":
        ordered_cols = ["收盘价", "日涨跌(%)", "MA5", "MA20", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比", "MA5-MA20交叉日"]
    else:
        ordered_cols = ["收盘价", "日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)", "MA5", "MA20", "MA5乖离率(%)", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比", "MA5-MA20交叉日"]

    df = pd.DataFrame(records)
    for col in ["标的"] + ordered_cols:
        if col not in df.columns:
            df[col] = "N/A"
    df = df[["标的"] + ordered_cols].rename(columns={"标的": id_col})

    # 除名称与交叉日期外，全部转为数值；无法转为数值的用空值，避免写成字符串数字。
    numeric_source_cols = [c for c in ordered_cols if c != "MA5-MA20交叉日"]
    for col in numeric_source_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 与快照图保持一致：按MA20乖离率降序，其次按分位数降序。
    if "MA20乖离率(%)" in df.columns:
        df = df.sort_values(["MA20乖离率(%)", "MA20乖离率分位(%)"], ascending=False, na_position="last")

    # 成交量类字段按万/亿缩放后写入Excel，表头同步带单位。
    vol_cols = [c for c in ["成交量", "MA5均量", "MA20均量"] if c in df.columns]
    max_vol = max((df[c].abs().max() for c in vol_cols if df[c].notna().any()), default=0)
    vol_div, vol_unit = _get_volume_display_unit(kind, max_vol)
    rename_map = {c: f"{c}({vol_unit})" for c in vol_cols}
    for col in vol_cols:
        df[col] = df[col] / vol_div
    df = df.rename(columns=rename_map)

    # 数字格式：这里的百分号是“文本显示符号”，不让Excel再乘100。
    # 例如原值2.35，显示为2.35%，而不是235%。
    number_formats = {}
    for col in df.columns:
        if col in {id_col, "MA5-MA20交叉日"}:
            number_formats[col] = "@"
        elif col in {"日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "MA5乖离率(%)", "MA20乖离率(%)", "20日滚动波动率(%)"}:
            number_formats[col] = '+0.00"%";-0.00"%";0.00"%"'
        elif col == "MA20乖离率分位(%)":
            number_formats[col] = '0"%"'
        elif col in {"收盘价", "MA5", "MA20"}:
            number_formats[col] = "0.000"
        elif col == "量比":
            number_formats[col] = "0.00"
        elif col == "资金净流入估算(亿元)":
            number_formats[col] = "+0.00;-0.00;0.00"
        else:
            number_formats[col] = "0.00"

    # 颜色复用PNG快照的逻辑，保证图片与Excel的红绿深浅一致。
    df_text, colors, _, note = _format_snapshot_tables(records, kind=kind)
    return df.reset_index(drop=True), colors, number_formats, note


def plot_fishbowl_snapshot(records: List[dict], save_dir: str, filename: str, title: str, kind="etf"):
    df_text, cell_colors, _, note = _format_snapshot_tables(records, kind=kind)
    if df_text.empty:
        return None
    fig_w = max(20, len(df_text.columns)*1.35 + 2)
    fig_h = max(5.5, len(df_text)*0.40 + 3.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h)); fig.patch.set_facecolor("#FAFAFA"); ax.axis("off")
    ax.set_title(f"{title}\n（按MA20乖离率从大到小排序；{note}）", fontsize=11, fontweight="bold", pad=14)
    tbl = ax.table(cellText=df_text.values.tolist(), colLabels=df_text.columns.tolist(), cellLoc="center", loc="center", cellColours=cell_colors)
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.1); tbl.scale(1.06, 1.56)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor("#D0D7DE")
        cell.set_linewidth(0.35)
        if r == 0:
            cell.set_facecolor("#EAF2F8")
            cell.set_text_props(fontweight="bold")
    plt.tight_layout(); out = os.path.join(save_dir, filename)
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    import gc; gc.collect()
    print(f"  [OK] 鱼盆快照 → {out}")
    return out


def _rgb_to_hex(color):
    if color in (None, "white", ""):
        return None
    if isinstance(color, tuple):
        vals = color[:3]
        nums = [int(max(0, min(255, round(v*255 if isinstance(v, float) and v <= 1 else v)))) for v in vals]
        return "".join(f"{v:02X}" for v in nums)
    return None


def export_snapshot_excel(index_records, etf_records, root_dir: str):
    """导出市场快照Excel，并保留快照颜色。

    重点修复：
    旧版为了显示百分号、正负号和成交量单位，把所有单元格都写成了字符串，
    Excel会提示“以文本形式存储的数字”，左上角出现绿色三角。

    新版做法：
    - 第一列和交叉日保留文本；
    - 价格、涨跌幅、乖离率、量比、成交量等字段写入真正数值；
    - 通过 number_format 显示百分号、正负号和小数位；
    - 继续按PNG快照逻辑给单元格填充底色。
    """
    try:
        from openpyxl import Workbook
        from openpyxl.styles import PatternFill, Border, Side, Alignment, Font
    except Exception as e:
        print(f"  [WARN] openpyxl不可用，跳过快照数据表: {e}")
        return None

    out = os.path.join(root_dir, "市场快照数据表.xlsx")
    wb = Workbook()
    wb.remove(wb.active)

    thin = Side(style="thin", color="D0D7DE")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for sheet_name, records, kind in [("宽基快照", index_records, "index"), ("行业ETF快照", etf_records, "etf")]:
        df_num, colors, number_formats, note = _snapshot_numeric_tables(records, kind=kind)
        ws = wb.create_sheet(sheet_name)
        if df_num.empty:
            ws.append(["无数据"])
            continue

        # 表头：使用浅蓝底色，便于与主体内容区分。
        ws.append(df_num.columns.tolist())
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.fill = PatternFill(fill_type="solid", fgColor="EAF2F8")
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = border

        # 主体：真实数值写入Excel，再设置显示格式和颜色。
        for r_idx, row in enumerate(df_num.itertuples(index=False), start=2):
            ws.append(list(row))
            for c_idx, cell in enumerate(ws[r_idx], start=1):
                col_name = df_num.columns[c_idx - 1]
                cell.alignment = Alignment(horizontal="center", vertical="center")
                cell.border = border
                cell.number_format = number_formats.get(col_name, "General")

                # NaN写入Excel会变成文本nan或不美观，这里统一置空。
                if isinstance(cell.value, float) and np.isnan(cell.value):
                    cell.value = None

                hx = _rgb_to_hex(colors[r_idx - 2][c_idx - 1]) if colors else None
                if hx:
                    cell.fill = PatternFill(fill_type="solid", fgColor=hx)

        # 冻结首行，方便日常查看。
        ws.freeze_panes = "A2"

        # 自动列宽：不使用过宽列，保持快照表紧凑可读。
        for col_idx, col_name in enumerate(df_num.columns, start=1):
            values = [str(col_name)] + ["" if pd.isna(v) else str(v) for v in df_num.iloc[:, col_idx - 1].tolist()]
            width = min(max(max(len(v) for v in values) + 2, 10), 26)
            ws.column_dimensions[ws.cell(1, col_idx).column_letter].width = width

        # 在表格下方写入单位说明，不参与主表排序和阅读。
        note_row = len(df_num) + 3
        ws.cell(note_row, 1).value = note
        ws.cell(note_row, 1).font = Font(italic=True, color="666666")

    wb.save(out)
    print(f"  [OK] 快照数据表 → {out}")
    return out


def plot_etf_heatmap(records: List[dict], save_dir: str):
    """生成ETF轮动热力图。

    本版热力图不再展示MA5/MA20乖离率，而是回到资金轮动更直观的涨跌幅口径：
    - 纵轴：近20日涨跌幅在上，近5日涨跌幅在下；
    - 横轴：行业ETF；
    - 排序：按近5日涨跌幅从左到右降序，近5日相同则按近20日涨跌幅降序。
    """
    if not records:
        return None

    df = pd.DataFrame(records)
    need = ["标的", "近5日涨跌(%)", "近20日涨跌(%)"]
    for col in need:
        if col not in df.columns:
            return None

    mat = df[need].copy()
    mat["近5日涨跌(%)"] = pd.to_numeric(mat["近5日涨跌(%)"], errors="coerce")
    mat["近20日涨跌(%)"] = pd.to_numeric(mat["近20日涨跌(%)"], errors="coerce")

    # 横轴排序：近5日越强越靠左；近5日相同时，再比较近20日。
    mat = mat.sort_values(["近5日涨跌(%)", "近20日涨跌(%)"], ascending=[False, False], na_position="last")
    labels = mat["标的"].tolist()

    # 纵轴顺序按用户要求：20日在上，5日在下。
    values = np.vstack([mat["近20日涨跌(%)"].values, mat["近5日涨跌(%)"].values])
    if np.isnan(values).all():
        return None

    vmax = max(np.nanpercentile(np.abs(values), 95), 1.0)
    cmap = LinearSegmentedColormap.from_list("a_red_green", ["#1B5E20", "#A5D6A7", "#F5F5F5", "#FFCDD2", "#B71C1C"], N=256)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    fig_w = max(20, len(labels) * 0.70 + 4)
    fig, ax = plt.subplots(figsize=(fig_w, 6.2))
    fig.patch.set_facecolor("#FAFAFA")
    im = ax.imshow(values, aspect="auto", cmap=cmap, norm=norm)

    ax.set_yticks([0, 1])
    ax.set_yticklabels(["近20日涨跌幅", "近5日涨跌幅"], fontsize=11, fontweight="bold")
    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8.5)
    ax.set_title(
        f"行业ETF轮动热力图    截至{TODAY.strftime('%Y年%m月%d日')}\n"
        "（横轴按近5日涨跌幅从高到低排列；近20日在上，近5日在下）",
        fontsize=12,
        fontweight="bold",
        pad=12,
    )

    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            v = values[i, j]
            ax.text(j, i, "N/A" if pd.isna(v) else f"{v:+.2f}%", ha="center", va="center", fontsize=8, color="#263238")

    cbar = fig.colorbar(im, ax=ax, fraction=0.026, pad=0.02)
    cbar.ax.set_ylabel("涨跌幅(%)", rotation=270, labelpad=12)

    plt.tight_layout()
    out = os.path.join(save_dir, "ETF热力图.png")
    plt.savefig(out, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    import gc; gc.collect()
    print(f"  [OK] ETF热力图 → {out}")
    return out


def _collect_chart_paths_for_pdf(all_charts: dict) -> List[str]:
    """按报告阅读顺序收集所有PNG图表，后续逐页写入PDF。"""
    paths = []
    for key in ["index_fish_snapshot", "etf_fish_snapshot", "etf_heatmap"]:
        p = all_charts.get(key)
        if p and os.path.exists(p):
            paths.append(p)
    for _name, charts in all_charts.get("per_target", {}).items():
        for key in ["kline", "fish"]:
            p = charts.get(key)
            if p and os.path.exists(p):
                paths.append(p)
    return paths


def export_pdf_report(root_dir: str, all_charts: dict, date_tag: str):
    """将全部图表写入多页PDF，不依赖reportlab。使用Pillow直接把PNG合成为PDF。"""
    paths = _collect_chart_paths_for_pdf(all_charts)
    if not paths:
        print("\n  [WARN] 无可写入PDF的图表，跳过PDF生成。")
        return None
    out = os.path.join(root_dir, f"A股日度复盘_{date_tag}.pdf")
    try:
        from PIL import Image, ImageOps, ImageDraw, ImageFont
        pages = []
        # 首页：使用白底图片，避免依赖PDF文字字体。
        cover = Image.new("RGB", (2400, 1350), "white")
        draw = ImageDraw.Draw(cover)
        try:
            font_title = ImageFont.truetype("/System/Library/Fonts/PingFang.ttc", 90)
            font_sub = ImageFont.truetype("/System/Library/Fonts/PingFang.ttc", 54)
            font_small = ImageFont.truetype("/System/Library/Fonts/PingFang.ttc", 38)
        except Exception:
            font_title = font_sub = font_small = None
        def _center_text(y, text, font, fill="#263238"):
            bbox = draw.textbbox((0, 0), text, font=font)
            x = (cover.width - (bbox[2] - bbox[0])) / 2
            draw.text((x, y), text, font=font, fill=fill)
        _center_text(470, "A股市场日度复盘报告", font_title)
        _center_text(620, TODAY.strftime("%Y年%m月%d日"), font_sub, "#37474F")
        _center_text(735, f"宽基指数 {len(BROAD_INDICES)} 只 + 行业指数 {len(SECTOR_INDICES)} 只 + 行业ETF {len(SECTOR_ETFS)} 只", font_small, "#607D8B")
        pages.append(cover)
        for p in paths:
            img = Image.open(p)
            img = ImageOps.exif_transpose(img)
            if img.mode in ("RGBA", "LA"):
                bg = Image.new("RGB", img.size, "white")
                bg.paste(img, mask=img.split()[-1])
                img = bg
            else:
                img = img.convert("RGB")
            # 给每张图加一点白边，PDF阅读时更不贴边；不降低图片分辨率。
            img = ImageOps.expand(img, border=40, fill="white")
            pages.append(img)
        pages[0].save(out, "PDF", save_all=True, append_images=pages[1:], resolution=180.0)
        for im in pages:
            try:
                im.close()
            except Exception:
                pass
        print(f"\n  [OK] PDF复盘报告 → {out}")
        return out
    except Exception as e:
        print(f"\n  [WARN] PDF复盘报告生成失败: {e}")
        return None


def export_word(root_dir: str, all_charts: dict, date_tag: str):
    try:
        from docx import Document
        from docx.shared import Inches, Pt
        from docx.enum.text import WD_ALIGN_PARAGRAPH
        from docx.enum.section import WD_ORIENT
    except ImportError:
        print("\n  [WARN] python-docx 未安装，跳过 Word 生成。")
        return None
    doc = Document()
    section = doc.sections[0]
    section.orientation = WD_ORIENT.LANDSCAPE
    section.page_width = int(11906 * 914400 / 1440)
    section.page_height = int(8268 * 914400 / 1440)
    section.left_margin = section.right_margin = Inches(0.6)
    section.top_margin = section.bottom_margin = Inches(0.5)
    img_w = Inches(9.5)
    def add_heading(text, level=1):
        p = doc.add_heading(text, level=level); p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    def add_image(path, width=None):
        if path and os.path.exists(path):
            doc.add_picture(path, width=width or img_w)
        else:
            doc.add_paragraph(f"[图片缺失: {os.path.basename(str(path))}]")
    p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = p.add_run(f"A股市场日度复盘报告\n{TODAY.strftime('%Y年%m月%d日')}")
    run.font.size = Pt(24); run.font.bold = True
    doc.add_paragraph(
        f"涵盖宽基指数 {len(BROAD_INDICES)} 只 + 行业ETF {len(SECTOR_ETFS)} 只\n"
        f"展示窗口：{DISPLAY_BARS}个交易日    MA200预热：{PREHEAT_BARS}个交易日\n"
        f"技术指标：MA / BOLL / MACD / KDJ / RSI / 量比    鱼盆均线：MA5 / MA20"
    )
    doc.add_page_break()
    add_heading("一、市场概况", 1)
    add_heading("1.1 宽基指数鱼盆快照", 2); add_image(all_charts.get("index_fish_snapshot"))
    add_heading("1.2 行业ETF鱼盆快照", 2); add_image(all_charts.get("etf_fish_snapshot"))
    add_heading("1.3 行业ETF轮动热力图", 2); add_image(all_charts.get("etf_heatmap"))
    doc.add_page_break()
    add_heading("二、宽基指数详情", 1)
    for name, symbol, _, kind, __ in BROAD_INDICES:
        charts = all_charts.get("per_target", {}).get(name, {})
        if not charts:
            continue
        add_heading(f"2. {name}（{symbol}）", 2)
        for key, label in [("kline", "日K线技术分析"), ("fish", "鱼盆模型")]:
            pth = charts.get(key)
            if pth and os.path.exists(pth):
                doc.add_paragraph(label).runs[0].font.bold = True
                add_image(pth)
        doc.add_page_break()
    add_heading("三、行业ETF详情", 1)
    for name, symbol, _, kind, __ in SECTOR_ETFS:
        charts = all_charts.get("per_target", {}).get(name, {})
        if not charts:
            continue
        add_heading(f"3. {name}（{symbol}）", 2)
        for key, label in [("kline", "日K线技术分析"), ("fish", "鱼盆模型")]:
            pth = charts.get(key)
            if pth and os.path.exists(pth):
                doc.add_paragraph(label).runs[0].font.bold = True
                add_image(pth)
        doc.add_page_break()
    out = os.path.join(root_dir, f"A股日度复盘_{date_tag}.docx")
    doc.save(out)
    print(f"\n  [OK] Word复盘文档 → {out}")
    return out


def _get_last_cross_label(df_fish: pd.DataFrame) -> str:
    if df_fish is None or df_fish.empty or "MA5-MA20交叉" not in df_fish.columns:
        return "N/A"
    ev = df_fish[df_fish["MA5-MA20交叉"] != ""]
    if ev.empty:
        return "N/A"
    idx = ev.index[-1]; val = ev["MA5-MA20交叉"].iloc[-1]
    return f"{val} {idx.strftime('%Y-%m-%d')}"


def _safe_last(s: pd.Series, dec=3, default="N/A"):
    try:
        return round(float(s.dropna().iloc[-1]), dec)
    except Exception:
        return default




# ══════════════════════════════════════════════════════════════════════════════
# § v1.14 增量优化覆盖函数
# ══════════════════════════════════════════════════════════════════════════════

ANALYSIS_CACHE_VERSION = "v122"
ANALYSIS_CACHE_DIR = str(BASE_DIR / "data_cache_daily_tracker" / "analysis")
CACHE_LAST_CHANGED: Dict[Tuple[str, str], bool] = {}


def _analysis_cache_file_path(symbol: str, kind: str) -> str:
    """单标的“已计算指标”缓存路径。文件名含版本号，防止旧口径误用。"""
    os.makedirs(ANALYSIS_CACHE_DIR, exist_ok=True)
    return os.path.join(ANALYSIS_CACHE_DIR, f"{kind}_{symbol}_{ANALYSIS_CACHE_VERSION}.csv")


def _read_cached_analysis(symbol: str, kind: str) -> pd.DataFrame:
    """读取已计算好的全历史指标表，避免无增量时重复计算分位数。"""
    path = _analysis_cache_file_path(symbol, kind)
    if not os.path.exists(path):
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        if "日期" not in df.columns:
            return pd.DataFrame()
        df["日期"] = pd.to_datetime(df["日期"], errors="coerce")
        df = df.dropna(subset=["日期"]).sort_values("日期").set_index("日期")
        text_cols = {"MA5-MA20交叉", "金叉/死叉天数"}
        for col in df.columns:
            if col not in text_cols:
                # pandas 3.x 对 errors="ignore" 兼容性较差，统一用 coerce，文本列单独排除。
                df[col] = pd.to_numeric(df[col], errors="coerce")
        print(f"    [CACHE] 读取本地指标缓存（{len(df)}日，{ANALYSIS_CACHE_VERSION}）")
        return df
    except Exception as e:
        print(f"    [WARN] 读取指标缓存失败({symbol}): {e}")
        return pd.DataFrame()


def _write_cached_analysis(symbol: str, kind: str, df: pd.DataFrame):
    """写入已计算好的全历史指标表，供下次无增量时直接复用。"""
    if df is None or df.empty:
        return
    path = _analysis_cache_file_path(symbol, kind)
    out = df.copy().sort_index()
    out = out[~out.index.duplicated(keep="last")]
    out.reset_index().rename(columns={out.index.name or "index": "日期"}).to_csv(path, index=False, encoding="utf-8-sig")


def fetch_kline_cached(symbol: str, kind: str, name="") -> pd.DataFrame:
    """优先读取原始K线缓存，并记录本轮是否存在增量/修正。

    CACHE_LAST_CHANGED[(kind, symbol)] = True 表示需要重算指标和分位数；
    False 表示原始K线没有变化，可优先复用指标缓存。
    """
    key = (kind, symbol)
    CACHE_LAST_CHANGED[key] = True
    if not CACHE_ENABLED:
        return fetch_kline(symbol, kind, name)

    cached = _read_cached_kline(symbol, kind)
    if cached.empty:
        df = fetch_kline(symbol, kind, name)
        if not df.empty:
            _write_cached_kline(symbol, kind, df)
            print(f"    [CACHE] 已写入本地历史K线（{len(df)}日）")
        CACHE_LAST_CHANGED[key] = True
        return df

    latest = cached.index.max()
    inc_start = (pd.Timestamp(latest) - pd.Timedelta(days=CACHE_INCREMENT_LOOKBACK_DAYS)).strftime("%Y%m%d")
    inc = _fetch_online_with_start(symbol, kind, name, inc_start)
    if inc is not None and not inc.empty:
        merged = pd.concat([cached, inc]).sort_index()
        merged = merged[~merged.index.duplicated(keep="last")]
        # 判断是否真的新增或修正。若只是重复拉取了完全相同的数据，则无需重算分位数。
        try:
            comparable_cols = sorted(set(cached.columns).intersection(set(merged.columns)))
            old_cmp = cached[comparable_cols].sort_index()
            new_cmp = merged.reindex(old_cmp.index)[comparable_cols].sort_index()
            changed = (len(merged) != len(cached)) or (not old_cmp.round(8).equals(new_cmp.round(8)))
        except Exception:
            changed = len(merged) != len(cached)
        if changed:
            _write_cached_kline(symbol, kind, merged)
            added = max(len(merged) - len(cached), 0)
            print(f"    [CACHE] 增量更新完成：新增/修正 {added} 日，缓存共 {len(merged)} 日")
        else:
            print(f"    [CACHE] 原始K线无增量，缓存共 {len(cached)} 日")
        CACHE_LAST_CHANGED[key] = bool(changed)
        return merged if changed else cached

    print("    [CACHE] 增量源失败，使用本地缓存继续运行")
    CACHE_LAST_CHANGED[key] = False
    return cached


def add_fish_interval_columns(out: pd.DataFrame) -> pd.DataFrame:
    """根据MA5/MA20金叉死叉，逐日计算当前鱼盆区间状态。

    规则：
    - 金叉/死叉当日记为T；
    - 下一交易日记为T+1；
    - 区间涨跌幅 = 当前收盘 / 金叉或死叉当日收盘 - 1。
    """
    if out is None or out.empty or "MA5-MA20交叉" not in out.columns or "收盘" not in out.columns:
        return out
    labels = []
    rets = []
    last_type = None
    last_pos = None
    last_price = np.nan
    closes = pd.to_numeric(out["收盘"], errors="coerce")
    crosses = out["MA5-MA20交叉"].fillna("").astype(str)
    for i, (dt, sig) in enumerate(crosses.items()):
        if "金叉" in sig:
            last_type = "金叉"
            last_pos = i
            last_price = closes.iloc[i]
        elif "死叉" in sig:
            last_type = "死叉"
            last_pos = i
            last_price = closes.iloc[i]
        if last_type is None or last_pos is None or pd.isna(last_price) or last_price == 0:
            labels.append("N/A")
            rets.append(np.nan)
        else:
            n = i - last_pos
            labels.append(f"{last_type}T" if n == 0 else f"{last_type}T+{n}")
            rets.append((closes.iloc[i] / last_price - 1) * 100)
    out["金叉/死叉天数"] = labels
    out["区间涨跌幅(%)"] = pd.Series(rets, index=out.index)
    return out


def prepare_analysis_df(df_raw: pd.DataFrame, kind="etf", benchmark_close: Optional[pd.Series] = None) -> pd.DataFrame:
    """基于全历史原始K线计算完整指标表。"""
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    df = df_raw.copy().sort_index()
    cc, oc, hc, lc, vc, ac, gc = infer_close(df), infer_open(df), infer_high(df), infer_low(df), infer_vol(df), infer_amt(df), infer_chg(df)
    if cc is None:
        return pd.DataFrame()
    out = pd.DataFrame(index=df.index)
    out["收盘"] = pd.to_numeric(df[cc], errors="coerce")
    out["开盘"] = pd.to_numeric(df[oc], errors="coerce") if oc and oc in df.columns else out["收盘"]
    out["最高"] = pd.to_numeric(df[hc], errors="coerce") if hc and hc in df.columns else out[["开盘", "收盘"]].max(axis=1)
    out["最低"] = pd.to_numeric(df[lc], errors="coerce") if lc and lc in df.columns else out[["开盘", "收盘"]].min(axis=1)
    out["成交量"] = normalize_volume_units(pd.to_numeric(df[vc], errors="coerce"), kind=kind, symbol="", amount=(pd.to_numeric(df[ac], errors="coerce") if ac and ac in df.columns else None)) if vc and vc in df.columns else pd.Series(np.nan, index=out.index)
    out["成交额"] = pd.to_numeric(df[ac], errors="coerce") if ac and ac in df.columns else out["收盘"] * out["成交量"]
    out["涨跌幅"] = pd.to_numeric(df[gc], errors="coerce") if gc and gc in df.columns else out["收盘"].pct_change() * 100
    for k, v in calc_ma(out["收盘"]).items():
        out[k] = v
    bu, bm, bl = calc_bollinger(out["收盘"])
    out["BOLL上轨"], out["BOLL中轨"], out["BOLL下轨"] = bu, bm, bl
    dif, dea, hist = calc_macd(out["收盘"])
    out["DIF"], out["DEA"], out["MACD柱"] = dif, dea, hist
    k, d, j = calc_kdj(out["最高"], out["最低"], out["收盘"])
    out["K"], out["D"], out["J"] = k, d, j
    out["RSI6"], out["RSI12"], out["RSI24"] = calc_rsi(out["收盘"], 6), calc_rsi(out["收盘"], 12), calc_rsi(out["收盘"], 24)
    if out["成交量"].notna().any():
        vma5, vma20, vr = calc_volume_indicators(out["成交量"])
    else:
        vma5 = pd.Series(np.nan, index=out.index)
        vma20 = pd.Series(np.nan, index=out.index)
        vr = pd.Series(np.nan, index=out.index)
    out["MA5均量"], out["MA20均量"], out["量比"] = vma5, vma20, vr
    out = _assert_volume_indicators(out)
    out["近5日涨跌(%)"], out["近20日涨跌(%)"] = out["收盘"].pct_change(5) * 100, out["收盘"].pct_change(20) * 100
    out["MA5乖离率(%)"] = calc_deviation(out["收盘"], out["MA5"])
    out["MA20乖离率(%)"] = calc_deviation(out["收盘"], out["MA20"])
    # 分位数必须用全历史逐点计算：每个点只使用该点之前及当日的历史样本。
    out["MA20乖离率分位(%)"] = calc_dev_pct(out["MA20乖离率(%)"])
    out["MA20乖离率有符号分位(%)"] = calc_signed_dev_pct(out["MA20乖离率(%)"], out["MA20乖离率分位(%)"])
    golden, death = find_crossovers(out["MA5"], out["MA20"])
    cross = pd.Series("", index=out.index)
    for dt in golden:
        if dt in cross.index:
            cross.loc[dt] = "金叉↑"
    for dt in death:
        if dt in cross.index:
            cross.loc[dt] = "死叉↓"
    out["MA5-MA20交叉"] = cross
    out = add_fish_interval_columns(out)
    if kind == "etf":
        if benchmark_close is not None and not benchmark_close.empty:
            bench = benchmark_close.reindex(out.index).ffill()
            bench_ret = bench.pct_change() * 100
            out["沪深300涨跌幅(%)"] = bench_ret
            out["超额收益(%)"] = out["涨跌幅"] - bench_ret
        direction = _price_direction_factor(out["开盘"], out["最高"], out["最低"], out["收盘"])
        abnormal_amt = (out["成交额"] - out["成交额"].rolling(5, min_periods=1).mean()).clip(lower=0)
        out["资金净流入估算(亿元)"] = abnormal_amt * direction / 1e8
        out["20日滚动波动率(%)"] = out["收盘"].pct_change().rolling(20, min_periods=20).std() * 100
    return out.dropna(how="all")


def get_analysis_df_cached(symbol: str, kind: str, name: str, df_raw: pd.DataFrame, benchmark_close: Optional[pd.Series] = None, force_recalc: bool = False) -> pd.DataFrame:
    """按“有增量才重算”的原则获取全历史指标表。"""
    changed = CACHE_LAST_CHANGED.get((kind, symbol), True)
    if CACHE_ENABLED and (not changed) and (not force_recalc):
        cached = _read_cached_analysis(symbol, kind)
        if not cached.empty:
            return cached
    df = prepare_analysis_df(df_raw, kind=kind, benchmark_close=benchmark_close)
    if CACHE_ENABLED and df is not None and not df.empty:
        _write_cached_analysis(symbol, kind, df)
        reason = "基准更新" if force_recalc and kind == "etf" else "原始K线更新/首次计算"
        print(f"    [CACHE] 已重算并写入指标缓存（{len(df)}日，{reason}）")
    return df


def build_fish_data_table(df_full: pd.DataFrame, kind="etf") -> pd.DataFrame:
    """从全历史指标表构造鱼盆数据表，Excel和快照均复用该口径。"""
    if df_full is None or df_full.empty:
        return pd.DataFrame()
    cols = [
        "收盘", "涨跌幅", "MA5", "MA20", "MA20乖离率(%)", "MA20乖离率分位(%)",
        "MA20乖离率有符号分位(%)", "成交量", "MA5均量", "MA20均量", "量比",
        "MA5-MA20交叉", "金叉/死叉天数", "区间涨跌幅(%)",
    ]
    if kind == "etf":
        cols += ["近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)"]
    use = [c for c in cols if c in df_full.columns]
    fish = df_full[use].copy()
    fish = fish.rename(columns={"收盘": "收盘价", "涨跌幅": "日涨跌(%)"})
    return fish


def plot_fish_bowl(name, symbol, df, save_dir, kind="etf"):
    """生成鱼盆模型图；返回全历史鱼盆数据表。"""
    df_full = df.copy().sort_index()
    ref_lines = calc_dev_reference_lines(df_full["MA20乖离率(%)"]) if "MA20乖离率(%)" in df_full.columns else {}
    df = select_display_window(df_full, DISPLAY_BARS)
    if df.empty:
        return pd.DataFrame(), None
    dates = df.index
    x = np.arange(len(df))
    close = df["收盘"]
    chg = df["涨跌幅"]
    vol = df["成交量"] if "成交量" in df.columns else pd.Series(np.nan, index=df.index)
    has_v = vol.notna().any() and vol.fillna(0).sum() > 0
    ms, ml = df["MA5"], df["MA20"]
    dev, pct = df["MA20乖离率(%)"], df["MA20乖离率分位(%)"]
    pct_signed = df["MA20乖离率有符号分位(%)"] if "MA20乖离率有符号分位(%)" in df.columns else calc_signed_dev_pct(dev, pct)
    golden, death = find_crossovers(ms, ml)
    display_title = _target_display_title(name, symbol, kind)
    fn_prefix = _target_fn_prefix(name, symbol, kind)
    fig = plt.figure(figsize=(21, 20))
    fig.patch.set_facecolor("#FAFAFA")
    fig.suptitle(f"{display_title}   鱼盆模型（MA5/MA20）    截至{TODAY.strftime('%Y年%m月%d日')}   （近{len(df)}个交易日）", fontsize=12, fontweight="bold", y=0.999)
    gs = gridspec.GridSpec(3, 1, figure=fig, height_ratios=[3, 1.5, 2.5], hspace=0.28)
    ax1 = fig.add_subplot(gs[0]); ax2 = fig.add_subplot(gs[1], sharex=ax1); ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax1.set_facecolor("#F8F9FA")
    ax1.fill_between(x, close.min()*0.990, close.max()*1.010, where=(ms.values > ml.values), alpha=0.04, color="#EF5350")
    ax1.fill_between(x, close.min()*0.990, close.max()*1.010, where=(ms.values <= ml.values), alpha=0.04, color="#26A69A")
    _draw_candlestick(ax1, x, df["开盘"] if "开盘" in df.columns else None, df["最高"] if "最高" in df.columns else None, df["最低"] if "最低" in df.columns else None, close)
    if "最高" in df.columns and "最低" in df.columns:
        annotate_stage_extrema(ax1, x, df["最高"], df["最低"], kind)
    ax1.plot(x, ms.values, C["fish_mas"], lw=1.2, label="MA5")
    ax1.plot(x, ml.values, C["fish_mal"], lw=1.5, label="MA20")
    for gd in golden:
        i = df.index.get_indexer([gd])[0]
        yp = close.loc[gd]
        ax1.scatter(i, yp*0.986, marker="^", s=95, color=C["fish_golden"], zorder=6)
        ax1.annotate(f"金叉\n{gd.strftime('%m/%d')}", xy=(i, yp*0.986), xytext=(0,-22), textcoords="offset points", fontsize=6.5, color=C["fish_golden"], ha="center", va="top")
    for dd in death:
        i = df.index.get_indexer([dd])[0]
        yp = close.loc[dd]
        ax1.scatter(i, yp*1.014, marker="v", s=95, color=C["fish_death"], zorder=6)
        ax1.annotate(f"死叉\n{dd.strftime('%m/%d')}", xy=(i, yp*1.014), xytext=(0,10), textcoords="offset points", fontsize=6.5, color=C["fish_death"], ha="center", va="bottom")
    state = "多头期" if ms.iloc[-1] > ml.iloc[-1] else "空头期"
    sc = "#C62828" if ms.iloc[-1] > ml.iloc[-1] else "#2E7D32"
    ax1.set_ylabel("价格", fontsize=9)
    cl_leg=[Line2D([0],[0],marker="^",color="w",markerfacecolor=C["fish_golden"],markersize=9,label=f"金叉({len(golden)}次)"), Line2D([0],[0],marker="v",color="w",markerfacecolor=C["fish_death"],markersize=9,label=f"死叉({len(death)}次)")]
    h1,l1=ax1.get_legend_handles_labels(); ax1.legend(h1+cl_leg, l1+[x.get_label() for x in cl_leg], loc="upper left", fontsize=8, ncol=3, framealpha=0.90)
    ax1.grid(True, alpha=0.22, ls="--")
    extra = ""
    if "金叉/死叉天数" in df.columns:
        extra = f"\n区间:{df['金叉/死叉天数'].iloc[-1]}  区间涨跌:{_fmtv(df['区间涨跌幅(%)'].iloc[-1],2)}%"
    ax1.text(0.995,0.97,f"收盘:{_fmtv(close.iloc[-1],3)}  日涨跌:{_fmtv(chg.iloc[-1],2)}%  {state}\nMA5:{_fmtv(ms.iloc[-1],3)}  MA20:{_fmtv(ml.iloc[-1],3)}{extra}", transform=ax1.transAxes, fontsize=8.5, ha="right", va="top", color=sc, bbox=dict(boxstyle="round,pad=0.35", fc="white", ec=sc, alpha=0.62))
    format_xaxis(ax1, dates, "W")
    ax2.set_facecolor("#F8F9FA")
    if has_v:
        vms = df["MA5均量"]
        vml = df["MA20均量"] if "MA20均量" in df.columns else vol.rolling(20, min_periods=1).mean()
        vr = df["量比"]
        bc = [C["fish_vol_up"] if c >= 0 else C["fish_vol_dn"] for c in chg.fillna(0)]
        dv, unit = _get_volume_display_unit(kind, vol.max(skipna=True))
        ax2.bar(x, vol.values/dv, color=bc, alpha=0.60, width=0.7, label=f"成交量({unit})")
        ax2.plot(x, vms.values/dv, color=C["fish_vmas"], lw=1.1, label=f"MA5均量({unit})")
        ax2.plot(x, vml.values/dv, color=C["fish_vmal"], lw=1.1, label=f"MA20均量({unit})")
        ax2.set_ylabel(f"成交量({unit})", fontsize=8)
        ax2r = ax2.twinx(); ax2r.plot(x, vr.values, color=C["fish_vratio"], lw=1.1, alpha=0.70, label="量比(量/MA5)"); ax2r.axhline(1, color=C["fish_vratio"], lw=0.5, ls="--", alpha=0.40)
        ax2r.set_ylabel("量比", fontsize=7, color=C["fish_vratio"]); ax2r.tick_params(axis="y", labelcolor=C["fish_vratio"], labelsize=6); ax2r.set_ylim(0, max((vr.dropna().quantile(0.99) if vr.notna().any() else 2.0)*1.15, 3.0))
        from matplotlib.lines import Line2D as _L2D
        extra_h = [_L2D([0],[0],color=C["fish_vratio"],lw=1.1,alpha=0.70,label="量比(量/MA5)")]
        h2,l2=ax2.get_legend_handles_labels(); ax2.legend(h2+extra_h, l2+["量比(量/MA5)"], loc="upper left", fontsize=7.5, framealpha=0.88)
        ax2.text(0.995,0.96,f"量:{_fmtv(vol.iloc[-1]/dv,2)}{unit}  MA5:{_fmtv(vms.iloc[-1]/dv,2)}{unit}  MA20:{_fmtv(vml.iloc[-1]/dv,2)}{unit}  量比:{_fmtv(vr.iloc[-1],2)}", transform=ax2.transAxes, fontsize=7.5, ha="right", va="top", color=C["fish_vratio"], bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=C["fish_vratio"], alpha=0.62))
    ax2.grid(True, alpha=0.20, ls="--")
    format_xaxis(ax2, dates, "W")
    ax3.set_facecolor("#F8F9FA")
    y_min = np.nanmin(dev.values) if np.isfinite(dev.values).any() else -1
    y_max = np.nanmax(dev.values) if np.isfinite(dev.values).any() else 1
    if not pd.isna(ref_lines.get("pos80", np.nan)):
        ax3.axhspan(ref_lines["pos80"], max(y_max, ref_lines["pos80"]), color=C["vol_up"], alpha=0.10, zorder=0)
    if not pd.isna(ref_lines.get("neg80", np.nan)):
        ax3.axhspan(min(y_min, ref_lines["neg80"]), ref_lines["neg80"], color=C["vol_dn"], alpha=0.10, zorder=0)
    colors = [_dev_color(dv, pv) for dv, pv in zip(dev.values, pct.values)]
    ax3.bar(x, dev.values, color=colors, width=0.7, label="MA20乖离率")
    ax3.axhline(0, color="#455A64", lw=0.8, zorder=3)
    line_handles = []
    if not pd.isna(ref_lines.get("pos20", np.nan)):
        line_handles.append(ax3.axhline(ref_lines["pos20"], color="#FFCCBC", lw=0.9, ls="--", alpha=0.85, label=f"正乖离20%≈{ref_lines['pos20']:.1f}%"))
        line_handles.append(ax3.axhline(ref_lines["pos80"], color=C["dev_pos_hi"], lw=0.9, ls="--", alpha=0.85, label=f"正乖离80%≈{ref_lines['pos80']:.1f}%"))
    if not pd.isna(ref_lines.get("neg20", np.nan)):
        line_handles.append(ax3.axhline(ref_lines["neg20"], color="#B2DFDB", lw=0.9, ls="--", alpha=0.85, label=f"负乖离20%≈{ref_lines['neg20']:.1f}%"))
        line_handles.append(ax3.axhline(ref_lines["neg80"], color=C["dev_neg_hi"], lw=0.9, ls="--", alpha=0.85, label=f"负乖离80%≈{ref_lines['neg80']:.1f}%"))
    ax3.set_ylabel("MA20乖离率(%)", fontsize=8)
    ax3.grid(True, alpha=0.18, ls="--")
    if line_handles:
        ax3.legend(handles=line_handles, loc="upper left", fontsize=7, ncol=2, framealpha=0.90)
    ax3r = ax3.twinx()
    ax3r.plot(x, pct_signed.values, color="#BDBDBD", lw=1.45, ls="-", alpha=0.98, label="有符号分位数")
    for yv in [-80, -20, 0, 20, 80]:
        ax3r.axhline(yv, color="#90A4AE", lw=0.55 if yv else 0.75, ls=":" if yv else "-", alpha=0.65)
    ax3r.set_ylim(-100, 100)
    ax3r.set_ylabel("有符号分位数(%)", fontsize=8, color="#BDBDBD")
    ax3r.tick_params(axis="y", labelcolor="#BDBDBD", labelsize=7)
    signed_last = pct_signed.iloc[-1]
    status, pc3 = dev_status_label(dev.iloc[-1], signed_last)
    ax3.text(0.995,0.96,f"乖离率:{_fmtv(dev.iloc[-1],2)}%  分位:{_fmtv(signed_last,1)}%  {status}", transform=ax3.transAxes, fontsize=8, ha="right", va="top", color=pc3, bbox=dict(boxstyle="round,pad=0.3", fc="white", ec=pc3, alpha=0.62))
    _draw_colorbar(fig, ax3)
    format_xaxis(ax3, dates, "W")
    plt.tight_layout(rect=[0, 0, 0.94, 0.997]); os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, f"{fn_prefix}_鱼盆模型.png")
    plt.savefig(out, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    import gc; gc.collect()
    print(f"    [OK] 鱼盆图   → {os.path.basename(out)}")
    fish = build_fish_data_table(df_full, kind=kind)
    return fish, out


def export_excel(name, symbol, df_analysis, df_fish, save_dir, kind="etf"):
    """导出单标的全历史数据表：K线、技术指标、鱼盆数据全部使用全历史。"""
    if (df_analysis is None or df_analysis.empty) and (df_fish is None or df_fish.empty):
        print(f"    [SKIP] {name} 无数据，跳过Excel")
        return None
    fn_prefix = _target_fn_prefix(name, symbol, kind)
    os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, f"{fn_prefix}_数据表.xlsx")
    try:
        engine = "xlsxwriter"
        try:
            import xlsxwriter  # noqa: F401
        except Exception:
            engine = "openpyxl"
        with pd.ExcelWriter(out, engine=engine) as w:
            if df_analysis is not None and not df_analysis.empty:
                cols1 = ["开盘", "收盘", "最高", "最低", "成交量", "成交额", "涨跌幅", "MA5", "MA10", "MA20", "MA60", "MA200", "BOLL上轨", "BOLL中轨", "BOLL下轨", "近5日涨跌(%)", "近20日涨跌(%)"]
                cols1 = [c for c in cols1 if c in df_analysis.columns]
                df_analysis[cols1].round(4).to_excel(w, sheet_name="日K线数据")
                cols2 = ["DIF", "DEA", "MACD柱", "K", "D", "J", "RSI6", "RSI12", "RSI24", "MA5均量", "MA20均量", "量比", "MA5乖离率(%)", "MA20乖离率(%)", "MA20乖离率分位(%)", "MA20乖离率有符号分位(%)", "MA5-MA20交叉", "金叉/死叉天数", "区间涨跌幅(%)"]
                if kind == "etf":
                    cols2 += ["沪深300涨跌幅(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)"]
                cols2 = [c for c in cols2 if c in df_analysis.columns]
                df_analysis[cols2].round(4).to_excel(w, sheet_name="技术指标")
            if df_fish is not None and not df_fish.empty:
                df_fish.round(4).to_excel(w, sheet_name="鱼盆数据")
        print(f"    [OK] 数据表  → {os.path.basename(out)}")
        return out
    except Exception as e:
        print(f"    [WARN] Excel导出失败({name}): {e}")
        return None


def _fish_ordered_cols(kind="etf"):
    tail = ["MA5-MA20交叉日", "金叉/死叉天数", "区间涨跌幅(%)"]
    if kind == "index":
        return ["收盘价", "日涨跌(%)", "MA5", "MA20", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比"] + tail
    return ["收盘价", "日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)", "MA5", "MA20", "MA5乖离率(%)", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比"] + tail


def _format_snapshot_tables(records: List[dict], kind="etf"):
    id_col = "指数(代码)" if kind == "index" else "ETF(代码)"
    if not records:
        return pd.DataFrame(), [], id_col, ""
    ordered_cols = _fish_ordered_cols(kind)
    df = pd.DataFrame(records)
    for col in ["标的"] + ordered_cols:
        if col not in df.columns:
            df[col] = "N/A"
    df = df[["标的"] + ordered_cols].rename(columns={"标的": id_col})
    text_cols = {id_col, "MA5-MA20交叉日", "金叉/死叉天数"}
    for col in ordered_cols:
        if col not in text_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "MA20乖离率(%)" in df.columns:
        df = df.sort_values(["MA20乖离率(%)", "MA20乖离率分位(%)"], ascending=False, na_position="last")
    vol_cols = [c for c in ["成交量", "MA5均量", "MA20均量"] if c in df.columns]
    max_vol = max((df[c].abs().max() for c in vol_cols if df[c].notna().any()), default=0)
    vol_div, vol_unit = _get_volume_display_unit(kind, max_vol)
    vol_col_labels = {c: f"{c}({vol_unit})" for c in vol_cols}
    display_cols = [vol_col_labels.get(c, c) for c in df.columns]
    cell_text, cell_colors = [], []
    def fmt(col, row):
        v = row.get(col)
        if pd.isna(v) if not isinstance(v, str) else (str(v) == "N/A"):
            return "N/A", "white"
        if col == id_col:
            return str(v), "white"
        if col == "MA20乖离率分位(%)":
            pct_v = float(v); dev_v = float(row.get("MA20乖离率(%)", np.nan)); t = max(0, min(1, pct_v/100.0))
            if not pd.isna(dev_v):
                clr = (_lerp((1.0,0.98,0.96),(0.85,0.18,0.18),t)+(0.85,)) if dev_v >= 0 else (_lerp((0.96,0.99,0.97),(0.18,0.55,0.22),t)+(0.85,))
            else:
                clr = "white"
            return f"{pct_v:.0f}%", clr
        if col in {"MA5乖离率(%)", "MA20乖离率(%)"}:
            fv = float(v); clr = (1.0,0.94,0.94,0.60) if fv > 0 else (0.94,1.0,0.95,0.60) if fv < 0 else "white"
            return f"{fv:+.2f}%", clr
        if col in {"日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "区间涨跌幅(%)"}:
            fv = float(v); clr = (1.0,0.90,0.90,0.50) if fv >= 0 else (0.90,1.0,0.90,0.50)
            return f"{fv:+.2f}%", clr
        if col == "20日滚动波动率(%)":
            return f"{float(v):.2f}%", "white"
        if col in {"MA5-MA20交叉日", "金叉/死叉天数"}:
            sv = str(v); clr = (1.0,0.92,0.92,0.70) if "金叉" in sv else (0.92,1.0,0.92,0.70) if "死叉" in sv else "white"
            return sv, clr
        if col in {"MA5", "MA20", "收盘价"}:
            return f"{float(v):.3f}", "white"
        if col in vol_cols:
            return f"{float(v)/vol_div:.2f}", "white"
        if col == "量比":
            fv = float(v); clr = (1.0,0.93,0.93,0.50) if fv > 1.2 else (0.93,1.0,0.93,0.50) if fv < 0.8 else "white"
            return f"{fv:.2f}", clr
        if col == "资金净流入估算(亿元)":
            fv = float(v); clr = (1.0,0.93,0.93,0.55) if fv > 0 else (0.93,1.0,0.93,0.55) if fv < 0 else "white"
            return f"{fv:+.2f}", clr
        try:
            return f"{float(v):.2f}", "white"
        except Exception:
            return str(v), "white"
    for _, row in df.iterrows():
        rt, rc = [], []
        for col in df.columns:
            t, c = fmt(col, row)
            rt.append(t); rc.append(c)
        cell_text.append(rt); cell_colors.append(rc)
    df_text = pd.DataFrame(cell_text, columns=display_cols)
    note = f"成交量/均量单位：{vol_unit}；量比=成交量÷MA5均量"
    return df_text, cell_colors, id_col, note


def _snapshot_numeric_tables(records: List[dict], kind="etf"):
    id_col = "指数(代码)" if kind == "index" else "ETF(代码)"
    if not records:
        return pd.DataFrame(), [], {}, ""
    ordered_cols = _fish_ordered_cols(kind)
    df = pd.DataFrame(records)
    for col in ["标的"] + ordered_cols:
        if col not in df.columns:
            df[col] = "N/A"
    df = df[["标的"] + ordered_cols].rename(columns={"标的": id_col})
    text_cols = {id_col, "MA5-MA20交叉日", "金叉/死叉天数"}
    for col in ordered_cols:
        if col not in text_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "MA20乖离率(%)" in df.columns:
        df = df.sort_values(["MA20乖离率(%)", "MA20乖离率分位(%)"], ascending=False, na_position="last")
    vol_cols = [c for c in ["成交量", "MA5均量", "MA20均量"] if c in df.columns]
    max_vol = max((df[c].abs().max() for c in vol_cols if df[c].notna().any()), default=0)
    vol_div, vol_unit = _get_volume_display_unit(kind, max_vol)
    rename_map = {c: f"{c}({vol_unit})" for c in vol_cols}
    for col in vol_cols:
        df[col] = df[col] / vol_div
    df = df.rename(columns=rename_map)
    number_formats = {}
    for col in df.columns:
        if col in {id_col, "MA5-MA20交叉日", "金叉/死叉天数"}:
            number_formats[col] = "@"
        elif col in {"日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "MA5乖离率(%)", "MA20乖离率(%)", "20日滚动波动率(%)", "区间涨跌幅(%)"}:
            number_formats[col] = '+0.00"%";-0.00"%";0.00"%"'
        elif col == "MA20乖离率分位(%)":
            number_formats[col] = '0"%"'
        elif col in {"收盘价", "MA5", "MA20"}:
            number_formats[col] = "0.000"
        elif col == "量比":
            number_formats[col] = "0.00"
        elif col == "资金净流入估算(亿元)":
            number_formats[col] = "+0.00;-0.00;0.00"
        else:
            number_formats[col] = "0.00"
    df_text, colors, _, note = _format_snapshot_tables(records, kind=kind)
    return df.reset_index(drop=True), colors, number_formats, note


def _safe_last_text(s: pd.Series, default="N/A"):
    try:
        x = s.dropna().iloc[-1]
        return str(x) if str(x) else default
    except Exception:
        return default

def run_tracker(data_provider: Optional[Dict[str, pd.DataFrame]] = None):
    date_tag = TODAY.strftime("%Y%m%d")
    root_dir = os.path.join(OUTPUT_DIR, date_tag)
    os.makedirs(root_dir, exist_ok=True)
    print("=" * 76)
    print(f"  A股市场日度跟踪系统 v1.22    {TODAY.strftime('%Y-%m-%d')}")
    print(f"  宽基:{len(BROAD_INDICES)}  ETF:{len(SECTOR_ETFS)}  合计:{len(ALL_TARGETS)}")
    print(f"  展示窗口:{DISPLAY_BARS}日  MA200预热:{PREHEAT_BARS}日  鱼盆:MA5/MA20  输出:{os.path.abspath(root_dir)}")
    print("=" * 76)
    all_charts = {"per_target": {}}
    idx_fish_recs = []
    etf_fish_recs = []

    if data_provider and "000300" in data_provider:
        benchmark_raw = _normalize_ohlc(data_provider["000300"])
        benchmark_changed = True
        benchmark_df = prepare_analysis_df(benchmark_raw, kind="index", symbol="000300")
    else:
        benchmark_raw = fetch_kline_cached("000300", "index", "沪深300")
        benchmark_changed = CACHE_LAST_CHANGED.get(("index", "000300"), True)
        benchmark_df = get_analysis_df_cached("000300", "index", "沪深300", benchmark_raw, force_recalc=False)
    benchmark_close = benchmark_df["收盘"] if benchmark_df is not None and not benchmark_df.empty else pd.Series(dtype=float)

    for name, symbol, pepb_sym, kind, legu_code in ALL_TARGETS:
        print(f"\n▶ {name} ({symbol})  [{kind.upper()}]")
        save_dir = os.path.join(root_dir, "indices" if kind == "index" else "etfs", safe_fn(name))
        os.makedirs(save_dir, exist_ok=True)
        if data_provider and symbol in data_provider:
            df_raw = _normalize_ohlc(data_provider[symbol])
            df_analysis = prepare_analysis_df(df_raw, kind=kind, benchmark_close=benchmark_close if kind == "etf" else None, symbol=symbol)
        else:
            df_raw = fetch_kline_cached(symbol, kind, name)
            if df_raw is None or df_raw.empty:
                print(f"    [WARN] {name} 无原始K线数据，跳过")
                continue
            # ETF的超额收益依赖沪深300；若基准有增量，则ETF指标也要重算。
            force_recalc = bool(benchmark_changed) if kind == "etf" else False
            df_analysis = get_analysis_df_cached(symbol, kind, name, df_raw, benchmark_close=benchmark_close if kind == "etf" else None, force_recalc=force_recalc)
        if df_analysis.empty:
            print(f"    [WARN] {name} 处理后数据为空，跳过")
            continue
        df_view = select_display_window(df_analysis, DISPLAY_BARS)
        if df_view.empty:
            print(f"    [WARN] {name} 展示窗口为空，跳过")
            continue
        kline_path = plot_daily_kline(name, symbol, df_view, save_dir, kind)
        df_fish, fish_path = plot_fish_bowl(name, symbol, df_analysis, save_dir, kind)
        all_charts["per_target"][name] = {"kline": kline_path, "fish": fish_path}
        export_excel(name, symbol, df_analysis, df_fish, save_dir, kind)
        rec = {
            "标的": f"{name}({symbol})" if kind == "etf" else name,
            "收盘价": _safe_last(df_view["收盘"], 3), "日涨跌(%)": _safe_last(df_view["涨跌幅"], 2),
            "近5日涨跌(%)": _safe_last(df_view["近5日涨跌(%)"], 2), "近20日涨跌(%)": _safe_last(df_view["近20日涨跌(%)"], 2),
            "MA5": _safe_last(df_view["MA5"], 3), "MA20": _safe_last(df_view["MA20"], 3),
            "MA5乖离率(%)": _safe_last(df_view["MA5乖离率(%)"], 2), "MA20乖离率(%)": _safe_last(df_view["MA20乖离率(%)"], 2), "MA20乖离率分位(%)": _safe_last(df_view["MA20乖离率分位(%)"], 1),
            "成交量": _safe_last(df_view["成交量"], 0), "MA5均量": _safe_last(df_view["MA5均量"], 0), "MA20均量": _safe_last(df_view["MA20均量"], 0),
            "量比": _safe_last(df_view["量比"], 2), "MA5-MA20交叉日": _get_last_cross_label(df_fish),
            "金叉/死叉天数": _safe_last_text(df_view["金叉/死叉天数"]) if "金叉/死叉天数" in df_view.columns else "N/A",
            "区间涨跌幅(%)": _safe_last(df_view["区间涨跌幅(%)"], 2) if "区间涨跌幅(%)" in df_view.columns else "N/A",
        }
        if kind == "etf":
            rec["超额收益(%)"] = _safe_last(df_view["超额收益(%)"], 2) if "超额收益(%)" in df_view.columns else "N/A"
            rec["资金净流入估算(亿元)"] = _safe_last(df_view["资金净流入估算(亿元)"], 2) if "资金净流入估算(亿元)" in df_view.columns else "N/A"
            rec["20日滚动波动率(%)"] = _safe_last(df_view["20日滚动波动率(%)"], 2) if "20日滚动波动率(%)" in df_view.columns else "N/A"
            etf_fish_recs.append(rec)
        else:
            idx_fish_recs.append(rec)

    all_charts["index_fish_snapshot"] = plot_fishbowl_snapshot(idx_fish_recs, root_dir, "宽基鱼盆快照.png", f"A股宽基指数鱼盆快照    {TODAY.strftime('%Y年%m月%d日')}", kind="index")
    all_charts["etf_fish_snapshot"] = plot_fishbowl_snapshot(etf_fish_recs, root_dir, "ETF鱼盆快照.png", f"行业ETF鱼盆快照    {TODAY.strftime('%Y年%m月%d日')}", kind="etf")
    all_charts["etf_heatmap"] = plot_etf_heatmap(etf_fish_recs, root_dir)
    all_charts["snapshot_excel"] = export_snapshot_excel(idx_fish_recs, etf_fish_recs, root_dir)
    all_charts["pdf"] = export_pdf_report(root_dir, all_charts, date_tag)
    print("\n" + "=" * 76)
    print("  日度跟踪完成")
    print(f"  输出目录: {os.path.abspath(root_dir)}")
    print("=" * 76)
    return {"root_dir": root_dir, "charts": all_charts, "index_records": idx_fish_recs, "etf_records": etf_fish_recs}




# ══════════════════════════════════════════════════════════════════════════════
# § v1.18 成交量口径修复覆盖函数
# ══════════════════════════════════════════════════════════════════════════════

# v1.18 统一成交量内部口径为“手”。
# 说明：
# 1) 多数宽基指数源（东方财富/新浪）返回的成交量更接近“股”口径，
#    需要除以100转为“手”，才与券商软件中常见的亿手口径一致。
# 2) 中证2000在当前兜底链路中常来自乐咕 close-only/简化源，缓存中的成交量已接近券商亿手口径，
#    不再二次除以100。
# 3) ETF源返回口径在本程序历史版本中已经与万手口径一致，保持不变。
# 4) 所有均量和量比均由规范化后的同一条“成交量”序列 rolling 计算，避免混用单位。
INDEX_VOLUME_DIV100_SYMBOLS = "ALL_INDEX_AND_ALL_ETF"  # v1.22：所有宽基指数与ETF历史成交量统一÷100
INDEX_VOLUME_KEEP_SYMBOLS = set()


def normalize_volume_units(vol: pd.Series, kind: str = "", symbol: str = "", amount: Optional[pd.Series] = None) -> pd.Series:
    """统一成交量内部口径为“手”。

    v1.22口径：
    - 所有宽基指数：历史每日成交量统一除以100；
    - 所有行业ETF：历史每日成交量统一除以100；
    - 0或负数视为缺失，避免参与均量和量比计算；
    - amount参数仅保留兼容旧调用，不再用成交额/成交量自动猜单位。
    """
    idx = vol.index if hasattr(vol, "index") else None
    v = pd.to_numeric(vol, errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float64")
    v = v.where(v > 0)
    if not v.notna().any():
        return pd.Series(np.nan, index=idx, dtype="float64")
    scale = 100.0 if kind in {"index", "etf"} else 1.0
    return (v / scale).astype("float64")


def calc_volume_indicators(vol):
    """由同一成交量序列计算MA5均量、MA20均量和量比。"""
    vol = pd.to_numeric(vol, errors="coerce").replace([np.inf, -np.inf], np.nan)
    vol = vol.where(vol > 0)
    vma5 = vol.rolling(5, min_periods=1).mean()
    vma20 = vol.rolling(20, min_periods=1).mean()
    ratio = vol / vma5.replace(0, np.nan)
    return vma5, vma20, ratio


def _assert_volume_indicators(out: pd.DataFrame, name: str = ""):
    """强制所有均量与量比回到成交量自身rolling口径。"""
    if out is None or out.empty or "成交量" not in out.columns:
        return out
    vol = pd.to_numeric(out["成交量"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    vol = vol.where(vol > 0)
    out["成交量"] = vol
    out["MA5均量"] = vol.rolling(5, min_periods=1).mean()
    out["MA20均量"] = vol.rolling(20, min_periods=1).mean()
    out["量比"] = vol / out["MA5均量"].replace(0, np.nan)
    return out


def prepare_analysis_df(df_raw: pd.DataFrame, kind="etf", benchmark_close: Optional[pd.Series] = None, symbol: str = "") -> pd.DataFrame:
    """基于全历史原始K线计算完整指标表（v1.18成交量修正版）。"""
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    df = df_raw.copy().sort_index()
    cc, oc, hc, lc, vc, ac, gc = infer_close(df), infer_open(df), infer_high(df), infer_low(df), infer_vol(df), infer_amt(df), infer_chg(df)
    if cc is None:
        return pd.DataFrame()
    out = pd.DataFrame(index=df.index)
    out["收盘"] = pd.to_numeric(df[cc], errors="coerce")
    out["开盘"] = pd.to_numeric(df[oc], errors="coerce") if oc and oc in df.columns else out["收盘"]
    out["最高"] = pd.to_numeric(df[hc], errors="coerce") if hc and hc in df.columns else out[["开盘", "收盘"]].max(axis=1)
    out["最低"] = pd.to_numeric(df[lc], errors="coerce") if lc and lc in df.columns else out[["开盘", "收盘"]].min(axis=1)
    if vc and vc in df.columns:
        out["成交量"] = normalize_volume_units(pd.to_numeric(df[vc], errors="coerce"), kind=kind, symbol=symbol)
    else:
        out["成交量"] = pd.Series(np.nan, index=out.index)
    out["成交额"] = pd.to_numeric(df[ac], errors="coerce") if ac and ac in df.columns else out["收盘"] * out["成交量"]
    out["涨跌幅"] = pd.to_numeric(df[gc], errors="coerce") if gc and gc in df.columns else out["收盘"].pct_change() * 100
    for k, v in calc_ma(out["收盘"]).items():
        out[k] = v
    bu, bm, bl = calc_bollinger(out["收盘"])
    out["BOLL上轨"], out["BOLL中轨"], out["BOLL下轨"] = bu, bm, bl
    dif, dea, hist = calc_macd(out["收盘"])
    out["DIF"], out["DEA"], out["MACD柱"] = dif, dea, hist
    k, d, j = calc_kdj(out["最高"], out["最低"], out["收盘"])
    out["K"], out["D"], out["J"] = k, d, j
    out["RSI6"], out["RSI12"], out["RSI24"] = calc_rsi(out["收盘"], 6), calc_rsi(out["收盘"], 12), calc_rsi(out["收盘"], 24)
    out = _assert_volume_indicators(out)
    out["近5日涨跌(%)"], out["近20日涨跌(%)"] = out["收盘"].pct_change(5) * 100, out["收盘"].pct_change(20) * 100
    out["MA5乖离率(%)"] = calc_deviation(out["收盘"], out["MA5"])
    out["MA20乖离率(%)"] = calc_deviation(out["收盘"], out["MA20"])
    out["MA20乖离率分位(%)"] = calc_dev_pct(out["MA20乖离率(%)"])
    out["MA20乖离率有符号分位(%)"] = calc_signed_dev_pct(out["MA20乖离率(%)"], out["MA20乖离率分位(%)"])
    golden, death = find_crossovers(out["MA5"], out["MA20"])
    cross = pd.Series("", index=out.index)
    for dt in golden:
        if dt in cross.index:
            cross.loc[dt] = "金叉↑"
    for dt in death:
        if dt in cross.index:
            cross.loc[dt] = "死叉↓"
    out["MA5-MA20交叉"] = cross
    out = add_fish_interval_columns(out)
    if kind == "etf":
        if benchmark_close is not None and not benchmark_close.empty:
            bench = benchmark_close.reindex(out.index).ffill()
            bench_ret = bench.pct_change() * 100
            out["沪深300涨跌幅(%)"] = bench_ret
            out["超额收益(%)"] = out["涨跌幅"] - bench_ret
        direction = _price_direction_factor(out["开盘"], out["最高"], out["最低"], out["收盘"])
        abnormal_amt = (out["成交额"] - out["成交额"].rolling(5, min_periods=1).mean()).clip(lower=0)
        out["资金净流入估算(亿元)"] = abnormal_amt * direction / 1e8
        out["20日滚动波动率(%)"] = out["收盘"].pct_change().rolling(20, min_periods=20).std() * 100
    return out.dropna(how="all")


def get_analysis_df_cached(symbol: str, kind: str, name: str, df_raw: pd.DataFrame, benchmark_close: Optional[pd.Series] = None, force_recalc: bool = False) -> pd.DataFrame:
    """按“有增量才重算”的原则获取全历史指标表；v1.18因成交量口径升级，缓存版本强制更新。"""
    changed = CACHE_LAST_CHANGED.get((kind, symbol), True)
    if CACHE_ENABLED and (not changed) and (not force_recalc):
        cached = _read_cached_analysis(symbol, kind)
        if not cached.empty:
            return cached
    df = prepare_analysis_df(df_raw, kind=kind, benchmark_close=benchmark_close, symbol=symbol)
    if CACHE_ENABLED and df is not None and not df.empty:
        _write_cached_analysis(symbol, kind, df)
        reason = "基准更新" if force_recalc and kind == "etf" else "原始K线更新/首次计算"
        print(f"    [CACHE] 已重算并写入指标缓存（{len(df)}日，{reason}）")
    return df




# ══════════════════════════════════════════════════════════════════════════════
# § v1.21 成交量折算与ETF展示单位修复覆盖函数
# ══════════════════════════════════════════════════════════════════════════════

# v1.21 的核心原则：
# 1) 程序内部成交量继续统一保存为“手”；
# 2) 科创50(000688)、北证50(899050)当前数据源返回成交量需在指标计算前÷100；
# 3) 所有ETF当前数据源返回成交量需在指标计算前÷100，避免部分历史日“股/手”混杂；
# 4) ETF图表与ETF快照的成交量/均量统一以“万手”展示，避免亿手口径下精度不足；
# 5) MA5均量、MA20均量、量比只允许由规范化后的“成交量(手)”重新 rolling 计算。
ANALYSIS_CACHE_VERSION = "v122"
RAW_KLINE_CACHE_VERSION = "raw_v122"
VOLUME_LINE_COLOR = "#BDBDBD"

# 仍然禁用系统代理，减少本地无效代理导致的数据源 ProxyError。
USE_SYSTEM_PROXY = False
if not USE_SYSTEM_PROXY:
    for _proxy_key in ["HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "http_proxy", "https_proxy", "all_proxy"]:
        os.environ.pop(_proxy_key, None)
    try:
        import requests as _requests
        _old_session_factory = _requests.Session
        def _no_proxy_session(*args, **kwargs):
            s = _old_session_factory(*args, **kwargs)
            s.trust_env = False
            s.proxies = {}
            return s
        _requests.Session = _no_proxy_session
    except Exception:
        pass

# v1.22成交量折算规则：所有宽基指数和所有ETF的历史每日成交量均÷100，内部单位统一为“手”。
INDEX_VOLUME_DIV100_SYMBOLS = "ALL_INDEX_AND_ALL_ETF"  # v1.22：所有宽基指数与ETF历史成交量统一÷100
INDEX_VOLUME_KEEP_SYMBOLS = set()


def _cache_file_path(symbol: str, kind: str) -> str:
    """v1.22 使用独立原始K线缓存目录，避免旧缓存成交量口径污染。"""
    folder = os.path.join(CACHE_DIR, RAW_KLINE_CACHE_VERSION, kind)
    os.makedirs(folder, exist_ok=True)
    return os.path.join(folder, f"{symbol}.csv")


def _get_volume_display_unit(kind: str, max_vol) -> Tuple[float, str]:
    """图表/快照成交量展示单位。

    ETF统一显示为万手；指数仍按数值大小在亿手/万手之间切换。
    注意：这只影响图表和快照展示，不影响内部计算口径。
    """
    if kind == "etf":
        return 1e4, "万手"
    try:
        mv = float(max_vol) if pd.notna(max_vol) else 0.0
    except Exception:
        mv = 0.0
    return (1e8, "亿手") if mv >= VOLUME_DISPLAY_E_THRESHOLD else (1e4, "万手")


def normalize_volume_units(vol: pd.Series, kind: str = "", symbol: str = "", amount: Optional[pd.Series] = None) -> pd.Series:
    """统一成交量内部口径为“手”。

    v1.22口径：
    - 所有宽基指数：历史每日成交量统一除以100；
    - 所有行业ETF：历史每日成交量统一除以100；
    - 0或负数视为缺失，避免参与均量和量比计算；
    - amount参数仅保留兼容旧调用，不再用成交额/成交量自动猜单位。
    """
    idx = vol.index if hasattr(vol, "index") else None
    v = pd.to_numeric(vol, errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float64")
    v = v.where(v > 0)
    if not v.notna().any():
        return pd.Series(np.nan, index=idx, dtype="float64")
    scale = 100.0 if kind in {"index", "etf"} else 1.0
    return (v / scale).astype("float64")


def calc_volume_indicators(vol):
    """由规范化后的成交量(手)计算 MA5均量、MA20均量和量比。"""
    vol = pd.to_numeric(vol, errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float64")
    vol = vol.where(vol > 0)
    vma5 = vol.rolling(5, min_periods=1).mean()
    vma20 = vol.rolling(20, min_periods=1).mean()
    ratio = vol / vma5.replace(0, np.nan)
    return vma5, vma20, ratio


def _assert_volume_indicators(out: pd.DataFrame, name: str = ""):
    """强制修正均量和量比，防止旧缓存或旧字段污染。"""
    if out is None or out.empty or "成交量" not in out.columns:
        return out
    vol = pd.to_numeric(out["成交量"], errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float64")
    vol = vol.where(vol > 0)
    out["成交量"] = vol
    out["MA5均量"] = vol.rolling(5, min_periods=1).mean()
    out["MA20均量"] = vol.rolling(20, min_periods=1).mean()
    out["量比"] = vol / out["MA5均量"].replace(0, np.nan)
    return out


def prepare_analysis_df(df_raw: pd.DataFrame, kind="etf", benchmark_close: Optional[pd.Series] = None, symbol: str = "") -> pd.DataFrame:
    """基于全历史K线计算完整指标表（v1.22成交量统一折算版）。"""
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    df = df_raw.copy().sort_index()
    cc, oc, hc, lc, vc, ac, gc = infer_close(df), infer_open(df), infer_high(df), infer_low(df), infer_vol(df), infer_amt(df), infer_chg(df)
    if cc is None:
        return pd.DataFrame()

    out = pd.DataFrame(index=df.index)
    out["收盘"] = pd.to_numeric(df[cc], errors="coerce")
    out["开盘"] = pd.to_numeric(df[oc], errors="coerce") if oc and oc in df.columns else out["收盘"]
    out["最高"] = pd.to_numeric(df[hc], errors="coerce") if hc and hc in df.columns else out[["开盘", "收盘"]].max(axis=1)
    out["最低"] = pd.to_numeric(df[lc], errors="coerce") if lc and lc in df.columns else out[["开盘", "收盘"]].min(axis=1)

    if vc and vc in df.columns:
        out["成交量"] = normalize_volume_units(pd.to_numeric(df[vc], errors="coerce"), kind=kind, symbol=symbol)
    else:
        out["成交量"] = pd.Series(np.nan, index=out.index, dtype="float64")

    # 成交额保留原始成交额口径；若源缺失，则用价格×成交量(手)×100近似还原成金额。
    out["成交额"] = pd.to_numeric(df[ac], errors="coerce") if ac and ac in df.columns else out["收盘"] * out["成交量"] * 100
    out["涨跌幅"] = pd.to_numeric(df[gc], errors="coerce") if gc and gc in df.columns else out["收盘"].pct_change() * 100

    for k, v in calc_ma(out["收盘"]).items():
        out[k] = v
    bu, bm, bl = calc_bollinger(out["收盘"])
    out["BOLL上轨"], out["BOLL中轨"], out["BOLL下轨"] = bu, bm, bl
    dif, dea, hist = calc_macd(out["收盘"])
    out["DIF"], out["DEA"], out["MACD柱"] = dif, dea, hist
    k, d, j = calc_kdj(out["最高"], out["最低"], out["收盘"])
    out["K"], out["D"], out["J"] = k, d, j
    out["RSI6"], out["RSI12"], out["RSI24"] = calc_rsi(out["收盘"], 6), calc_rsi(out["收盘"], 12), calc_rsi(out["收盘"], 24)

    out = _assert_volume_indicators(out)
    out["近5日涨跌(%)"], out["近20日涨跌(%)"] = out["收盘"].pct_change(5) * 100, out["收盘"].pct_change(20) * 100
    out["MA5乖离率(%)"] = calc_deviation(out["收盘"], out["MA5"])
    out["MA20乖离率(%)"] = calc_deviation(out["收盘"], out["MA20"])
    out["MA20乖离率分位(%)"] = calc_dev_pct(out["MA20乖离率(%)"])
    out["MA20乖离率有符号分位(%)"] = calc_signed_dev_pct(out["MA20乖离率(%)"], out["MA20乖离率分位(%)"])

    golden, death = find_crossovers(out["MA5"], out["MA20"])
    cross = pd.Series("", index=out.index, dtype="object")
    for dt in golden:
        if dt in cross.index:
            cross.loc[dt] = "金叉↑"
    for dt in death:
        if dt in cross.index:
            cross.loc[dt] = "死叉↓"
    out["MA5-MA20交叉"] = cross.fillna("")
    out = add_fish_interval_columns(out)

    if kind == "etf":
        if benchmark_close is not None and not benchmark_close.empty:
            bench = benchmark_close.reindex(out.index).ffill()
            bench_ret = bench.pct_change() * 100
            out["沪深300涨跌幅(%)"] = bench_ret
            out["超额收益(%)"] = out["涨跌幅"] - bench_ret
        direction = _price_direction_factor(out["开盘"], out["最高"], out["最低"], out["收盘"])
        abnormal_amt = (out["成交额"] - out["成交额"].rolling(5, min_periods=1).mean()).clip(lower=0)
        out["资金净流入估算(亿元)"] = abnormal_amt * direction / 1e8
        out["20日滚动波动率(%)"] = out["收盘"].pct_change().rolling(20, min_periods=20).std() * 100
    return out.dropna(how="all")


def get_analysis_df_cached(symbol: str, kind: str, name: str, df_raw: pd.DataFrame, benchmark_close: Optional[pd.Series] = None, force_recalc: bool = False) -> pd.DataFrame:
    """按“有增量才重算”的原则获取全历史指标表；v1.22因所有宽基指数与ETF成交量统一÷100而强制新缓存。"""
    changed = CACHE_LAST_CHANGED.get((kind, symbol), True)
    if CACHE_ENABLED and (not changed) and (not force_recalc):
        cached = _read_cached_analysis(symbol, kind)
        if not cached.empty:
            return cached
    df = prepare_analysis_df(df_raw, kind=kind, benchmark_close=benchmark_close, symbol=symbol)
    if CACHE_ENABLED and df is not None and not df.empty:
        _write_cached_analysis(symbol, kind, df)
        reason = "基准更新" if force_recalc and kind == "etf" else "原始K线更新/首次计算"
        print(f"    [CACHE] 已重算并写入指标缓存（{len(df)}日，{reason}）")
    return df


def _get_last_cross_label(df_fish: pd.DataFrame) -> str:
    """最后一次金叉/死叉标签；只接受明确交叉事件。"""
    if df_fish is None or df_fish.empty or "MA5-MA20交叉" not in df_fish.columns:
        return "N/A"
    s = df_fish["MA5-MA20交叉"].fillna("").astype(str).str.strip()
    ev = s[s.isin(["金叉↑", "死叉↓"])]
    if ev.empty:
        return "N/A"
    idx = ev.index[-1]
    val = ev.iloc[-1]
    return f"{val} {idx.strftime('%Y-%m-%d')}"

# 覆盖鱼盆第三面板分位趋势线颜色。
C["signed_pct_line"] = VOLUME_LINE_COLOR


# ══════════════════════════════════════════════════════════════════════════════
# § v1.23 行业指数分析板块
# ══════════════════════════════════════════════════════════════════════════════

TRACKER_VERSION = "v1.23"
ANALYSIS_CACHE_VERSION = "v123"
RAW_KLINE_CACHE_VERSION = "raw_v123"
INCLUDE_SUPPLEMENT_SECTOR_INDEXES = True
SECTOR_INDEX_VALUATION_ENABLED = True
SECTOR_INDEX_REQUEST_SLEEP_SECONDS = 0.15

import time


def _target_display_title(name, symbol, kind):
    """图表标题：ETF和行业指数保留代码，宽基沿用原来的简洁名称。"""
    if kind in {"etf", "sector_index"} and symbol:
        return f"{name}({symbol})"
    return str(name)


def _target_fn_prefix(name, symbol, kind):
    if kind in {"etf", "sector_index"} and symbol:
        return safe_fn(f"{name}_{symbol}")
    return safe_fn(name)


SECTOR_INDEXES_BASE = [
    dict(tag="银行", etf_name="银行ETF", etf_code="512800", csi_code="399986", cni_code=None, index_name="中证银行指数", funddb_name="中证银行", sw_proxy_code="801780", sw_proxy_level="一级行业"),
    dict(tag="证券", etf_name="证券ETF", etf_code="512880", csi_code="399975", cni_code=None, index_name="中证全指证券公司指数", funddb_name="证券公司", sw_proxy_code="801193", sw_proxy_level="二级行业"),
    dict(tag="证券保险", etf_name="证券保险ETF", etf_code="512070", csi_code="H30035", cni_code=None, index_name="沪深300非银行金融指数", funddb_name="300非银", sw_proxy_code="801790", sw_proxy_level="一级行业"),
    dict(tag="房地产", etf_name="房地产ETF", etf_code="512200", csi_code="931775", cni_code=None, index_name="中证全指房地产指数", funddb_name="中证全指房地产", sw_proxy_code="801180", sw_proxy_level="一级行业"),
    dict(tag="红利", etf_name="红利ETF", etf_code="510880", csi_code="000015", cni_code=None, index_name="上证红利指数", funddb_name="上证红利", sw_proxy_code=None, sw_proxy_level=None),
    dict(tag="创新药", etf_name="创新药ETF", etf_code="159992", csi_code="931152", cni_code=None, index_name="中证创新药产业指数", funddb_name="创新药", sw_proxy_code="801150", sw_proxy_level="一级行业"),
    dict(tag="医药", etf_name="医药ETF", etf_code="512010", csi_code="000913", cni_code=None, index_name="沪深300医药卫生指数", funddb_name="300医药", sw_proxy_code="801150", sw_proxy_level="一级行业"),
    dict(tag="医疗", etf_name="医疗ETF", etf_code="512170", csi_code="399989", cni_code=None, index_name="中证医疗指数", funddb_name="中证医疗", sw_proxy_code="801150", sw_proxy_level="一级行业"),
    dict(tag="AI", etf_name="AI ETF", etf_code="515070", csi_code="930713", cni_code=None, index_name="中证人工智能主题指数", funddb_name="人工智能", sw_proxy_code="801750", sw_proxy_level="一级行业"),
    dict(tag="半导体", etf_name="半导体ETF", etf_code="512480", csi_code="H30184", cni_code=None, index_name="中证全指半导体产品与设备指数", funddb_name="半导体", sw_proxy_code="801080", sw_proxy_level="一级行业"),
    dict(tag="半导体设备", etf_name="半导体设备ETF", etf_code="159516", csi_code="931743", cni_code=None, index_name="中证半导体材料设备主题指数", funddb_name="半导体材料设备", sw_proxy_code="801080", sw_proxy_level="一级行业"),
    dict(tag="科创芯片", etf_name="科创芯片ETF", etf_code="588200", csi_code="950162", cni_code=None, index_name="上证科创板芯片指数", funddb_name="科创芯片", sw_proxy_code="801080", sw_proxy_level="一级行业"),
    dict(tag="计算机", etf_name="计算机ETF", etf_code="159998", csi_code="930651", cni_code=None, index_name="中证计算机主题指数", funddb_name="计算机", sw_proxy_code="801750", sw_proxy_level="一级行业"),
    dict(tag="通信", etf_name="通信ETF", etf_code="515880", csi_code="931160", cni_code=None, index_name="中证全指通信设备指数", funddb_name="通信", sw_proxy_code="801770", sw_proxy_level="一级行业"),
    dict(tag="消费电子", etf_name="消费电子ETF", etf_code="159732", csi_code=None, cni_code="980030", index_name="国证消费电子主题指数", funddb_name="消费电子", sw_proxy_code="801080", sw_proxy_level="一级行业"),
    dict(tag="游戏", etf_name="游戏ETF", etf_code="516010", csi_code="930901", cni_code=None, index_name="中证动漫游戏指数", funddb_name="动漫游戏", sw_proxy_code="801760", sw_proxy_level="一级行业"),
    dict(tag="机器人", etf_name="机器人ETF", etf_code="159770", csi_code="H30590", cni_code=None, index_name="中证机器人指数", funddb_name="机器人", sw_proxy_code="801890", sw_proxy_level="一级行业"),
    dict(tag="新能源", etf_name="新能源ETF", etf_code="516160", csi_code="399808", cni_code=None, index_name="中证新能源指数", funddb_name="中证新能源", sw_proxy_code="801730", sw_proxy_level="一级行业"),
    dict(tag="新能源车", etf_name="新能源车ETF", etf_code="515030", csi_code="399976", cni_code=None, index_name="CS新能车指数", funddb_name="CS新能车", sw_proxy_code="801880", sw_proxy_level="一级行业"),
    dict(tag="光伏", etf_name="光伏ETF", etf_code="515790", csi_code="931151", cni_code=None, index_name="中证光伏产业指数", funddb_name="光伏产业", sw_proxy_code="801730", sw_proxy_level="一级行业"),
    dict(tag="电网", etf_name="电网ETF", etf_code="159326", csi_code="931994", cni_code=None, index_name="中证电网设备主题指数", funddb_name="电网设备", sw_proxy_code="801730", sw_proxy_level="一级行业"),
    dict(tag="煤炭", etf_name="煤炭ETF", etf_code="515220", csi_code="399998", cni_code=None, index_name="中证煤炭指数", funddb_name="中证煤炭", sw_proxy_code="801950", sw_proxy_level="一级行业"),
    dict(tag="消费", etf_name="消费ETF", etf_code="159928", csi_code="000932", cni_code=None, index_name="中证主要消费指数", funddb_name="中证消费", sw_proxy_code="801120", sw_proxy_level="一级行业"),
    dict(tag="酒", etf_name="酒ETF", etf_code="512690", csi_code="399987", cni_code=None, index_name="中证酒指数", funddb_name="中证酒", sw_proxy_code="801120", sw_proxy_level="一级行业"),
    dict(tag="家电", etf_name="家电ETF", etf_code="159996", csi_code="930697", cni_code=None, index_name="中证全指家用电器指数", funddb_name="家用电器", sw_proxy_code="801110", sw_proxy_level="一级行业"),
    dict(tag="传媒", etf_name="传媒ETF", etf_code="512980", csi_code="399971", cni_code=None, index_name="中证传媒指数", funddb_name="中证传媒", sw_proxy_code="801760", sw_proxy_level="一级行业"),
    dict(tag="有色", etf_name="有色ETF", etf_code="512400", csi_code="000819", cni_code=None, index_name="中证申万有色金属指数", funddb_name="有色金属", sw_proxy_code="801050", sw_proxy_level="一级行业"),
    dict(tag="化工", etf_name="化工ETF", etf_code="159870", csi_code="000813", cni_code=None, index_name="中证细分化工产业主题指数", funddb_name="细分化工", sw_proxy_code="801030", sw_proxy_level="一级行业"),
    dict(tag="石油", etf_name="石油ETF", etf_code="561360", csi_code="H30198", cni_code=None, index_name="中证油气产业指数", funddb_name="中证能源", sw_proxy_code="801960", sw_proxy_level="一级行业"),
    dict(tag="建材", etf_name="建材ETF", etf_code="159745", csi_code="931009", cni_code=None, index_name="中证全指建筑材料指数", funddb_name="建筑材料", sw_proxy_code="801710", sw_proxy_level="一级行业"),
    dict(tag="农业", etf_name="农业ETF", etf_code="159825", csi_code="000949", cni_code=None, index_name="中证农业主题指数", funddb_name="中证农业", sw_proxy_code="801010", sw_proxy_level="一级行业"),
    dict(tag="军工", etf_name="军工ETF", etf_code="512660", csi_code="399967", cni_code=None, index_name="中证军工指数", funddb_name="中证军工", sw_proxy_code="801740", sw_proxy_level="一级行业"),
    dict(tag="卫星", etf_name="卫星ETF", etf_code="159206", csi_code=None, cni_code="980018", index_name="国证商用卫星通信产业指数", funddb_name="卫星通信", sw_proxy_code="801770", sw_proxy_level="一级行业"),
    dict(tag="中概互联", etf_name="中概互联ETF", etf_code="513050", csi_code="H30533", cni_code=None, index_name="中证海外中国互联网50指数", funddb_name="中概互联", sw_proxy_code=None, sw_proxy_level=None),
]

SECTOR_INDEXES_SUPPLEMENT = [
    dict(tag="公用事业", etf_name="补充指数", etf_code=None, csi_code="000995", cni_code=None, index_name="中证全指公用事业指数", funddb_name="公用事业", sw_proxy_code="801160", sw_proxy_level="一级行业"),
    dict(tag="电力", etf_name="补充指数", etf_code=None, csi_code="H30199", cni_code=None, index_name="中证全指电力公用事业指数", funddb_name="电力", sw_proxy_code="801161", sw_proxy_level="二级行业"),
    dict(tag="绿色电力", etf_name="补充指数", etf_code=None, csi_code="931897", cni_code=None, index_name="中证绿色电力指数", funddb_name="绿色电力", sw_proxy_code="801160", sw_proxy_level="一级行业"),
    dict(tag="交通运输", etf_name="补充指数", etf_code=None, csi_code="H30171", cni_code=None, index_name="中证全指运输指数", funddb_name="交通运输", sw_proxy_code="801170", sw_proxy_level="一级行业"),
    dict(tag="钢铁", etf_name="补充指数", etf_code=None, csi_code="932113", cni_code=None, index_name="中证全指钢铁行业指数", funddb_name="钢铁", sw_proxy_code="801040", sw_proxy_level="一级行业"),
    dict(tag="建筑装饰", etf_name="补充指数", etf_code=None, csi_code="932117", cni_code=None, index_name="中证全指建筑装饰行业指数", funddb_name="建筑装饰", sw_proxy_code="801720", sw_proxy_level="一级行业"),
    dict(tag="基建工程", etf_name="补充指数", etf_code=None, csi_code="399995", cni_code=None, index_name="中证基建工程指数", funddb_name="基建工程", sw_proxy_code="801720", sw_proxy_level="一级行业"),
    dict(tag="机械设备", etf_name="补充指数", etf_code=None, csi_code="000812", cni_code=None, index_name="中证细分机械设备产业主题指数", funddb_name="细分机械", sw_proxy_code="801890", sw_proxy_level="一级行业"),
    dict(tag="机械制造", etf_name="补充指数", etf_code=None, csi_code="932119", cni_code=None, index_name="中证全指机械制造行业指数", funddb_name="机械制造", sw_proxy_code="801890", sw_proxy_level="一级行业"),
    dict(tag="工业母机", etf_name="补充指数", etf_code=None, csi_code="931866", cni_code=None, index_name="中证机床指数", funddb_name="机床", sw_proxy_code="801890", sw_proxy_level="一级行业"),
    dict(tag="5G通信/CPO代理", etf_name="补充指数", etf_code=None, csi_code="931079", cni_code=None, index_name="中证5G通信主题指数", funddb_name="5G通信", sw_proxy_code="801770", sw_proxy_level="一级行业"),
    dict(tag="电子50/PCB代理", etf_name="补充指数", etf_code=None, csi_code="931461", cni_code=None, index_name="中证电子50指数", funddb_name="电子50", sw_proxy_code="801080", sw_proxy_level="一级行业"),
    dict(tag="电池/储能代理", etf_name="补充指数", etf_code=None, csi_code="931719", cni_code=None, index_name="中证电池主题指数", funddb_name="电池", sw_proxy_code="801730", sw_proxy_level="一级行业"),
    dict(tag="风电", etf_name="补充指数", etf_code=None, csi_code="931672", cni_code=None, index_name="中证风电产业指数", funddb_name="风电", sw_proxy_code="801730", sw_proxy_level="一级行业"),
]

SECTOR_INDICES = SECTOR_INDEXES_BASE + (SECTOR_INDEXES_SUPPLEMENT if INCLUDE_SUPPLEMENT_SECTOR_INDEXES else [])


def _sector_index_symbol(item: dict) -> str:
    return str(item.get("csi_code") or item.get("cni_code") or item.get("etf_code") or "").strip()


SECTOR_INDEX_BY_SYMBOL = {_sector_index_symbol(item): item for item in SECTOR_INDICES if _sector_index_symbol(item)}


def _sector_index_name(item: dict) -> str:
    return str(item.get("tag") or item.get("index_name") or _sector_index_symbol(item))


def _normalize_sector_index_kline(df: pd.DataFrame, item: dict, source: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    rename_map = {
        "date": "日期", "Date": "日期", "时间": "日期", "交易日期": "日期",
        "开盘价": "开盘", "最高价": "最高", "最低价": "最低", "收盘价": "收盘",
        "成交金额": "成交额", "成交额(元)": "成交额", "amount": "成交额", "Amount": "成交额",
        "volume": "成交量", "Volume": "成交量", "pct_chg": "涨跌幅", "change_pct": "涨跌幅",
        "涨跌": "涨跌额", "滚动市盈率": "PE_TTM_中证K线",
    }
    df = df.rename(columns={c: rename_map[c] for c in df.columns if c in rename_map})
    df = df.loc[:, ~df.columns.duplicated(keep="first")]
    df = _normalize_ohlc(df)
    if df.empty:
        return df
    start_ts = pd.Timestamp(KLINE_START)
    end_ts = pd.Timestamp(KLINE_END)
    df = df[(df.index >= start_ts) & (df.index <= end_ts)]
    for col in ["开盘", "最高", "最低", "收盘", "成交量", "成交额", "涨跌幅", "涨跌额", "样本数量", "PE_TTM_中证K线"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df["指数标签"] = item.get("tag")
    df["指数代码"] = _sector_index_symbol(item)
    df["指数名称"] = item.get("index_name")
    df["对应ETF"] = item.get("etf_name")
    df["对应ETF代码"] = item.get("etf_code")
    df["K线数据源"] = source
    df = df.sort_index()
    df = df[~df.index.duplicated(keep="last")]
    return df


def fetch_sector_index_kline_one(item: dict) -> pd.DataFrame:
    _require_akshare()
    tag = _sector_index_name(item)
    csi_code = item.get("csi_code")
    if csi_code:
        try:
            df = ak.stock_zh_index_hist_csindex(symbol=csi_code, start_date=KLINE_START, end_date=KLINE_END)
            out = _normalize_sector_index_kline(df, item, "AKShare-中证官网stock_zh_index_hist_csindex")
            if not out.empty:
                print(f"    [SRC-SECTOR] 中证官网指数日线（{tag}/{csi_code}，{len(out)}日）")
                return out
        except Exception as e:
            print(f"    [WARN] 中证行业指数({tag}/{csi_code}): {type(e).__name__}")
        time.sleep(SECTOR_INDEX_REQUEST_SLEEP_SECONDS)

    cni_code = item.get("cni_code")
    if cni_code:
        for symbol in [cni_code, item.get("index_name"), item.get("tag")]:
            if not symbol:
                continue
            try:
                df = ak.index_hist_cni(symbol=symbol)
                out = _normalize_sector_index_kline(df, item, f"AKShare-国证index_hist_cni({symbol})")
                if not out.empty:
                    print(f"    [SRC-SECTOR] 国证指数日线（{tag}/{symbol}，{len(out)}日）")
                    return out
            except Exception as e:
                print(f"    [WARN] 国证行业指数({tag}/{symbol}): {type(e).__name__}")
            time.sleep(SECTOR_INDEX_REQUEST_SLEEP_SECONDS)

    for symbol in [csi_code, cni_code]:
        if not symbol:
            continue
        try:
            if hasattr(ak, "index_zh_a_hist"):
                df = ak.index_zh_a_hist(symbol=symbol, period="daily", start_date=KLINE_START, end_date=KLINE_END)
                out = _normalize_sector_index_kline(df, item, f"AKShare-东方财富index_zh_a_hist({symbol})")
                if not out.empty:
                    print(f"    [SRC-SECTOR] 东方财富指数日线（{tag}/{symbol}，{len(out)}日）")
                    return out
        except Exception as e:
            print(f"    [WARN] 东方财富行业指数({tag}/{symbol}): {type(e).__name__}")
        time.sleep(SECTOR_INDEX_REQUEST_SLEEP_SECONDS)

    print(f"    [WARN] 行业指数K线({tag}) 所有数据源均失败")
    return pd.DataFrame()


_sector_funddb_names_cache = None
_sector_sw_daily_cache = {}


def _sector_normalize_date_col(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()
    df = df.copy()
    date_col = pick_col(df, ["日期", "date", "Date", "交易日期", "发布日期", "时间"])
    if date_col is None and len(df.columns) > 0:
        date_col = df.columns[0]
    if date_col is None:
        return pd.DataFrame()
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df = df.dropna(subset=[date_col]).rename(columns={date_col: "日期"})
    df = df.sort_values("日期")
    return df


def _sector_get_funddb_names() -> List[str]:
    global _sector_funddb_names_cache
    if _sector_funddb_names_cache is not None:
        return _sector_funddb_names_cache
    names = []
    try:
        if hasattr(ak, "index_value_name_funddb"):
            df = ak.index_value_name_funddb()
            if df is not None and not df.empty:
                names = df.iloc[:, 0].dropna().astype(str).unique().tolist()
    except Exception as e:
        print(f"    [WARN] 韭圈儿指数名称列表: {type(e).__name__}")
    _sector_funddb_names_cache = names
    return names


def _sector_funddb_candidates(item: dict) -> List[str]:
    candidates = []
    for key in ["funddb_name", "index_name", "tag"]:
        v = item.get(key)
        if v and str(v) not in candidates:
            candidates.append(str(v))
    extra = []
    for c in candidates:
        s = c
        for token in ["指数", "主题", "产业", "行业", "中证", "国证", "上证", "全指", "沪深300"]:
            s = s.replace(token, "")
        s = s.strip()
        if s and s not in candidates and s not in extra:
            extra.append(s)
    raw = candidates + extra
    names = _sector_get_funddb_names()
    if not names:
        return raw[:8]
    picked = []
    for c in raw:
        if c in names and c not in picked:
            picked.append(c)
    for c in raw:
        for n in names:
            if c and (c in n or n in c) and n not in picked:
                picked.append(n)
    for c in raw:
        if c not in picked:
            picked.append(c)
    return picked[:8]


def _sector_metric_frame(df: pd.DataFrame, metric_col: str, out_col: str) -> pd.DataFrame:
    df = _sector_normalize_date_col(df)
    if df.empty:
        return pd.DataFrame()
    if metric_col not in df.columns:
        for c in df.columns:
            if str(c).strip() == metric_col:
                df = df.rename(columns={c: metric_col})
                break
    if metric_col not in df.columns:
        return pd.DataFrame()
    out = df[["日期", metric_col]].copy().rename(columns={metric_col: out_col})
    out[out_col] = pd.to_numeric(out[out_col], errors="coerce")
    out = out.dropna(subset=[out_col]).set_index("日期").sort_index()
    out = out[~out.index.duplicated(keep="last")]
    return out


def _sector_fetch_funddb_metric(item: dict, indicator: str, out_col: str) -> Tuple[pd.DataFrame, Optional[str]]:
    for name in _sector_funddb_candidates(item):
        try:
            df = ak.index_value_hist_funddb(symbol=name, indicator=indicator)
            out = _sector_metric_frame(df, indicator, out_col)
            if not out.empty:
                return out, name
        except Exception:
            pass
        time.sleep(SECTOR_INDEX_REQUEST_SLEEP_SECONDS)
    return pd.DataFrame(), None


def _sector_fetch_csindex_value(item: dict) -> pd.DataFrame:
    csi_code = item.get("csi_code")
    if not csi_code:
        return pd.DataFrame()
    try:
        df = ak.stock_zh_index_value_csindex(symbol=csi_code)
        df = _sector_normalize_date_col(df)
        if df.empty:
            return pd.DataFrame()
        keep = ["日期"] + [c for c in ["市盈率1", "市盈率2", "股息率1", "股息率2"] if c in df.columns]
        out = df[keep].copy().rename(columns={
            "市盈率1": "PE1_中证估值", "市盈率2": "PE2_中证估值",
            "股息率1": "股息率1_中证估值", "股息率2": "股息率2_中证估值",
        })
        for c in out.columns:
            if c != "日期":
                out[c] = pd.to_numeric(out[c], errors="coerce")
        out = out.set_index("日期").sort_index()
        out = out[~out.index.duplicated(keep="last")]
        return out
    except Exception:
        return pd.DataFrame()


def _sector_fetch_sw_daily(level: str) -> pd.DataFrame:
    global _sector_sw_daily_cache
    if not level:
        return pd.DataFrame()
    key = (level, KLINE_START, KLINE_END)
    if key in _sector_sw_daily_cache:
        return _sector_sw_daily_cache[key]
    try:
        df = ak.index_analysis_daily_sw(symbol=level, start_date=KLINE_START, end_date=KLINE_END)
        df = _sector_normalize_date_col(df)
        if not df.empty:
            for c in ["市盈率", "市净率", "股息率", "收盘指数"]:
                if c in df.columns:
                    df[c] = pd.to_numeric(df[c], errors="coerce")
            _sector_sw_daily_cache[key] = df
            return df
    except Exception:
        pass
    _sector_sw_daily_cache[key] = pd.DataFrame()
    return pd.DataFrame()


def _sector_fetch_sw_proxy(item: dict) -> pd.DataFrame:
    code = item.get("sw_proxy_code")
    level = item.get("sw_proxy_level")
    if not code or not level:
        return pd.DataFrame()
    df = _sector_fetch_sw_daily(level)
    if df.empty or "指数代码" not in df.columns:
        return pd.DataFrame()
    temp = df[df["指数代码"].astype(str).str.replace(".0", "", regex=False) == str(code)].copy()
    if temp.empty:
        return pd.DataFrame()
    keep = ["日期"] + [c for c in ["市盈率", "市净率", "股息率"] if c in temp.columns]
    out = temp[keep].rename(columns={"市盈率": "PE_申万代理", "市净率": "PB_申万代理", "股息率": "股息率_申万代理"})
    out = out.set_index("日期").sort_index()
    out = out[~out.index.duplicated(keep="last")]
    return out


def _merge_indexed_frames(frames: Iterable[pd.DataFrame]) -> pd.DataFrame:
    valid = [f for f in frames if f is not None and not f.empty]
    if not valid:
        return pd.DataFrame()
    out = valid[0].copy()
    for f in valid[1:]:
        out = out.join(f, how="outer")
    return out.sort_index()


def fetch_sector_index_valuation_history(item: dict) -> pd.DataFrame:
    """获取行业指数估值。PE历史分位不在这里算，固定由K线滚动市盈率计算。"""
    if not SECTOR_INDEX_VALUATION_ENABLED:
        return pd.DataFrame()
    csval = _sector_fetch_csindex_value(item)
    funddb_pb, funddb_name_pb = _sector_fetch_funddb_metric(item, "市净率", "市净率_funddb")
    funddb_dy, funddb_name_dy = _sector_fetch_funddb_metric(item, "股息率", "股息率_funddb")
    sw_proxy = _sector_fetch_sw_proxy(item)
    raw = _merge_indexed_frames([csval, funddb_pb, funddb_dy, sw_proxy])
    if raw.empty:
        return pd.DataFrame()

    out = pd.DataFrame(index=raw.index)
    out["PB"] = np.nan
    out["PB来源"] = ""
    for c in ["市净率_funddb", "PB_申万代理"]:
        if c in raw.columns:
            mask = out["PB"].isna() & raw[c].notna()
            out.loc[mask, "PB"] = raw.loc[mask, c]
            out.loc[mask, "PB来源"] = f"韭圈儿历史估值({funddb_name_pb or ''})" if c == "市净率_funddb" else "申万行业日报估值代理"

    out["股息率"] = np.nan
    out["股息率来源"] = ""
    for c in ["股息率2_中证估值", "股息率_funddb", "股息率_申万代理", "股息率1_中证估值"]:
        if c in raw.columns:
            mask = out["股息率"].isna() & raw[c].notna()
            out.loc[mask, "股息率"] = raw.loc[mask, c]
            if c == "股息率2_中证估值":
                src = "中证估值-D/P2"
            elif c == "股息率1_中证估值":
                src = "中证估值-D/P1"
            elif c == "股息率_funddb":
                src = f"韭圈儿历史估值({funddb_name_dy or ''})"
            else:
                src = "申万行业日报估值代理"
            out.loc[mask, "股息率来源"] = src
    return out.sort_index()


def fetch_sector_index_kline(symbol: str, name="") -> pd.DataFrame:
    item = SECTOR_INDEX_BY_SYMBOL.get(str(symbol))
    if item is None:
        item = next((x for x in SECTOR_INDICES if _sector_index_name(x) == name or x.get("index_name") == name), None)
    if item is None:
        print(f"    [WARN] 未找到行业指数映射: {name}/{symbol}")
        return pd.DataFrame()
    kline = fetch_sector_index_kline_one(item)
    if kline.empty:
        return kline
    valuation = fetch_sector_index_valuation_history(item)
    if valuation is not None and not valuation.empty:
        valuation = valuation.copy()
        for source_col in ["PB来源", "股息率来源"]:
            if source_col in valuation.columns:
                valuation[source_col] = valuation[source_col].replace("", np.nan)
        aligned = valuation.reindex(kline.index).ffill()
        kline = kline.join(aligned, how="left")
    return kline


def fetch_kline(symbol: str, kind: str, name="") -> pd.DataFrame:
    if kind == "sector_index":
        return fetch_sector_index_kline(symbol, name)
    return fetch_kline_index(symbol, name) if kind == "index" else fetch_kline_etf(symbol, name)


def normalize_volume_units(vol: pd.Series, kind: str = "", symbol: str = "", amount: Optional[pd.Series] = None) -> pd.Series:
    """统一成交量内部口径为“手”；行业指数沿用宽基指数折算口径。"""
    idx = vol.index if hasattr(vol, "index") else None
    v = pd.to_numeric(vol, errors="coerce").replace([np.inf, -np.inf], np.nan).astype("float64")
    v = v.where(v > 0)
    if not v.notna().any():
        return pd.Series(np.nan, index=idx, dtype="float64")
    scale = 100.0 if kind in {"index", "etf", "sector_index"} else 1.0
    return (v / scale).astype("float64")


def calc_history_percentile(s: pd.Series, positive_only: bool = True) -> pd.Series:
    """全历史静态分位数。PE分位固定用中证K线“滚动市盈率”列。"""
    vals = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    if positive_only:
        vals = vals.where(vals > 0)
    res = pd.Series(np.nan, index=vals.index, dtype="float64")
    valid = vals.dropna()
    if not valid.empty:
        res.loc[valid.index] = valid.rank(method="average", pct=True) * 100
    return res


def _attach_sector_index_raw_valuation(out: pd.DataFrame, df_raw: pd.DataFrame, kind: str = "") -> pd.DataFrame:
    if kind != "sector_index" or out is None or out.empty or df_raw is None or df_raw.empty:
        return out
    df = df_raw.copy().sort_index()
    if "PE_TTM_中证K线" in df.columns:
        pe = pd.to_numeric(df["PE_TTM_中证K线"], errors="coerce").reindex(out.index)
        out["PE_TTM"] = pe
        out["PE_TTM来源"] = pd.Series("", index=out.index, dtype="object")
        out.loc[pe.notna(), "PE_TTM来源"] = "中证K线-滚动市盈率"
        out["PE_TTM历史分位(%)"] = calc_history_percentile(pe, positive_only=True)
    for col in ["PB", "股息率"]:
        if col in df.columns:
            out[col] = pd.to_numeric(df[col], errors="coerce").reindex(out.index)
    for col in ["PB来源", "股息率来源", "指数标签", "指数代码", "指数名称", "对应ETF", "对应ETF代码", "K线数据源"]:
        if col in df.columns:
            out[col] = df[col].reindex(out.index)
    return out


def prepare_analysis_df(df_raw: pd.DataFrame, kind="etf", benchmark_close: Optional[pd.Series] = None, symbol: str = "") -> pd.DataFrame:
    """基于全历史K线计算完整指标表；v1.23增加行业指数估值字段。"""
    if df_raw is None or df_raw.empty:
        return pd.DataFrame()
    df = df_raw.copy().sort_index()
    cc, oc, hc, lc, vc, ac, gc = infer_close(df), infer_open(df), infer_high(df), infer_low(df), infer_vol(df), infer_amt(df), infer_chg(df)
    if cc is None:
        return pd.DataFrame()

    out = pd.DataFrame(index=df.index)
    out["收盘"] = pd.to_numeric(df[cc], errors="coerce")
    out["开盘"] = pd.to_numeric(df[oc], errors="coerce") if oc and oc in df.columns else out["收盘"]
    out["最高"] = pd.to_numeric(df[hc], errors="coerce") if hc and hc in df.columns else out[["开盘", "收盘"]].max(axis=1)
    out["最低"] = pd.to_numeric(df[lc], errors="coerce") if lc and lc in df.columns else out[["开盘", "收盘"]].min(axis=1)

    if vc and vc in df.columns:
        out["成交量"] = normalize_volume_units(pd.to_numeric(df[vc], errors="coerce"), kind=kind, symbol=symbol)
    else:
        out["成交量"] = pd.Series(np.nan, index=out.index, dtype="float64")

    out["成交额"] = pd.to_numeric(df[ac], errors="coerce") if ac and ac in df.columns else out["收盘"] * out["成交量"] * 100
    out["涨跌幅"] = pd.to_numeric(df[gc], errors="coerce") if gc and gc in df.columns else out["收盘"].pct_change() * 100
    out = _attach_sector_index_raw_valuation(out, df, kind=kind)

    for k, v in calc_ma(out["收盘"]).items():
        out[k] = v
    bu, bm, bl = calc_bollinger(out["收盘"])
    out["BOLL上轨"], out["BOLL中轨"], out["BOLL下轨"] = bu, bm, bl
    dif, dea, hist = calc_macd(out["收盘"])
    out["DIF"], out["DEA"], out["MACD柱"] = dif, dea, hist
    k, d, j = calc_kdj(out["最高"], out["最低"], out["收盘"])
    out["K"], out["D"], out["J"] = k, d, j
    out["RSI6"], out["RSI12"], out["RSI24"] = calc_rsi(out["收盘"], 6), calc_rsi(out["收盘"], 12), calc_rsi(out["收盘"], 24)

    out = _assert_volume_indicators(out)
    out["近5日涨跌(%)"], out["近20日涨跌(%)"] = out["收盘"].pct_change(5) * 100, out["收盘"].pct_change(20) * 100
    out["MA5乖离率(%)"] = calc_deviation(out["收盘"], out["MA5"])
    out["MA20乖离率(%)"] = calc_deviation(out["收盘"], out["MA20"])
    out["MA20乖离率分位(%)"] = calc_dev_pct(out["MA20乖离率(%)"])
    out["MA20乖离率有符号分位(%)"] = calc_signed_dev_pct(out["MA20乖离率(%)"], out["MA20乖离率分位(%)"])

    golden, death = find_crossovers(out["MA5"], out["MA20"])
    cross = pd.Series("", index=out.index, dtype="object")
    for dt in golden:
        if dt in cross.index:
            cross.loc[dt] = "金叉↑"
    for dt in death:
        if dt in cross.index:
            cross.loc[dt] = "死叉↓"
    out["MA5-MA20交叉"] = cross.fillna("")
    out = add_fish_interval_columns(out)

    if kind == "etf":
        if benchmark_close is not None and not benchmark_close.empty:
            bench = benchmark_close.reindex(out.index).ffill()
            bench_ret = bench.pct_change() * 100
            out["沪深300涨跌幅(%)"] = bench_ret
            out["超额收益(%)"] = out["涨跌幅"] - bench_ret
        direction = _price_direction_factor(out["开盘"], out["最高"], out["最低"], out["收盘"])
        abnormal_amt = (out["成交额"] - out["成交额"].rolling(5, min_periods=1).mean()).clip(lower=0)
        out["资金净流入估算(亿元)"] = abnormal_amt * direction / 1e8
        out["20日滚动波动率(%)"] = out["收盘"].pct_change().rolling(20, min_periods=20).std() * 100
    return out.dropna(how="all")


def _read_cached_analysis(symbol: str, kind: str) -> pd.DataFrame:
    path = _analysis_cache_file_path(symbol, kind)
    if not os.path.exists(path):
        return pd.DataFrame()
    try:
        df = pd.read_csv(path)
        if "日期" not in df.columns:
            return pd.DataFrame()
        df["日期"] = pd.to_datetime(df["日期"], errors="coerce")
        df = df.dropna(subset=["日期"]).sort_values("日期").set_index("日期")
        text_cols = {
            "MA5-MA20交叉", "金叉/死叉天数", "PE_TTM来源", "PB来源", "股息率来源",
            "指数标签", "指数代码", "指数名称", "对应ETF", "对应ETF代码", "K线数据源",
        }
        for col in df.columns:
            if col not in text_cols:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        print(f"    [CACHE] 读取本地指标缓存（{len(df)}日，{ANALYSIS_CACHE_VERSION}）")
        return df
    except Exception as e:
        print(f"    [WARN] 读取指标缓存失败({symbol}): {e}")
        return pd.DataFrame()


def get_analysis_df_cached(symbol: str, kind: str, name: str, df_raw: pd.DataFrame, benchmark_close: Optional[pd.Series] = None, force_recalc: bool = False) -> pd.DataFrame:
    changed = CACHE_LAST_CHANGED.get((kind, symbol), True)
    if CACHE_ENABLED and (not changed) and (not force_recalc):
        cached = _read_cached_analysis(symbol, kind)
        if not cached.empty:
            return cached
    df = prepare_analysis_df(df_raw, kind=kind, benchmark_close=benchmark_close, symbol=symbol)
    if CACHE_ENABLED and df is not None and not df.empty:
        _write_cached_analysis(symbol, kind, df)
        reason = "基准更新" if force_recalc and kind == "etf" else "原始K线/估值更新或首次计算"
        print(f"    [CACHE] 已重算并写入指标缓存（{len(df)}日，{reason}）")
    return df


def build_fish_data_table(df_full: pd.DataFrame, kind="etf") -> pd.DataFrame:
    if df_full is None or df_full.empty:
        return pd.DataFrame()
    cols = [
        "收盘", "涨跌幅", "MA5", "MA20", "MA20乖离率(%)", "MA20乖离率分位(%)",
        "MA20乖离率有符号分位(%)", "成交量", "MA5均量", "MA20均量", "量比",
        "MA5-MA20交叉", "金叉/死叉天数", "区间涨跌幅(%)",
    ]
    if kind == "etf":
        cols += ["近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)"]
    if kind == "sector_index":
        cols += ["近5日涨跌(%)", "近20日涨跌(%)", "PE_TTM", "PE_TTM历史分位(%)", "PB", "股息率"]
    use = [c for c in cols if c in df_full.columns]
    fish = df_full[use].copy()
    fish = fish.rename(columns={"收盘": "收盘价", "涨跌幅": "日涨跌(%)"})
    return fish


def export_excel(name, symbol, df_analysis, df_fish, save_dir, kind="etf"):
    """导出单标全历史数据表；行业指数额外输出估值数据sheet。"""
    if (df_analysis is None or df_analysis.empty) and (df_fish is None or df_fish.empty):
        print(f"    [SKIP] {name} 无数据，跳过Excel")
        return None
    fn_prefix = _target_fn_prefix(name, symbol, kind)
    os.makedirs(save_dir, exist_ok=True)
    out = os.path.join(save_dir, f"{fn_prefix}_数据表.xlsx")
    try:
        engine = "xlsxwriter"
        try:
            import xlsxwriter  # noqa: F401
        except Exception:
            engine = "openpyxl"
        with pd.ExcelWriter(out, engine=engine) as w:
            if df_analysis is not None and not df_analysis.empty:
                cols1 = ["开盘", "收盘", "最高", "最低", "成交量", "成交额", "涨跌幅", "MA5", "MA10", "MA20", "MA60", "MA200", "BOLL上轨", "BOLL中轨", "BOLL下轨", "近5日涨跌(%)", "近20日涨跌(%)"]
                cols1 = [c for c in cols1 if c in df_analysis.columns]
                df_analysis[cols1].round(4).to_excel(w, sheet_name="日K线数据")
                cols2 = ["DIF", "DEA", "MACD柱", "K", "D", "J", "RSI6", "RSI12", "RSI24", "MA5均量", "MA20均量", "量比", "MA5乖离率(%)", "MA20乖离率(%)", "MA20乖离率分位(%)", "MA20乖离率有符号分位(%)", "MA5-MA20交叉", "金叉/死叉天数", "区间涨跌幅(%)"]
                if kind == "etf":
                    cols2 += ["沪深300涨跌幅(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)"]
                cols2 = [c for c in cols2 if c in df_analysis.columns]
                df_analysis[cols2].round(4).to_excel(w, sheet_name="技术指标")
                if kind == "sector_index":
                    val_cols = ["PE_TTM", "PE_TTM历史分位(%)", "PE_TTM来源", "PB", "PB来源", "股息率", "股息率来源"]
                    val_cols = [c for c in val_cols if c in df_analysis.columns]
                    if val_cols:
                        df_analysis[val_cols].round(4).to_excel(w, sheet_name="估值数据")
            if df_fish is not None and not df_fish.empty:
                df_fish.round(4).to_excel(w, sheet_name="鱼盆数据")
        print(f"    [OK] 数据表  → {os.path.basename(out)}")
        return out
    except Exception as e:
        print(f"    [WARN] Excel导出失败({name}): {e}")
        return None


def _snapshot_id_col(kind="etf"):
    if kind == "index":
        return "指数(代码)"
    if kind == "sector_index":
        return "行业指数(代码)"
    return "ETF(代码)"


def _fish_ordered_cols(kind="etf"):
    tail = ["MA5-MA20交叉日", "金叉/死叉天数", "区间涨跌幅(%)"]
    if kind == "index":
        return ["收盘价", "日涨跌(%)", "MA5", "MA20", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比"] + tail
    if kind == "sector_index":
        return ["收盘价", "日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "PE_TTM", "PE_TTM历史分位(%)", "PB", "股息率", "MA5", "MA20", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比"] + tail
    return ["收盘价", "日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "资金净流入估算(亿元)", "20日滚动波动率(%)", "MA5", "MA20", "MA5乖离率(%)", "MA20乖离率(%)", "MA20乖离率分位(%)", "成交量", "MA5均量", "MA20均量", "量比"] + tail


def _format_snapshot_tables(records: List[dict], kind="etf"):
    id_col = _snapshot_id_col(kind)
    if not records:
        return pd.DataFrame(), [], id_col, ""
    ordered_cols = _fish_ordered_cols(kind)
    df = pd.DataFrame(records)
    for col in ["标的"] + ordered_cols:
        if col not in df.columns:
            df[col] = "N/A"
    df = df[["标的"] + ordered_cols].rename(columns={"标的": id_col})
    text_cols = {id_col, "MA5-MA20交叉日", "金叉/死叉天数"}
    for col in ordered_cols:
        if col not in text_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "MA20乖离率(%)" in df.columns:
        df = df.sort_values(["MA20乖离率(%)", "MA20乖离率分位(%)"], ascending=False, na_position="last")
    vol_cols = [c for c in ["成交量", "MA5均量", "MA20均量"] if c in df.columns]
    max_vol = max((df[c].abs().max() for c in vol_cols if df[c].notna().any()), default=0)
    vol_div, vol_unit = _get_volume_display_unit(kind, max_vol)
    vol_col_labels = {c: f"{c}({vol_unit})" for c in vol_cols}
    display_cols = [vol_col_labels.get(c, c) for c in df.columns]
    cell_text, cell_colors = [], []

    def fmt(col, row):
        v = row.get(col)
        if pd.isna(v) if not isinstance(v, str) else (str(v) == "N/A"):
            return "N/A", "white"
        if col == id_col:
            return str(v), "white"
        if col in {"MA20乖离率分位(%)", "PE_TTM历史分位(%)"}:
            pct_v = float(v)
            if col == "MA20乖离率分位(%)":
                dev_v = float(row.get("MA20乖离率(%)", np.nan)); t = max(0, min(1, pct_v / 100.0))
                if not pd.isna(dev_v):
                    clr = (_lerp((1.0, 0.98, 0.96), (0.85, 0.18, 0.18), t) + (0.85,)) if dev_v >= 0 else (_lerp((0.96, 0.99, 0.97), (0.18, 0.55, 0.22), t) + (0.85,))
                else:
                    clr = "white"
            else:
                t = max(0, min(1, pct_v / 100.0))
                clr = _lerp((0.93, 1.0, 0.94), (1.0, 0.90, 0.90), t) + (0.70,)
            return f"{pct_v:.0f}%", clr
        if col in {"MA5乖离率(%)", "MA20乖离率(%)"}:
            fv = float(v); clr = (1.0, 0.94, 0.94, 0.60) if fv > 0 else (0.94, 1.0, 0.95, 0.60) if fv < 0 else "white"
            return f"{fv:+.2f}%", clr
        if col in {"日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "区间涨跌幅(%)"}:
            fv = float(v); clr = (1.0, 0.90, 0.90, 0.50) if fv >= 0 else (0.90, 1.0, 0.90, 0.50)
            return f"{fv:+.2f}%", clr
        if col in {"20日滚动波动率(%)", "股息率"}:
            return f"{float(v):.2f}%", "white"
        if col in {"MA5-MA20交叉日", "金叉/死叉天数"}:
            sv = str(v); clr = (1.0, 0.92, 0.92, 0.70) if "金叉" in sv else (0.92, 1.0, 0.92, 0.70) if "死叉" in sv else "white"
            return sv, clr
        if col in {"MA5", "MA20", "收盘价"}:
            return f"{float(v):.3f}", "white"
        if col in {"PE_TTM", "PB"}:
            return f"{float(v):.2f}", "white"
        if col in vol_cols:
            return f"{float(v) / vol_div:.2f}", "white"
        if col == "量比":
            fv = float(v); clr = (1.0, 0.93, 0.93, 0.50) if fv > 1.2 else (0.93, 1.0, 0.93, 0.50) if fv < 0.8 else "white"
            return f"{fv:.2f}", clr
        if col == "资金净流入估算(亿元)":
            fv = float(v); clr = (1.0, 0.93, 0.93, 0.55) if fv > 0 else (0.93, 1.0, 0.93, 0.55) if fv < 0 else "white"
            return f"{fv:+.2f}", clr
        try:
            return f"{float(v):.2f}", "white"
        except Exception:
            return str(v), "white"

    for _, row in df.iterrows():
        rt, rc = [], []
        for col in df.columns:
            t, c = fmt(col, row)
            rt.append(t); rc.append(c)
        cell_text.append(rt); cell_colors.append(rc)
    df_text = pd.DataFrame(cell_text, columns=display_cols)
    note = f"成交量/均量单位：{vol_unit}；量比=成交量÷MA5均量"
    if kind == "sector_index":
        note += "；PE分位=中证K线滚动市盈率全历史分位"
    return df_text, cell_colors, id_col, note


def _snapshot_numeric_tables(records: List[dict], kind="etf"):
    id_col = _snapshot_id_col(kind)
    if not records:
        return pd.DataFrame(), [], {}, ""
    ordered_cols = _fish_ordered_cols(kind)
    df = pd.DataFrame(records)
    for col in ["标的"] + ordered_cols:
        if col not in df.columns:
            df[col] = "N/A"
    df = df[["标的"] + ordered_cols].rename(columns={"标的": id_col})
    text_cols = {id_col, "MA5-MA20交叉日", "金叉/死叉天数"}
    for col in ordered_cols:
        if col not in text_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "MA20乖离率(%)" in df.columns:
        df = df.sort_values(["MA20乖离率(%)", "MA20乖离率分位(%)"], ascending=False, na_position="last")
    vol_cols = [c for c in ["成交量", "MA5均量", "MA20均量"] if c in df.columns]
    max_vol = max((df[c].abs().max() for c in vol_cols if df[c].notna().any()), default=0)
    vol_div, vol_unit = _get_volume_display_unit(kind, max_vol)
    rename_map = {c: f"{c}({vol_unit})" for c in vol_cols}
    for col in vol_cols:
        df[col] = df[col] / vol_div
    df = df.rename(columns=rename_map)
    number_formats = {}
    for col in df.columns:
        if col in {id_col, "MA5-MA20交叉日", "金叉/死叉天数"}:
            number_formats[col] = "@"
        elif col in {"日涨跌(%)", "近5日涨跌(%)", "近20日涨跌(%)", "超额收益(%)", "MA5乖离率(%)", "MA20乖离率(%)", "20日滚动波动率(%)", "区间涨跌幅(%)", "股息率"}:
            number_formats[col] = '+0.00"%";-0.00"%";0.00"%"'
        elif col in {"MA20乖离率分位(%)", "PE_TTM历史分位(%)"}:
            number_formats[col] = '0"%"'
        elif col in {"收盘价", "MA5", "MA20"}:
            number_formats[col] = "0.000"
        elif col in {"PE_TTM", "PB", "量比"}:
            number_formats[col] = "0.00"
        elif col == "资金净流入估算(亿元)":
            number_formats[col] = "+0.00;-0.00;0.00"
        else:
            number_formats[col] = "0.00"
    _, colors, _, note = _format_snapshot_tables(records, kind=kind)
    return df.reset_index(drop=True), colors, number_formats, note


def export_snapshot_excel(index_records, etf_records, root_dir: str, sector_index_records=None):
    """导出市场快照Excel：宽基、行业指数、行业ETF分sheet。"""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import PatternFill, Border, Side, Alignment, Font
    except Exception as e:
        print(f"  [WARN] openpyxl不可用，跳过快照数据表: {e}")
        return None

    out = os.path.join(root_dir, "市场快照数据表.xlsx")
    wb = Workbook()
    wb.remove(wb.active)
    thin = Side(style="thin", color="D0D7DE")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    sheets = [("宽基快照", index_records, "index")]
    if sector_index_records is not None:
        sheets.append(("行业指数快照", sector_index_records, "sector_index"))
    sheets.append(("行业ETF快照", etf_records, "etf"))

    for sheet_name, records, kind in sheets:
        df_num, colors, number_formats, note = _snapshot_numeric_tables(records, kind=kind)
        ws = wb.create_sheet(sheet_name)
        if df_num.empty:
            ws.append(["无数据"])
            continue
        ws.append(df_num.columns.tolist())
        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.fill = PatternFill(fill_type="solid", fgColor="EAF2F8")
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = border
        for r_idx, row in enumerate(df_num.itertuples(index=False), start=2):
            ws.append(list(row))
            for c_idx, cell in enumerate(ws[r_idx], start=1):
                col_name = df_num.columns[c_idx - 1]
                cell.alignment = Alignment(horizontal="center", vertical="center")
                cell.border = border
                cell.number_format = number_formats.get(col_name, "General")
                if isinstance(cell.value, float) and np.isnan(cell.value):
                    cell.value = None
                hx = _rgb_to_hex(colors[r_idx - 2][c_idx - 1]) if colors else None
                if hx:
                    cell.fill = PatternFill(fill_type="solid", fgColor=hx)
        ws.freeze_panes = "A2"
        for col_idx, col_name in enumerate(df_num.columns, start=1):
            values = [str(col_name)] + ["" if pd.isna(v) else str(v) for v in df_num.iloc[:, col_idx - 1].tolist()]
            width = min(max(max(len(v) for v in values) + 2, 10), 28)
            ws.column_dimensions[ws.cell(1, col_idx).column_letter].width = width
        note_row = len(df_num) + 3
        ws.cell(note_row, 1).value = note
        ws.cell(note_row, 1).font = Font(italic=True, color="666666")
    wb.save(out)
    print(f"  [OK] 快照数据表 → {out}")
    return out


def _collect_chart_paths_for_pdf(all_charts: dict) -> List[str]:
    paths = []
    for key in ["index_fish_snapshot", "sector_index_fish_snapshot", "etf_fish_snapshot", "etf_heatmap"]:
        p = all_charts.get(key)
        if p and os.path.exists(p):
            paths.append(p)
    for _name, charts in all_charts.get("per_target", {}).items():
        for key in ["kline", "fish"]:
            p = charts.get(key)
            if p and os.path.exists(p):
                paths.append(p)
    return paths


def _target_save_folder(kind: str) -> str:
    if kind == "index":
        return "indices"
    if kind == "sector_index":
        return "sector_indices"
    return "etfs"


def _build_snapshot_record(name: str, symbol: str, kind: str, df_view: pd.DataFrame, df_fish: pd.DataFrame) -> dict:
    rec = {
        "标的": _target_display_title(name, symbol, kind) if kind != "index" else name,
        "收盘价": _safe_last(df_view["收盘"], 3),
        "日涨跌(%)": _safe_last(df_view["涨跌幅"], 2),
        "近5日涨跌(%)": _safe_last(df_view["近5日涨跌(%)"], 2),
        "近20日涨跌(%)": _safe_last(df_view["近20日涨跌(%)"], 2),
        "MA5": _safe_last(df_view["MA5"], 3),
        "MA20": _safe_last(df_view["MA20"], 3),
        "MA5乖离率(%)": _safe_last(df_view["MA5乖离率(%)"], 2),
        "MA20乖离率(%)": _safe_last(df_view["MA20乖离率(%)"], 2),
        "MA20乖离率分位(%)": _safe_last(df_view["MA20乖离率分位(%)"], 1),
        "成交量": _safe_last(df_view["成交量"], 0),
        "MA5均量": _safe_last(df_view["MA5均量"], 0),
        "MA20均量": _safe_last(df_view["MA20均量"], 0),
        "量比": _safe_last(df_view["量比"], 2),
        "MA5-MA20交叉日": _get_last_cross_label(df_fish),
        "金叉/死叉天数": _safe_last_text(df_view["金叉/死叉天数"]) if "金叉/死叉天数" in df_view.columns else "N/A",
        "区间涨跌幅(%)": _safe_last(df_view["区间涨跌幅(%)"], 2) if "区间涨跌幅(%)" in df_view.columns else "N/A",
    }
    if kind == "etf":
        rec["超额收益(%)"] = _safe_last(df_view["超额收益(%)"], 2) if "超额收益(%)" in df_view.columns else "N/A"
        rec["资金净流入估算(亿元)"] = _safe_last(df_view["资金净流入估算(亿元)"], 2) if "资金净流入估算(亿元)" in df_view.columns else "N/A"
        rec["20日滚动波动率(%)"] = _safe_last(df_view["20日滚动波动率(%)"], 2) if "20日滚动波动率(%)" in df_view.columns else "N/A"
    if kind == "sector_index":
        rec["PE_TTM"] = _safe_last(df_view["PE_TTM"], 2) if "PE_TTM" in df_view.columns else "N/A"
        rec["PE_TTM历史分位(%)"] = _safe_last(df_view["PE_TTM历史分位(%)"], 1) if "PE_TTM历史分位(%)" in df_view.columns else "N/A"
        rec["PB"] = _safe_last(df_view["PB"], 2) if "PB" in df_view.columns else "N/A"
        rec["股息率"] = _safe_last(df_view["股息率"], 2) if "股息率" in df_view.columns else "N/A"
    return rec


def _run_one_target(name: str, symbol: str, kind: str, root_dir: str, benchmark_close: pd.Series, benchmark_changed: bool, data_provider: Optional[Dict[str, pd.DataFrame]] = None):
    print(f"\n▶ {name} ({symbol})  [{kind.upper()}]")
    save_dir = os.path.join(root_dir, _target_save_folder(kind), safe_fn(name))
    os.makedirs(save_dir, exist_ok=True)
    if data_provider and symbol in data_provider:
        df_raw = _normalize_ohlc(data_provider[symbol])
        df_analysis = prepare_analysis_df(df_raw, kind=kind, benchmark_close=benchmark_close if kind == "etf" else None, symbol=symbol)
    else:
        df_raw = fetch_kline_cached(symbol, kind, name)
        if df_raw is None or df_raw.empty:
            print(f"    [WARN] {name} 无原始K线数据，跳过")
            return None, None
        force_recalc = bool(benchmark_changed) if kind == "etf" else False
        df_analysis = get_analysis_df_cached(symbol, kind, name, df_raw, benchmark_close=benchmark_close if kind == "etf" else None, force_recalc=force_recalc)
    if df_analysis is None or df_analysis.empty:
        print(f"    [WARN] {name} 处理后数据为空，跳过")
        return None, None
    df_view = select_display_window(df_analysis, DISPLAY_BARS)
    if df_view.empty:
        print(f"    [WARN] {name} 展示窗口为空，跳过")
        return None, None
    kline_path = plot_daily_kline(name, symbol, df_view, save_dir, kind)
    df_fish, fish_path = plot_fish_bowl(name, symbol, df_analysis, save_dir, kind)
    export_excel(name, symbol, df_analysis, df_fish, save_dir, kind)
    rec = _build_snapshot_record(name, symbol, kind, df_view, df_fish)
    return {"kline": kline_path, "fish": fish_path}, rec


def run_tracker(data_provider: Optional[Dict[str, pd.DataFrame]] = None):
    date_tag = TODAY.strftime("%Y%m%d")
    root_dir = os.path.join(OUTPUT_DIR, date_tag)
    os.makedirs(root_dir, exist_ok=True)
    print("=" * 76)
    print(f"  A股市场日度跟踪系统 {TRACKER_VERSION}    {TODAY.strftime('%Y-%m-%d')}")
    print(f"  宽基:{len(BROAD_INDICES)}  行业指数:{len(SECTOR_INDICES)}  ETF:{len(SECTOR_ETFS)}  合计:{len(BROAD_INDICES) + len(SECTOR_INDICES) + len(SECTOR_ETFS)}")
    print(f"  展示窗口:{DISPLAY_BARS}日  MA200预热:{PREHEAT_BARS}日  鱼盆:MA5/MA20  输出:{os.path.abspath(root_dir)}")
    print("=" * 76)

    all_charts = {"per_target": {}}
    idx_fish_recs, sector_idx_recs, etf_fish_recs = [], [], []

    if data_provider and "000300" in data_provider:
        benchmark_raw = _normalize_ohlc(data_provider["000300"])
        benchmark_changed = True
        benchmark_df = prepare_analysis_df(benchmark_raw, kind="index", symbol="000300")
    else:
        benchmark_raw = fetch_kline_cached("000300", "index", "沪深300")
        benchmark_changed = CACHE_LAST_CHANGED.get(("index", "000300"), True)
        benchmark_df = get_analysis_df_cached("000300", "index", "沪深300", benchmark_raw, force_recalc=False)
    benchmark_close = benchmark_df["收盘"] if benchmark_df is not None and not benchmark_df.empty else pd.Series(dtype=float)

    for name, symbol, pepb_sym, kind, legu_code in ALL_TARGETS:
        charts, rec = _run_one_target(name, symbol, kind, root_dir, benchmark_close, benchmark_changed, data_provider=data_provider)
        if charts:
            all_charts["per_target"][f"{kind}:{name}"] = charts
        if rec:
            if kind == "etf":
                etf_fish_recs.append(rec)
            else:
                idx_fish_recs.append(rec)

    for item in SECTOR_INDICES:
        name = _sector_index_name(item)
        symbol = _sector_index_symbol(item)
        if not symbol:
            continue
        charts, rec = _run_one_target(name, symbol, "sector_index", root_dir, benchmark_close, benchmark_changed, data_provider=data_provider)
        if charts:
            all_charts["per_target"][f"sector_index:{name}"] = charts
        if rec:
            sector_idx_recs.append(rec)

    all_charts["index_fish_snapshot"] = plot_fishbowl_snapshot(idx_fish_recs, root_dir, "宽基鱼盆快照.png", f"A股宽基指数鱼盆快照    {TODAY.strftime('%Y年%m月%d日')}", kind="index")
    all_charts["sector_index_fish_snapshot"] = plot_fishbowl_snapshot(sector_idx_recs, root_dir, "行业指数鱼盆快照.png", f"A股行业指数鱼盆快照    {TODAY.strftime('%Y年%m月%d日')}", kind="sector_index")
    all_charts["etf_fish_snapshot"] = plot_fishbowl_snapshot(etf_fish_recs, root_dir, "ETF鱼盆快照.png", f"行业ETF鱼盆快照    {TODAY.strftime('%Y年%m月%d日')}", kind="etf")
    all_charts["etf_heatmap"] = plot_etf_heatmap(etf_fish_recs, root_dir)
    all_charts["snapshot_excel"] = export_snapshot_excel(idx_fish_recs, etf_fish_recs, root_dir, sector_index_records=sector_idx_recs)
    all_charts["pdf"] = export_pdf_report(root_dir, all_charts, date_tag)

    print("\n" + "=" * 76)
    print("  日度跟踪完成")
    print(f"  输出目录: {os.path.abspath(root_dir)}")
    print("=" * 76)
    return {
        "root_dir": root_dir,
        "charts": all_charts,
        "index_records": idx_fish_recs,
        "sector_index_records": sector_idx_recs,
        "etf_records": etf_fish_recs,
    }


if __name__ == "__main__":
    run_tracker()


  A股市场日度跟踪系统 v1.23    2026-05-24
  宽基:10  行业指数:48  ETF:35  合计:93
  展示窗口:100日  MA200预热:480日  鱼盆:MA5/MA20  输出:/Users/liam/Documents/Python/MarketTracker/output_daily_tracker/20260524
    [SRC-IDX] 东方财富指数日线（5191日）
    [CACHE] 已写入本地历史K线（5191日）
    [CACHE] 已重算并写入指标缓存（5191日，原始K线/估值更新或首次计算）

▶ 上证指数 (000001)  [INDEX]
    [SRC-IDX] 东方财富指数日线（8645日）
    [CACHE] 已写入本地历史K线（8645日）
    [CACHE] 已重算并写入指标缓存（8645日，原始K线/估值更新或首次计算）
    [OK] 日K线图 → 上证指数_日K线技术分析.png
    [OK] 鱼盆图   → 上证指数_鱼盆模型.png
    [OK] 数据表  → 上证指数_数据表.xlsx

▶ 上证50 (000016)  [INDEX]
    [WARN] 东方财富指数(000016): ProxyError
    [SRC-IDX] 新浪指数日线（5434日）
    [CACHE] 已写入本地历史K线（5434日）
    [CACHE] 已重算并写入指标缓存（5434日，原始K线/估值更新或首次计算）
    [OK] 日K线图 → 上证50_日K线技术分析.png
    [OK] 鱼盆图   → 上证50_鱼盆模型.png
    [OK] 数据表  → 上证50_数据表.xlsx

▶ 沪深300 (000300)  [INDEX]
    [CACHE] 读取本地历史K线（5191日）
    [WARN] 东方财富指数(000300): ProxyError
    [SRC-IDX] 新浪指数日线（12日）
    [CACHE] 增量更新完成：新增/修正 0 日，缓存共 5191 日
    [CACHE] 已重算并写入指标缓存（5191日，原始K线/估值更新或首次计算）
    [OK] 日K线图 → 沪深300_日K线技术

  0%|          | 0/2790 [00:00<?, ?it/s]